# Tabular MLP Baseline

## Purpose

This notebook trains the first deep learning baseline for MarketGuard India.

The model is a tabular multilayer perceptron that uses the same ordered 79
features as the existing Random Forest production models.

The objective is to determine whether a neural network can improve the
out-of-time performance of:

- NIFTY 50 outperformance prediction
- 10% downside-risk prediction

The Random Forest models remain the production baseline.

---

## Experiment Design

The primary experiments use the purged chronological split defined in
Notebook 11.

| Split | Period |
|---|---|
| Train | Through 2023-11-30 |
| Validation | 2024-01-02 through 2024-12-02 |
| Test | 2025-01-01 through 2026-06-16 |

The final 20 trading dates before each split boundary were removed from the
preceding split to prevent forward target windows from crossing into the next
evaluation period.

The legacy split will also be retained for direct comparison with the original
Random Forest benchmark.

---

## Initial Architecture

The first model will use a conservative tabular architecture:

```text
79 input features
        ↓
Linear layer
        ↓
Batch normalization
        ↓
ReLU activation
        ↓
Dropout
        ↓
Linear layer
        ↓
Batch normalization
        ↓
ReLU activation
        ↓
Dropout
        ↓
Binary output logit
```

The model will output one logit. Probabilities will be calculated using the
sigmoid function.

---

## Preprocessing

Preprocessing will be fitted using the training split only.

The process is:

1. Median imputation
2. Standard scaling
3. Conversion to 32-bit floating-point tensors

The fitted imputer and scaler will then transform validation and test data.

Validation and test data must not influence preprocessing statistics.

---

## Inital Training Rules

- Framework: PyTorch
- Device: CUDA when available
- Loss: binary cross-entropy with logits
- Optimizer: AdamW
- Early stopping: validation ROC-AUC
- Model selection: validation performance only
- Final test evaluation: once after model selection
- Reproducibility: fixed random seeds
- Best model checkpoint saved to disk

For the imbalanced downside target, class weighting will be calculated using
training data only.

---

## Evaluation Metrics

### Classification

- ROC-AUC
- Average precision
- Log loss
- Brier score
- Accuracy
- Balanced accuracy
- Precision
- Recall
- F1 score

### Probability Quality

- Predicted probability distribution
- Calibration curve
- Calibration error
- Brier score

### Ranking Quality

For outperformance:

- Probability deciles
- Top-decile versus bottom-decile return
- Excess return versus NIFTY 50
- Actual outperformance rate

For downside risk:

- Probability deciles
- Actual 5% downside rate
- Actual 10% downside rate
- Average worst-path return
- Highest-risk versus lowest-risk groups

---

## Model Promotion Rule

The neural network will not replace the Random Forest model solely because it
has a higher training or validation score.

Promotion requires:

1. Better untouched test performance
2. Stable ranking behavior
3. Acceptable probability calibration
4. Meaningful historical snapshot results
5. Reproducible training
6. No temporal or target leakage

Possible outcomes are:

```text
MLP improves both classification and ranking
        → Candidate for promotion

MLP and Random Forest provide complementary signals
        → Evaluate an ensemble

MLP does not provide reliable improvement
        → Retain the Random Forest baseline
```

---

## Notebook Stages

1. Environment and reproducibility setup
2. Data-contract loading
3. Purged dataset creation
4. Training-only preprocessing
5. PyTorch dataset and data-loader creation
6. Outperformance MLP training
7. Downside MLP training
8. Random Forest comparison
9. Ranking and calibration analysis
10. Artifact and report generation

### Imports , path , Device configuration

In [1]:
from pathlib import Path
import json
import random
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.set_float32_matmul_precision("high")


# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------

def find_project_root(start_path: Path) -> Path:
    """Find the MarketGuard repository root."""

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "config.yaml").exists() and (path / "README.md").exists():
            return path

    raise FileNotFoundError(
        "Could not find the project root containing config.yaml and README.md."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "targets"
    / "stock_features_with_targets_v1.parquet"
)

OUTPERFORM_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "best_random_forest_outperform_nifty50_20d_v1.joblib"
)

DOWNSIDE_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_downside_10pct_20d_v1.joblib"
)

DATA_CONTRACT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "deep_learning_experiments"
    / "deep_learning_data_contract_v1.json"
)

FEATURE_LIST_PATH = (
    PROJECT_ROOT
    / "reports"
    / "deep_learning_experiments"
    / "deep_learning_feature_list_v1.csv"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "deep_learning_experiments"
    / "tabular_mlp"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "deep_learning"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Device configuration
# ---------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    gpu_properties = torch.cuda.get_device_properties(0)

    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
    print(
        "GPU memory:",
        round(gpu_properties.total_memory / 1024**3, 2),
        "GB",
    )


# ---------------------------------------------------------
# Validate required files
# ---------------------------------------------------------

required_paths = [
    DATA_PATH,
    OUTPERFORM_MODEL_PATH,
    DOWNSIDE_MODEL_PATH,
    DATA_CONTRACT_PATH,
    FEATURE_LIST_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required experiment files are missing:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

print("\nAll required experiment files are available.")

Project root: E:\Projects\marketguard-india
Python: 3.12.13
pandas: 3.0.3
NumPy: 2.5.1
scikit-learn: 1.9.0
PyTorch: 2.13.0+cu126
Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
CUDA runtime: 12.6
GPU memory: 8.0 GB

All required experiment files are available.


### load the approved data contract, restore the exact 79-feature order, recreate both split methods

In [2]:
# ---------------------------------------------------------
# Load the approved data contract and dataset
# ---------------------------------------------------------

SPLIT_AUDIT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "deep_learning_experiments"
    / "deep_learning_split_audit_v1.csv"
)

PURGE_DATES_PATH = (
    PROJECT_ROOT
    / "reports"
    / "deep_learning_experiments"
    / "deep_learning_purge_dates_v1.csv"
)


def extract_feature_names(fitted_model) -> list[str]:
    """Extract ordered feature names from a fitted sklearn model or pipeline."""

    if hasattr(fitted_model, "feature_names_in_"):
        return list(fitted_model.feature_names_in_)

    if hasattr(fitted_model, "named_steps"):
        for step in fitted_model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise AttributeError(
        "Could not find feature_names_in_ in the fitted model."
    )


with DATA_CONTRACT_PATH.open("r", encoding="utf-8") as file:
    data_contract = json.load(file)

feature_list_df = (
    pd.read_csv(FEATURE_LIST_PATH)
    .sort_values("feature_order")
    .reset_index(drop=True)
)

saved_split_audit = pd.read_csv(
    SPLIT_AUDIT_PATH,
    parse_dates=["start_date", "end_date"],
)

purge_date_audit = pd.read_csv(
    PURGE_DATES_PATH,
    parse_dates=["date"],
)

data = pd.read_parquet(DATA_PATH)
data["date"] = pd.to_datetime(data["date"])

outperform_rf_model = joblib.load(OUTPERFORM_MODEL_PATH)
downside_rf_model = joblib.load(DOWNSIDE_MODEL_PATH)


# ---------------------------------------------------------
# Validate the exact feature contract
# ---------------------------------------------------------

feature_cols = feature_list_df["feature"].tolist()

outperform_rf_features = extract_feature_names(
    outperform_rf_model
)

downside_rf_features = extract_feature_names(
    downside_rf_model
)

if feature_cols != outperform_rf_features:
    raise ValueError(
        "Saved feature list does not match the outperform model."
    )

if feature_cols != downside_rf_features:
    raise ValueError(
        "Saved feature list does not match the downside model."
    )

if len(feature_cols) != data_contract["feature_count"]:
    raise ValueError(
        "Feature count does not match the saved data contract."
    )


# ---------------------------------------------------------
# Recreate the approved modeling population
# ---------------------------------------------------------

READY_COL = "target_ready_v1_20d"

OUTPERFORM_TARGET = data_contract["targets"]["outperform"]
DOWNSIDE_TARGET = data_contract["targets"]["downside"]

model_data = (
    data.loc[data[READY_COL].eq(1)]
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
    .copy()
)

if len(model_data) != data_contract["modeling_rows"]:
    raise ValueError(
        "Modeling row count does not match the data contract."
    )

if model_data["yf_ticker"].nunique() != data_contract["stock_count"]:
    raise ValueError(
        "Stock count does not match the data contract."
    )


# ---------------------------------------------------------
# Recreate the legacy split
# ---------------------------------------------------------

legacy_config = data_contract["legacy_split"]

TRAIN_END_DATE = pd.Timestamp(
    legacy_config["train_end_date"]
)

VALID_START_DATE = pd.Timestamp(
    legacy_config["validation_start_date"]
)

VALID_END_DATE = pd.Timestamp(
    legacy_config["validation_end_date"]
)

TEST_START_DATE = pd.Timestamp(
    legacy_config["test_start_date"]
)

legacy_train_mask = model_data["date"].le(
    TRAIN_END_DATE
)

legacy_valid_mask = (
    model_data["date"].ge(VALID_START_DATE)
    & model_data["date"].le(VALID_END_DATE)
)

legacy_test_mask = model_data["date"].ge(
    TEST_START_DATE
)

legacy_split_masks = {
    "train": legacy_train_mask,
    "valid": legacy_valid_mask,
    "test": legacy_test_mask,
}


# ---------------------------------------------------------
# Recreate the purged split using saved purge dates
# ---------------------------------------------------------

train_purge_dates = pd.DatetimeIndex(
    purge_date_audit.loc[
        purge_date_audit["purged_from_split"].eq("train"),
        "date",
    ]
)

valid_purge_dates = pd.DatetimeIndex(
    purge_date_audit.loc[
        purge_date_audit["purged_from_split"].eq("valid"),
        "date",
    ]
)

purged_train_mask = (
    legacy_train_mask
    & ~model_data["date"].isin(train_purge_dates)
)

purged_valid_mask = (
    legacy_valid_mask
    & ~model_data["date"].isin(valid_purge_dates)
)

purged_test_mask = legacy_test_mask.copy()

purged_split_masks = {
    "train": purged_train_mask,
    "valid": purged_valid_mask,
    "test": purged_test_mask,
}

all_split_masks = {
    "legacy": legacy_split_masks,
    "purged": purged_split_masks,
}


# ---------------------------------------------------------
# Independently rebuild and verify the split audit
# ---------------------------------------------------------

actual_audit_rows = []

for split_method, split_masks in all_split_masks.items():
    for split_name, split_mask in split_masks.items():
        split_frame = model_data.loc[split_mask]

        actual_audit_rows.append(
            {
                "split_method": split_method,
                "split": split_name,
                "rows": len(split_frame),
                "stocks": split_frame["yf_ticker"].nunique(),
                "trading_dates": split_frame["date"].nunique(),
                "start_date": split_frame["date"].min(),
                "end_date": split_frame["date"].max(),
            }
        )

actual_split_audit = pd.DataFrame(actual_audit_rows)

comparison = saved_split_audit.merge(
    actual_split_audit,
    on=["split_method", "split"],
    suffixes=("_expected", "_actual"),
    validate="one_to_one",
)

audit_columns = [
    "rows",
    "stocks",
    "trading_dates",
    "start_date",
    "end_date",
]

for column in audit_columns:
    expected = comparison[f"{column}_expected"]
    actual = comparison[f"{column}_actual"]

    if not expected.equals(actual):
        raise ValueError(
            f"Split audit mismatch found in column: {column}"
        )


# ---------------------------------------------------------
# Display verified configuration
# ---------------------------------------------------------

print("Modeling rows:", len(model_data))
print("Stocks:", model_data["yf_ticker"].nunique())
print("Feature count:", len(feature_cols))
print("Outperformance target:", OUTPERFORM_TARGET)
print("Downside target:", DOWNSIDE_TARGET)

print("\nTraining purge dates:", len(train_purge_dates))
print(
    train_purge_dates.min().date(),
    "to",
    train_purge_dates.max().date(),
)

print("\nValidation purge dates:", len(valid_purge_dates))
print(
    valid_purge_dates.min().date(),
    "to",
    valid_purge_dates.max().date(),
)

print("\nSplit audit verified successfully.")

display(actual_split_audit)

Modeling rows: 307453
Stocks: 90
Feature count: 79
Outperformance target: target_outperform_nifty50_20d
Downside target: target_big_downside_10pct_20d

Training purge dates: 20
2023-12-01 to 2023-12-29

Validation purge dates: 20
2024-12-03 to 2024-12-31

Split audit verified successfully.


,split_method,split,rows,stocks,trading_dates,start_date,end_date
0,legacy,train,253538,90,3156,2011-01-03,2023-12-29
1,legacy,valid,21780,90,242,2024-01-02,2024-12-31
2,legacy,test,32135,90,359,2025-01-01,2026-06-16
3,purged,train,251738,90,3136,2011-01-03,2023-11-30
4,purged,valid,19980,90,222,2024-01-02,2024-12-02
5,purged,test,32135,90,359,2025-01-01,2026-06-16


### Preparing the MLP inputs using only the purged training data to fit preprocessing.

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# Select the primary purged split
# ---------------------------------------------------------

PRIMARY_SPLIT_METHOD = "purged"

primary_masks = all_split_masks[PRIMARY_SPLIT_METHOD]

train_mask = primary_masks["train"]
valid_mask = primary_masks["valid"]
test_mask = primary_masks["test"]


# ---------------------------------------------------------
# Build raw feature matrices
# ---------------------------------------------------------

X_train_raw = model_data.loc[
    train_mask,
    feature_cols,
].copy()

X_valid_raw = model_data.loc[
    valid_mask,
    feature_cols,
].copy()

X_test_raw = model_data.loc[
    test_mask,
    feature_cols,
].copy()


# ---------------------------------------------------------
# Build target arrays
# ---------------------------------------------------------

y_train_outperform = (
    model_data.loc[train_mask, OUTPERFORM_TARGET]
    .astype(np.float32)
    .to_numpy()
)

y_valid_outperform = (
    model_data.loc[valid_mask, OUTPERFORM_TARGET]
    .astype(np.float32)
    .to_numpy()
)

y_test_outperform = (
    model_data.loc[test_mask, OUTPERFORM_TARGET]
    .astype(np.float32)
    .to_numpy()
)

y_train_downside = (
    model_data.loc[train_mask, DOWNSIDE_TARGET]
    .astype(np.float32)
    .to_numpy()
)

y_valid_downside = (
    model_data.loc[valid_mask, DOWNSIDE_TARGET]
    .astype(np.float32)
    .to_numpy()
)

y_test_downside = (
    model_data.loc[test_mask, DOWNSIDE_TARGET]
    .astype(np.float32)
    .to_numpy()
)


# ---------------------------------------------------------
# Fit preprocessing using training data only
# ---------------------------------------------------------

mlp_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

X_train = mlp_preprocessor.fit_transform(
    X_train_raw
)

X_valid = mlp_preprocessor.transform(
    X_valid_raw
)

X_test = mlp_preprocessor.transform(
    X_test_raw
)


# ---------------------------------------------------------
# Convert to compact float32 arrays
# ---------------------------------------------------------

X_train = np.ascontiguousarray(
    X_train,
    dtype=np.float32,
)

X_valid = np.ascontiguousarray(
    X_valid,
    dtype=np.float32,
)

X_test = np.ascontiguousarray(
    X_test,
    dtype=np.float32,
)


# ---------------------------------------------------------
# Validate transformed arrays
# ---------------------------------------------------------

expected_shapes = {
    "train": (int(train_mask.sum()), len(feature_cols)),
    "valid": (int(valid_mask.sum()), len(feature_cols)),
    "test": (int(test_mask.sum()), len(feature_cols)),
}

actual_shapes = {
    "train": X_train.shape,
    "valid": X_valid.shape,
    "test": X_test.shape,
}

if actual_shapes != expected_shapes:
    raise ValueError(
        "Unexpected transformed feature shapes.\n"
        f"Expected: {expected_shapes}\n"
        f"Actual: {actual_shapes}"
    )

for split_name, X in [
    ("train", X_train),
    ("valid", X_valid),
    ("test", X_test),
]:
    if not np.isfinite(X).all():
        raise ValueError(
            f"Non-finite values remain in the {split_name} features."
        )

target_arrays = {
    "train_outperform": y_train_outperform,
    "valid_outperform": y_valid_outperform,
    "test_outperform": y_test_outperform,
    "train_downside": y_train_downside,
    "valid_downside": y_valid_downside,
    "test_downside": y_test_downside,
}

for target_name, target_values in target_arrays.items():
    if not np.isfinite(target_values).all():
        raise ValueError(
            f"Non-finite values found in target array: {target_name}"
        )


# ---------------------------------------------------------
# Calculate training-only downside class weight
# ---------------------------------------------------------

downside_positive_count = float(
    y_train_downside.sum()
)

downside_negative_count = float(
    len(y_train_downside) - downside_positive_count
)

downside_pos_weight = (
    downside_negative_count
    / downside_positive_count
)

if downside_pos_weight <= 0:
    raise ValueError(
        "Calculated downside positive-class weight is invalid."
    )


# ---------------------------------------------------------
# Display preprocessing audit
# ---------------------------------------------------------

preprocessing_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(X_train),
            "features": X_train.shape[1],
            "outperform_positive_pct": (
                y_train_outperform.mean() * 100
            ),
            "downside_positive_pct": (
                y_train_downside.mean() * 100
            ),
        },
        {
            "split": "valid",
            "rows": len(X_valid),
            "features": X_valid.shape[1],
            "outperform_positive_pct": (
                y_valid_outperform.mean() * 100
            ),
            "downside_positive_pct": (
                y_valid_downside.mean() * 100
            ),
        },
        {
            "split": "test",
            "rows": len(X_test),
            "features": X_test.shape[1],
            "outperform_positive_pct": (
                y_test_outperform.mean() * 100
            ),
            "downside_positive_pct": (
                y_test_downside.mean() * 100
            ),
        },
    ]
)

print("Primary split method:", PRIMARY_SPLIT_METHOD)
print("Feature dtype:", X_train.dtype)
print("Training feature shape:", X_train.shape)
print("Validation feature shape:", X_valid.shape)
print("Test feature shape:", X_test.shape)

print(
    "\nDownside training positive-class weight:",
    round(downside_pos_weight, 4),
)

print(
    "\nMaximum absolute transformed training mean:",
    round(
        float(
            np.abs(
                X_train.mean(axis=0)
            ).max()
        ),
        6,
    ),
)

print(
    "Training feature standard-deviation range:",
    round(
        float(
            X_train.std(axis=0).min()
        ),
        6,
    ),
    "to",
    round(
        float(
            X_train.std(axis=0).max()
        ),
        6,
    ),
)

display(preprocessing_summary)

Primary split method: purged
Feature dtype: float32
Training feature shape: (251738, 79)
Validation feature shape: (19980, 79)
Test feature shape: (32135, 79)

Downside training positive-class weight: 5.7333

Maximum absolute transformed training mean: 0.00023
Training feature standard-deviation range: 0.998906 to 1.001233


,split,rows,features,outperform_positive_pct,downside_positive_pct
0,train,251738,79,51.214756,14.851552
1,valid,19980,79,51.126129,10.490491
2,test,32135,79,51.728642,11.196515


The preprocessing stage passed correctly.

        Training means are effectively zero.
        Standard deviations are effectively one.
        No non-finite values remain.
        The downside pos_weight of 5.7333 means each positive downside example can receive about 5.73 times the loss weight of a negative example.
        We will use loss weighting rather than oversampling, so the original chronological training distribution remains intact.

### PyTorch datasets and data loaders

In [4]:
from torch.utils.data import DataLoader, TensorDataset


# ---------------------------------------------------------
# Data-loader configuration
# ---------------------------------------------------------

TRAIN_BATCH_SIZE = 4096
EVAL_BATCH_SIZE = 8192

PIN_MEMORY = DEVICE.type == "cuda"
NUM_WORKERS = 0


def create_tensor_dataset(
    features: np.ndarray,
    targets: np.ndarray,
) -> TensorDataset:
    """
    Create a TensorDataset using independent, contiguous,
    writable float32 NumPy arrays.
    """

    feature_array = np.array(
        features,
        dtype=np.float32,
        order="C",
        copy=True,
    )

    target_array = np.array(
        targets,
        dtype=np.float32,
        order="C",
        copy=True,
    )

    if feature_array.ndim != 2:
        raise ValueError(
            f"Expected 2D features, found shape {feature_array.shape}."
        )

    if target_array.ndim != 1:
        raise ValueError(
            f"Expected 1D targets, found shape {target_array.shape}."
        )

    if len(feature_array) != len(target_array):
        raise ValueError(
            "Feature and target row counts do not match."
        )

    if not feature_array.flags.writeable:
        raise ValueError(
            "Feature array is unexpectedly read-only."
        )

    if not target_array.flags.writeable:
        raise ValueError(
            "Target array is unexpectedly read-only."
        )

    feature_tensor = torch.from_numpy(feature_array)
    target_tensor = torch.from_numpy(target_array)

    return TensorDataset(
        feature_tensor,
        target_tensor,
    )


def create_data_loader(
    dataset: TensorDataset,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    """Create a reproducible PyTorch data loader."""

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        generator=generator if shuffle else None,
    )


# ---------------------------------------------------------
# Create datasets
# ---------------------------------------------------------

outperform_train_dataset = create_tensor_dataset(
    X_train,
    y_train_outperform,
)

outperform_valid_dataset = create_tensor_dataset(
    X_valid,
    y_valid_outperform,
)

outperform_test_dataset = create_tensor_dataset(
    X_test,
    y_test_outperform,
)

downside_train_dataset = create_tensor_dataset(
    X_train,
    y_train_downside,
)

downside_valid_dataset = create_tensor_dataset(
    X_valid,
    y_valid_downside,
)

downside_test_dataset = create_tensor_dataset(
    X_test,
    y_test_downside,
)


# ---------------------------------------------------------
# Create data loaders
# ---------------------------------------------------------

outperform_train_loader = create_data_loader(
    outperform_train_dataset,
    TRAIN_BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED,
)

outperform_valid_loader = create_data_loader(
    outperform_valid_dataset,
    EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)

outperform_test_loader = create_data_loader(
    outperform_test_dataset,
    EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)

downside_train_loader = create_data_loader(
    downside_train_dataset,
    TRAIN_BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED,
)

downside_valid_loader = create_data_loader(
    downside_valid_dataset,
    EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)

downside_test_loader = create_data_loader(
    downside_test_dataset,
    EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)


# ---------------------------------------------------------
# Validate one batch
# ---------------------------------------------------------

sample_features, sample_targets = next(
    iter(outperform_train_loader)
)

print("CUDA pin memory:", PIN_MEMORY)
print("Data-loader workers:", NUM_WORKERS)
print("Sample feature batch:", sample_features.shape)
print("Sample target batch:", sample_targets.shape)
print("Sample feature dtype:", sample_features.dtype)
print("Sample target dtype:", sample_targets.dtype)

print(
    "Feature tensor writable source:",
    outperform_train_dataset.tensors[0]
    .numpy()
    .flags.writeable,
)

print(
    "Target tensor writable source:",
    outperform_train_dataset.tensors[1]
    .numpy()
    .flags.writeable,
)

loader_summary = pd.DataFrame(
    [
        {
            "target": target_name,
            "split": split_name,
            "rows": len(dataset),
            "batch_size": batch_size,
            "batches": len(loader),
            "shuffle": shuffle,
        }
        for target_name, split_name, dataset, batch_size, loader, shuffle in [
            (
                "outperform",
                "train",
                outperform_train_dataset,
                TRAIN_BATCH_SIZE,
                outperform_train_loader,
                True,
            ),
            (
                "outperform",
                "valid",
                outperform_valid_dataset,
                EVAL_BATCH_SIZE,
                outperform_valid_loader,
                False,
            ),
            (
                "outperform",
                "test",
                outperform_test_dataset,
                EVAL_BATCH_SIZE,
                outperform_test_loader,
                False,
            ),
            (
                "downside",
                "train",
                downside_train_dataset,
                TRAIN_BATCH_SIZE,
                downside_train_loader,
                True,
            ),
            (
                "downside",
                "valid",
                downside_valid_dataset,
                EVAL_BATCH_SIZE,
                downside_valid_loader,
                False,
            ),
            (
                "downside",
                "test",
                downside_test_dataset,
                EVAL_BATCH_SIZE,
                downside_test_loader,
                False,
            ),
        ]
    ]
)

display(loader_summary)

CUDA pin memory: True
Data-loader workers: 0
Sample feature batch: torch.Size([4096, 79])
Sample target batch: torch.Size([4096])
Sample feature dtype: torch.float32
Sample target dtype: torch.float32
Feature tensor writable source: True
Target tensor writable source: True


,target,split,rows,batch_size,batches,shuffle
0,outperform,train,251738,4096,62,True
1,outperform,valid,19980,8192,3,False
2,outperform,test,32135,8192,4,False
3,downside,train,251738,4096,62,True
4,downside,valid,19980,8192,3,False
5,downside,test,32135,8192,4,False


### MLP architecture and run a forward-pass check before adding any training logic.

In [5]:
from collections.abc import Sequence

import torch.nn as nn


# ---------------------------------------------------------
# Initial tabular MLP configuration
# ---------------------------------------------------------

MLP_CONFIG = {
    "input_dim": len(feature_cols),
    "hidden_dims": [256, 128, 64],
    "dropout_rates": [0.20, 0.15, 0.10],
    "activation": "ReLU",
    "output_dim": 1,
}


class TabularMLP(nn.Module):
    """
    Feed-forward neural network for binary classification
    using standardized tabular features.

    The network returns raw logits. Sigmoid is applied only
    when probabilities are needed.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dims: Sequence[int] = (256, 128, 64),
        dropout_rates: Sequence[float] = (0.20, 0.15, 0.10),
    ) -> None:
        super().__init__()

        if input_dim <= 0:
            raise ValueError(
                f"input_dim must be positive, found {input_dim}."
            )

        if len(hidden_dims) != len(dropout_rates):
            raise ValueError(
                "hidden_dims and dropout_rates must have equal lengths."
            )

        if not hidden_dims:
            raise ValueError(
                "At least one hidden layer is required."
            )

        layers: list[nn.Module] = []

        previous_dim = input_dim

        for hidden_dim, dropout_rate in zip(
            hidden_dims,
            dropout_rates,
            strict=True,
        ):
            if hidden_dim <= 0:
                raise ValueError(
                    f"Hidden dimensions must be positive: {hidden_dim}"
                )

            if not 0 <= dropout_rate < 1:
                raise ValueError(
                    f"Invalid dropout rate: {dropout_rate}"
                )

            layers.extend(
                [
                    nn.Linear(previous_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout_rate),
                ]
            )

            previous_dim = hidden_dim

        self.feature_network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(previous_dim, 1)

        self._initialize_weights()

    def _initialize_weights(self) -> None:
        """Initialize linear layers for ReLU activations."""

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """
        Return one raw logit per row.

        Output shape:
            [batch_size]
        """

        hidden = self.feature_network(features)
        logits = self.output_layer(hidden)

        return logits.squeeze(-1)


def count_model_parameters(
    model: nn.Module,
) -> dict[str, int]:
    """Count total and trainable model parameters."""

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return {
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
    }


# ---------------------------------------------------------
# Instantiate the outperformance model
# ---------------------------------------------------------

outperform_mlp = TabularMLP(
    input_dim=MLP_CONFIG["input_dim"],
    hidden_dims=MLP_CONFIG["hidden_dims"],
    dropout_rates=MLP_CONFIG["dropout_rates"],
).to(DEVICE)

parameter_counts = count_model_parameters(
    outperform_mlp
)


# ---------------------------------------------------------
# Forward-pass validation
# ---------------------------------------------------------

dry_run_features = sample_features[:32].to(
    DEVICE,
    non_blocking=PIN_MEMORY,
)

outperform_mlp.eval()

with torch.inference_mode():
    dry_run_logits = outperform_mlp(
        dry_run_features
    )

    dry_run_probabilities = torch.sigmoid(
        dry_run_logits
    )

if dry_run_logits.shape != torch.Size([32]):
    raise ValueError(
        "Unexpected MLP output shape: "
        f"{dry_run_logits.shape}"
    )

if not torch.isfinite(dry_run_logits).all():
    raise ValueError(
        "The initial forward pass produced non-finite logits."
    )

if not (
    (dry_run_probabilities >= 0)
    & (dry_run_probabilities <= 1)
).all():
    raise ValueError(
        "The initial probabilities are outside the range [0, 1]."
    )

outperform_mlp.train()


# ---------------------------------------------------------
# Display architecture audit
# ---------------------------------------------------------

print(outperform_mlp)

print("\nInput features:", MLP_CONFIG["input_dim"])
print("Hidden dimensions:", MLP_CONFIG["hidden_dims"])
print("Dropout rates:", MLP_CONFIG["dropout_rates"])

print(
    "Total parameters:",
    f"{parameter_counts['total_parameters']:,}",
)

print(
    "Trainable parameters:",
    f"{parameter_counts['trainable_parameters']:,}",
)

print("\nDry-run device:", dry_run_logits.device)
print("Dry-run logits shape:", dry_run_logits.shape)

print(
    "Initial probability range:",
    round(
        float(dry_run_probabilities.min().item()),
        6,
    ),
    "to",
    round(
        float(dry_run_probabilities.max().item()),
        6,
    ),
)

if DEVICE.type == "cuda":
    print(
        "Allocated GPU memory:",
        round(
            torch.cuda.memory_allocated() / 1024**2,
            2,
        ),
        "MB",
    )

TabularMLP(
  (feature_network): Sequential(
    (0): Linear(in_features=79, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.15, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.1, inplace=False)
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)

Input features: 79
Hidden dimensions: [256, 128, 64]
Dropout rates: [0.2, 0.15, 0.1]
Total parameters: 62,593
Trainable parameters: 62,593

Dry-run device: cuda:0
Dry-run logits shape: torch.Size([32])
Initial probability rang

### define the shared training and evaluation utilities but does not train the model yet.

In [6]:
from collections.abc import Callable

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ---------------------------------------------------------
# Prediction and metric utilities
# ---------------------------------------------------------

def calculate_binary_metrics(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    threshold: float = 0.50,
) -> dict[str, float]:
    """Calculate probability and threshold-based binary metrics."""

    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    y_probability = np.asarray(
        y_probability,
        dtype=np.float64,
    )

    if y_true.ndim != 1:
        raise ValueError(
            f"Expected one-dimensional targets, found {y_true.shape}."
        )

    if y_probability.ndim != 1:
        raise ValueError(
            "Expected one-dimensional probabilities, "
            f"found {y_probability.shape}."
        )

    if len(y_true) != len(y_probability):
        raise ValueError(
            "Target and probability lengths do not match."
        )

    if not np.isfinite(y_probability).all():
        raise ValueError(
            "Non-finite predicted probabilities were found."
        )

    if not 0 < threshold < 1:
        raise ValueError(
            f"Threshold must be between zero and one: {threshold}"
        )

    clipped_probability = np.clip(
        y_probability,
        1e-7,
        1 - 1e-7,
    )

    y_predicted = (
        y_probability >= threshold
    ).astype(np.int64)

    unique_classes = np.unique(y_true)

    roc_auc = (
        roc_auc_score(
            y_true,
            y_probability,
        )
        if len(unique_classes) == 2
        else np.nan
    )

    average_precision = (
        average_precision_score(
            y_true,
            y_probability,
        )
        if len(unique_classes) == 2
        else np.nan
    )

    return {
        "rows": int(len(y_true)),
        "positive_rate_actual": float(y_true.mean()),
        "positive_rate_predicted": float(y_predicted.mean()),
        "average_probability": float(y_probability.mean()),
        "roc_auc": float(roc_auc),
        "average_precision": float(average_precision),
        "log_loss": float(
            log_loss(
                y_true,
                clipped_probability,
                labels=[0, 1],
            )
        ),
        "brier_score": float(
            brier_score_loss(
                y_true,
                y_probability,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                y_predicted,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_predicted,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_predicted,
                zero_division=0,
            )
        ),
        "threshold": float(threshold),
    }


def evaluate_binary_model(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: Callable,
    device: torch.device,
    threshold: float = 0.50,
) -> tuple[dict[str, float], np.ndarray, np.ndarray]:
    """
    Evaluate a binary classifier without updating its parameters.

    Returns:
        metrics
        actual targets
        predicted probabilities
    """

    model.eval()

    total_loss = 0.0
    total_rows = 0

    target_batches = []
    probability_batches = []

    with torch.inference_mode():
        for feature_batch, target_batch in data_loader:
            feature_batch = feature_batch.to(
                device,
                non_blocking=PIN_MEMORY,
            )

            target_batch = target_batch.to(
                device,
                non_blocking=PIN_MEMORY,
            )

            logits = model(feature_batch)

            if logits.shape != target_batch.shape:
                raise ValueError(
                    "Logit and target shapes do not match: "
                    f"{logits.shape} versus {target_batch.shape}"
                )

            loss = loss_function(
                logits,
                target_batch,
            )

            if not torch.isfinite(loss):
                raise ValueError(
                    "A non-finite evaluation loss was produced."
                )

            probabilities = torch.sigmoid(logits)

            batch_rows = len(target_batch)

            total_loss += (
                float(loss.item()) * batch_rows
            )

            total_rows += batch_rows

            target_batches.append(
                target_batch.detach().cpu().numpy()
            )

            probability_batches.append(
                probabilities.detach().cpu().numpy()
            )

    if total_rows == 0:
        raise ValueError(
            "The evaluation data loader contains no rows."
        )

    y_true = np.concatenate(
        target_batches
    ).astype(np.int64)

    y_probability = np.concatenate(
        probability_batches
    ).astype(np.float64)

    metrics = calculate_binary_metrics(
        y_true=y_true,
        y_probability=y_probability,
        threshold=threshold,
    )

    metrics["loss"] = (
        total_loss / total_rows
    )

    return metrics, y_true, y_probability


# ---------------------------------------------------------
# One-epoch training utility
# ---------------------------------------------------------

def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_function: Callable,
    device: torch.device,
    max_gradient_norm: float = 5.0,
) -> float:
    """Train a binary classifier for one complete epoch."""

    if max_gradient_norm <= 0:
        raise ValueError(
            "max_gradient_norm must be positive."
        )

    model.train()

    total_loss = 0.0
    total_rows = 0

    for feature_batch, target_batch in data_loader:
        feature_batch = feature_batch.to(
            device,
            non_blocking=PIN_MEMORY,
        )

        target_batch = target_batch.to(
            device,
            non_blocking=PIN_MEMORY,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(feature_batch)

        if logits.shape != target_batch.shape:
            raise ValueError(
                "Logit and target shapes do not match: "
                f"{logits.shape} versus {target_batch.shape}"
            )

        loss = loss_function(
            logits,
            target_batch,
        )

        if not torch.isfinite(loss):
            raise ValueError(
                "A non-finite training loss was produced."
            )

        loss.backward()

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_gradient_norm,
            error_if_nonfinite=True,
        )

        if not torch.isfinite(gradient_norm):
            raise ValueError(
                "A non-finite gradient norm was produced."
            )

        optimizer.step()

        batch_rows = len(target_batch)

        total_loss += (
            float(loss.item()) * batch_rows
        )

        total_rows += batch_rows

    if total_rows == 0:
        raise ValueError(
            "The training data loader contains no rows."
        )

    return total_loss / total_rows


# ---------------------------------------------------------
# Validate evaluation utilities on the untrained model
# ---------------------------------------------------------

outperform_loss_function = (
    nn.BCEWithLogitsLoss()
)

initial_valid_metrics, _, _ = evaluate_binary_model(
    model=outperform_mlp,
    data_loader=outperform_valid_loader,
    loss_function=outperform_loss_function,
    device=DEVICE,
    threshold=0.50,
)

initial_metric_display = pd.DataFrame(
    [
        {
            "stage": "untrained_forward_check",
            **initial_valid_metrics,
        }
    ]
)

print("Training and evaluation utilities created successfully.")
print("This is an untrained-model check, not a benchmark result.")

display(
    initial_metric_display[
        [
            "stage",
            "rows",
            "loss",
            "roc_auc",
            "average_precision",
            "average_probability",
            "positive_rate_predicted",
        ]
    ]
)

Training and evaluation utilities created successfully.
This is an untrained-model check, not a benchmark result.


,stage,rows,loss,roc_auc,average_precision,average_probability,positive_rate_predicted
0,untrained_forward_check,19980,0.769052,0.506527,0.514749,0.455741,0.394945


### Train the outperformance MLP using only train and validation data.

In [7]:
import copy
import time


# ---------------------------------------------------------
# Outperformance training configuration
# ---------------------------------------------------------

OUTPERFORM_TRAINING_CONFIG = {
    "max_epochs": 50,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "early_stopping_patience": 15,
    "minimum_auc_improvement": 1e-4,
    "gradient_clip_norm": 5.0,
    "scheduler_factor": 0.50,
    "scheduler_patience": 2,
    "minimum_learning_rate": 1e-5,
}

OUTPERFORM_CHECKPOINT_PATH = (
    MODEL_DIR
    / "tabular_mlp_outperform_purged_v1.pt"
)

OUTPERFORM_HISTORY_PATH = (
    REPORT_DIR
    / "tabular_mlp_outperform_training_history_v1.csv"
)


# ---------------------------------------------------------
# Reset random state before training
# ---------------------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# Recreate the shuffled loader because an earlier sample batch
# consumed part of its random-generator state.
outperform_train_loader = create_data_loader(
    dataset=outperform_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED,
)

# Separate ordered loader for final training-set evaluation.
outperform_train_eval_loader = create_data_loader(
    dataset=outperform_train_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)


# ---------------------------------------------------------
# Create a fresh model
# ---------------------------------------------------------

outperform_mlp = TabularMLP(
    input_dim=MLP_CONFIG["input_dim"],
    hidden_dims=MLP_CONFIG["hidden_dims"],
    dropout_rates=MLP_CONFIG["dropout_rates"],
).to(DEVICE)

outperform_loss_function = nn.BCEWithLogitsLoss()

outperform_optimizer = torch.optim.AdamW(
    outperform_mlp.parameters(),
    lr=OUTPERFORM_TRAINING_CONFIG["learning_rate"],
    weight_decay=OUTPERFORM_TRAINING_CONFIG["weight_decay"],
)

outperform_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        outperform_optimizer,
        mode="max",
        factor=OUTPERFORM_TRAINING_CONFIG["scheduler_factor"],
        patience=OUTPERFORM_TRAINING_CONFIG["scheduler_patience"],
        threshold=OUTPERFORM_TRAINING_CONFIG[
            "minimum_auc_improvement"
        ],
        min_lr=OUTPERFORM_TRAINING_CONFIG[
            "minimum_learning_rate"
        ],
    )
)


# ---------------------------------------------------------
# Train with validation ROC-AUC early stopping
# ---------------------------------------------------------

training_history = []

best_epoch = 0
best_valid_roc_auc = -np.inf
best_valid_metrics = None
best_model_state = None

epochs_without_improvement = 0

training_start_time = time.perf_counter()

for epoch in range(
    1,
    OUTPERFORM_TRAINING_CONFIG["max_epochs"] + 1,
):
    epoch_start_time = time.perf_counter()

    current_learning_rate = (
        outperform_optimizer.param_groups[0]["lr"]
    )

    train_loss = train_one_epoch(
        model=outperform_mlp,
        data_loader=outperform_train_loader,
        optimizer=outperform_optimizer,
        loss_function=outperform_loss_function,
        device=DEVICE,
        max_gradient_norm=OUTPERFORM_TRAINING_CONFIG[
            "gradient_clip_norm"
        ],
    )

    valid_metrics, _, _ = evaluate_binary_model(
        model=outperform_mlp,
        data_loader=outperform_valid_loader,
        loss_function=outperform_loss_function,
        device=DEVICE,
        threshold=0.50,
    )

    valid_roc_auc = valid_metrics["roc_auc"]

    outperform_scheduler.step(valid_roc_auc)

    epoch_seconds = (
        time.perf_counter() - epoch_start_time
    )

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_metrics["loss"],
            "valid_roc_auc": valid_roc_auc,
            "valid_average_precision": valid_metrics[
                "average_precision"
            ],
            "valid_log_loss": valid_metrics["log_loss"],
            "valid_brier_score": valid_metrics["brier_score"],
            "valid_accuracy": valid_metrics["accuracy"],
            "valid_balanced_accuracy": valid_metrics[
                "balanced_accuracy"
            ],
            "valid_precision": valid_metrics["precision"],
            "valid_recall": valid_metrics["recall"],
            "valid_f1": valid_metrics["f1"],
            "average_valid_probability": valid_metrics[
                "average_probability"
            ],
            "learning_rate": current_learning_rate,
            "epoch_seconds": epoch_seconds,
        }
    )

    improved = (
        valid_roc_auc
        > best_valid_roc_auc
        + OUTPERFORM_TRAINING_CONFIG[
            "minimum_auc_improvement"
        ]
    )

    if improved:
        best_epoch = epoch
        best_valid_roc_auc = valid_roc_auc
        best_valid_metrics = valid_metrics.copy()

        best_model_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor
            in outperform_mlp.state_dict().items()
        }

        epochs_without_improvement = 0
        improvement_marker = "  <-- best"

    else:
        epochs_without_improvement += 1
        improvement_marker = ""

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.6f} | "
        f"valid loss {valid_metrics['loss']:.6f} | "
        f"valid AUC {valid_roc_auc:.6f} | "
        f"valid AP {valid_metrics['average_precision']:.6f} | "
        f"LR {current_learning_rate:.2e} | "
        f"{epoch_seconds:.2f}s"
        f"{improvement_marker}"
    )

    if (
        epochs_without_improvement
        >= OUTPERFORM_TRAINING_CONFIG[
            "early_stopping_patience"
        ]
    ):
        print(
            "\nEarly stopping triggered after "
            f"{epochs_without_improvement} epochs "
            "without sufficient validation AUC improvement."
        )
        break


training_duration_seconds = (
    time.perf_counter() - training_start_time
)

if best_model_state is None:
    raise RuntimeError(
        "Training finished without recording a valid model."
    )


# ---------------------------------------------------------
# Restore the best validation model
# ---------------------------------------------------------

outperform_mlp.load_state_dict(
    best_model_state
)

outperform_mlp.to(DEVICE)
outperform_mlp.eval()


# ---------------------------------------------------------
# Evaluate only train and validation
# ---------------------------------------------------------

best_train_metrics, _, outperform_train_probabilities = (
    evaluate_binary_model(
        model=outperform_mlp,
        data_loader=outperform_train_eval_loader,
        loss_function=outperform_loss_function,
        device=DEVICE,
        threshold=0.50,
    )
)

best_valid_metrics_recomputed, (
    outperform_valid_targets
), outperform_valid_probabilities = evaluate_binary_model(
    model=outperform_mlp,
    data_loader=outperform_valid_loader,
    loss_function=outperform_loss_function,
    device=DEVICE,
    threshold=0.50,
)

if not np.isclose(
    best_valid_metrics_recomputed["roc_auc"],
    best_valid_roc_auc,
    atol=1e-10,
):
    raise ValueError(
        "Restored model validation ROC-AUC does not match "
        "the recorded best result."
    )


# ---------------------------------------------------------
# Save research checkpoint and training history
# ---------------------------------------------------------

training_history_df = pd.DataFrame(
    training_history
)

training_history_df.to_csv(
    OUTPERFORM_HISTORY_PATH,
    index=False,
)

checkpoint = {
    "model_name": "tabular_mlp_outperform",
    "model_version": "v1",
    "target": OUTPERFORM_TARGET,
    "split_method": PRIMARY_SPLIT_METHOD,
    "random_seed": RANDOM_SEED,
    "model_config": MLP_CONFIG,
    "training_config": OUTPERFORM_TRAINING_CONFIG,
    "feature_cols": feature_cols,
    "best_epoch": best_epoch,
    "best_valid_metrics": best_valid_metrics_recomputed,
    "training_duration_seconds": training_duration_seconds,
    "state_dict": best_model_state,
}

torch.save(
    checkpoint,
    OUTPERFORM_CHECKPOINT_PATH,
)


# ---------------------------------------------------------
# Display training result
# ---------------------------------------------------------

outperform_training_summary = pd.DataFrame(
    [
        {
            "split": "train",
            **best_train_metrics,
        },
        {
            "split": "valid",
            **best_valid_metrics_recomputed,
        },
    ]
)

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print(
    "Best validation ROC-AUC:",
    round(best_valid_roc_auc, 6),
)
print(
    "Total training time:",
    round(training_duration_seconds, 2),
    "seconds",
)
print("Checkpoint:", OUTPERFORM_CHECKPOINT_PATH)
print("Training history:", OUTPERFORM_HISTORY_PATH)

print(
    "\nThe test set has not been evaluated."
)

display(
    outperform_training_summary[
        [
            "split",
            "rows",
            "loss",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
        ]
    ]
)

Epoch 01 | train loss 0.709058 | valid loss 0.696950 | valid AUC 0.506593 | valid AP 0.512857 | LR 1.00e-03 | 3.34s  <-- best
Epoch 02 | train loss 0.692456 | valid loss 0.695479 | valid AUC 0.509964 | valid AP 0.516603 | LR 1.00e-03 | 3.25s  <-- best
Epoch 03 | train loss 0.687000 | valid loss 0.694032 | valid AUC 0.523855 | valid AP 0.525466 | LR 1.00e-03 | 3.31s  <-- best
Epoch 04 | train loss 0.683562 | valid loss 0.693764 | valid AUC 0.530320 | valid AP 0.528792 | LR 1.00e-03 | 3.09s  <-- best
Epoch 05 | train loss 0.680341 | valid loss 0.696663 | valid AUC 0.522457 | valid AP 0.522481 | LR 1.00e-03 | 3.28s
Epoch 06 | train loss 0.676924 | valid loss 0.698681 | valid AUC 0.524270 | valid AP 0.524400 | LR 1.00e-03 | 3.01s
Epoch 07 | train loss 0.674131 | valid loss 0.701124 | valid AUC 0.523705 | valid AP 0.520158 | LR 1.00e-03 | 3.19s
Epoch 08 | train loss 0.671216 | valid loss 0.702846 | valid AUC 0.518918 | valid AP 0.517817 | LR 5.00e-04 | 3.16s
Epoch 09 | train loss 0.670074 |

,split,rows,loss,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1
0,train,251738,0.671995,0.618407,0.624165,0.671995,0.239651,0.581708,0.579828,0.581005,0.657209,0.616762
1,valid,19980,0.693764,0.530320,0.528792,0.693764,0.250238,0.524274,0.522063,0.529677,0.620264,0.571403


## Outperformance MLP — Initial Training Result

The first tabular MLP was trained on the purged chronological split.

### Architecture

- Input features: 79
- Hidden layers: 256, 128, 64
- Batch normalization after each hidden linear layer
- ReLU activations
- Dropout rates: 0.20, 0.15, 0.10
- Trainable parameters: 62,593
- Loss: binary cross-entropy with logits
- Optimizer: AdamW
- Model selection metric: validation ROC-AUC

### Best Epoch

The highest validation ROC-AUC occurred at epoch 4.

| Split | ROC-AUC | Average Precision | Loss | Brier Score |
|---|---:|---:|---:|---:|
| Train | 0.6184 | 0.6242 | 0.6720 | 0.2397 |
| Validation | 0.5303 | 0.5288 | 0.6938 | 0.2502 |

### Interpretation

The training ROC-AUC reached 0.6184, while validation ROC-AUC reached only
0.5303.

Validation performance improved during the first four epochs and then
deteriorated even though training loss continued to decrease.

This indicates overfitting.

The best validation checkpoint was restored and saved. Later epochs were not
used for evaluation.

The untouched test set has not been evaluated.

### Current Decision

This result is not sufficient to promote the MLP.

A fair comparison requires retraining the Random Forest baseline on the same
purged training population and evaluating it on the same purged validation
period.

Only after comparing models on identical rows should the experiment proceed to
the untouched test set.

### retrain the Random Forest on the purged split

In [8]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


# ---------------------------------------------------------
# Purged Random Forest baseline
# ---------------------------------------------------------

PURGED_RF_OUTPERFORM_PATH = (
    MODEL_DIR
    / "random_forest_outperform_purged_research_v1.joblib"
)

PURGED_RF_OUTPERFORM_REPORT_PATH = (
    REPORT_DIR
    / "random_forest_outperform_purged_validation_v1.csv"
)


purged_rf_outperform = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_split=200,
                min_samples_leaf=400,
                max_features=0.7,
                max_depth=10,
                class_weight=None,
                bootstrap=True,
                random_state=RANDOM_SEED,
                n_jobs=15,
                verbose=0,
            ),
        ),
    ]
)


# ---------------------------------------------------------
# Fit using the purged training split only
# ---------------------------------------------------------

rf_training_start = time.perf_counter()

purged_rf_outperform.fit(
    X_train_raw,
    y_train_outperform.astype(np.int64),
)

rf_training_seconds = (
    time.perf_counter() - rf_training_start
)


# ---------------------------------------------------------
# Evaluate train and validation only
# ---------------------------------------------------------

purged_rf_train_probability = (
    purged_rf_outperform.predict_proba(
        X_train_raw
    )[:, 1]
)

purged_rf_valid_probability = (
    purged_rf_outperform.predict_proba(
        X_valid_raw
    )[:, 1]
)

purged_rf_train_metrics = calculate_binary_metrics(
    y_true=y_train_outperform,
    y_probability=purged_rf_train_probability,
    threshold=0.50,
)

purged_rf_valid_metrics = calculate_binary_metrics(
    y_true=y_valid_outperform,
    y_probability=purged_rf_valid_probability,
    threshold=0.50,
)


# ---------------------------------------------------------
# Compare Random Forest and MLP
# ---------------------------------------------------------

purged_validation_comparison = pd.DataFrame(
    [
        {
            "model": "random_forest",
            "split": "train",
            **purged_rf_train_metrics,
        },
        {
            "model": "random_forest",
            "split": "valid",
            **purged_rf_valid_metrics,
        },
        {
            "model": "tabular_mlp",
            "split": "train",
            **best_train_metrics,
        },
        {
            "model": "tabular_mlp",
            "split": "valid",
            **best_valid_metrics_recomputed,
        },
    ]
)

validation_only_comparison = (
    purged_validation_comparison
    .loc[
        purged_validation_comparison["split"].eq("valid")
    ]
    .sort_values(
        "roc_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Save the purged research baseline
# ---------------------------------------------------------

joblib.dump(
    purged_rf_outperform,
    PURGED_RF_OUTPERFORM_PATH,
)

purged_validation_comparison.to_csv(
    PURGED_RF_OUTPERFORM_REPORT_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print(
    "Purged Random Forest training time:",
    round(rf_training_seconds, 2),
    "seconds",
)

print(
    "Random Forest validation ROC-AUC:",
    round(purged_rf_valid_metrics["roc_auc"], 6),
)

print(
    "MLP validation ROC-AUC:",
    round(best_valid_metrics_recomputed["roc_auc"], 6),
)

print("\nThe test set remains untouched.")

display(
    validation_only_comparison[
        [
            "model",
            "rows",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "average_probability",
        ]
    ]
)

Purged Random Forest training time: 157.79 seconds
Random Forest validation ROC-AUC: 0.528908
MLP validation ROC-AUC: 0.53032

The test set remains untouched.


,model,rows,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1,average_probability
0,tabular_mlp,19980,0.530320,0.528792,0.693764,0.250238,0.524274,0.522063,0.529677,0.620264,0.571403,0.510942
1,random_forest,19980,0.528908,0.530654,0.691974,0.249410,0.522573,0.519090,0.525825,0.673715,0.590654,0.511692


The MLP’s AUC advantage is only about 0.14 percentage points, while the Random Forest is slightly better on probability quality and average precision. This is a mixed result, not evidence that the MLP is superior.

 First measure whether the tiny validation differences are stable using a paired bootstrap on the exact same validation rows.

In [9]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from tqdm.auto import tqdm


# ---------------------------------------------------------
# Paired bootstrap comparison on validation rows
# ---------------------------------------------------------

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 42

validation_targets = np.asarray(
    y_valid_outperform,
    dtype=np.int64,
)

mlp_validation_probability = np.asarray(
    outperform_valid_probabilities,
    dtype=np.float64,
)

rf_validation_probability = np.asarray(
    purged_rf_valid_probability,
    dtype=np.float64,
)

if not (
    len(validation_targets)
    == len(mlp_validation_probability)
    == len(rf_validation_probability)
):
    raise ValueError(
        "Validation target and probability lengths do not match."
    )


def calculate_probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict[str, float]:
    """Calculate threshold-independent probability metrics."""

    clipped_probability = np.clip(
        probability,
        1e-7,
        1 - 1e-7,
    )

    return {
        "roc_auc": roc_auc_score(
            y_true,
            probability,
        ),
        "average_precision": average_precision_score(
            y_true,
            probability,
        ),
        "log_loss": log_loss(
            y_true,
            clipped_probability,
            labels=[0, 1],
        ),
        "brier_score": brier_score_loss(
            y_true,
            probability,
        ),
    }


rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

row_count = len(validation_targets)

bootstrap_rows = []

for iteration in tqdm(
    range(BOOTSTRAP_ITERATIONS),
    desc="Paired validation bootstrap",
    unit="sample",
    dynamic_ncols=True,
):
    sampled_indices = rng.integers(
        low=0,
        high=row_count,
        size=row_count,
    )

    sampled_targets = validation_targets[
        sampled_indices
    ]

    # Extremely unlikely with this dataset, but skip samples
    # containing only one target class.
    if np.unique(sampled_targets).size < 2:
        continue

    mlp_metrics = calculate_probability_metrics(
        y_true=sampled_targets,
        probability=mlp_validation_probability[
            sampled_indices
        ],
    )

    rf_metrics = calculate_probability_metrics(
        y_true=sampled_targets,
        probability=rf_validation_probability[
            sampled_indices
        ],
    )

    bootstrap_rows.append(
        {
            "iteration": iteration + 1,
            "roc_auc_difference_mlp_minus_rf": (
                mlp_metrics["roc_auc"]
                - rf_metrics["roc_auc"]
            ),
            "average_precision_difference_mlp_minus_rf": (
                mlp_metrics["average_precision"]
                - rf_metrics["average_precision"]
            ),
            "log_loss_difference_mlp_minus_rf": (
                mlp_metrics["log_loss"]
                - rf_metrics["log_loss"]
            ),
            "brier_difference_mlp_minus_rf": (
                mlp_metrics["brier_score"]
                - rf_metrics["brier_score"]
            ),
        }
    )

bootstrap_comparison = pd.DataFrame(
    bootstrap_rows
)


# ---------------------------------------------------------
# Summarize confidence intervals
# ---------------------------------------------------------

summary_rows = []

metric_rules = {
    "roc_auc_difference_mlp_minus_rf": "higher_is_better",
    "average_precision_difference_mlp_minus_rf": "higher_is_better",
    "log_loss_difference_mlp_minus_rf": "lower_is_better",
    "brier_difference_mlp_minus_rf": "lower_is_better",
}

for metric, direction in metric_rules.items():
    values = bootstrap_comparison[
        metric
    ].to_numpy()

    ci_lower, ci_upper = np.quantile(
        values,
        [0.025, 0.975],
    )

    mean_difference = values.mean()

    if direction == "higher_is_better":
        mlp_win_rate = np.mean(values > 0)
    else:
        mlp_win_rate = np.mean(values < 0)

    summary_rows.append(
        {
            "metric": metric,
            "mean_difference_mlp_minus_rf": (
                mean_difference
            ),
            "ci_2_5_pct": ci_lower,
            "ci_97_5_pct": ci_upper,
            "mlp_bootstrap_win_rate": mlp_win_rate,
            "ci_excludes_zero": bool(
                (ci_lower > 0)
                or (ci_upper < 0)
            ),
            "better_direction": direction,
        }
    )

bootstrap_summary = pd.DataFrame(
    summary_rows
)


# ---------------------------------------------------------
# Save results
# ---------------------------------------------------------

BOOTSTRAP_DETAIL_PATH = (
    REPORT_DIR
    / "outperform_mlp_vs_rf_validation_bootstrap_v1.csv"
)

BOOTSTRAP_SUMMARY_PATH = (
    REPORT_DIR
    / "outperform_mlp_vs_rf_validation_bootstrap_summary_v1.csv"
)

bootstrap_comparison.to_csv(
    BOOTSTRAP_DETAIL_PATH,
    index=False,
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print(
    "Completed bootstrap samples:",
    len(bootstrap_comparison),
)

print(
    "\nPositive AUC/AP difference means the MLP is better."
)

print(
    "Negative log-loss/Brier difference means the MLP is better."
)

display(bootstrap_summary)

Paired validation bootstrap:   0%|          | 0/2000 [00:00<?, ?sample/s]

Completed bootstrap samples: 2000

Positive AUC/AP difference means the MLP is better.
Negative log-loss/Brier difference means the MLP is better.


,metric,mean_difference_mlp_minus_rf,ci_2_5_pct,ci_97_5_pct,mlp_bootstrap_win_rate,ci_excludes_zero,better_direction
0,roc_auc_difference_mlp_minus_rf,0.001501,-0.006593,0.009677,0.6420,False,higher_is_better
1,average_precision_difference_mlp_minus_rf,-0.001844,-0.009558,0.005896,0.3240,False,higher_is_better
2,log_loss_difference_mlp_minus_rf,0.001776,0.000182,0.003309,0.0105,True,lower_is_better
3,brier_difference_mlp_minus_rf,0.000822,0.000047,0.001567,0.0170,True,lower_is_better


## Paired Bootstrap Comparison: MLP versus Random Forest

The MLP and Random Forest were compared using 2,000 paired bootstrap samples
from the same purged validation rows.

Using paired samples ensures that both models are evaluated on exactly the same
resampled observations during every bootstrap iteration.

### Results

| Metric | Mean MLP − RF Difference | 95% Confidence Interval | MLP Win Rate | Reliable Difference |
|---|---:|---:|---:|---|
| ROC-AUC | +0.0015 | −0.0066 to +0.0097 | 64.20% | No |
| Average precision | −0.0018 | −0.0096 to +0.0059 | 32.40% | No |
| Log loss | +0.0018 | +0.0002 to +0.0033 | 1.05% | Yes |
| Brier score | +0.0008 | +0.00005 to +0.0016 | 1.70% | Yes |

For ROC-AUC and average precision, higher values are better.

For log loss and Brier score, lower values are better.

### Interpretation

The confidence intervals for ROC-AUC and average precision include zero.
Therefore, the small ranking differences between the MLP and Random Forest are
not reliable.

The confidence intervals for log loss and Brier score are entirely above zero.
Because these metrics are calculated as MLP minus Random Forest and lower values
are better, the result shows that the Random Forest produces reliably better
probabilities.

The MLP overfits the training data and does not demonstrate a reliable
validation-ranking improvement.

### Decision

The initial tabular MLP is not eligible to replace the Random Forest baseline.

However, the models may still contain partially different information.
A probability ensemble will therefore be evaluated before rejecting this MLP
completely.

The untouched test period remains unused.

### validation-only ensemble test

In [10]:
# ---------------------------------------------------------
# Validation-only Random Forest and MLP ensemble analysis
# ---------------------------------------------------------

ENSEMBLE_WEIGHTS = np.arange(
    0.0,
    1.01,
    0.05,
)

ensemble_rows = []

for mlp_weight in ENSEMBLE_WEIGHTS:
    rf_weight = 1.0 - mlp_weight

    ensemble_probability = (
        mlp_weight * mlp_validation_probability
        + rf_weight * rf_validation_probability
    )

    ensemble_metrics = calculate_binary_metrics(
        y_true=validation_targets,
        y_probability=ensemble_probability,
        threshold=0.50,
    )

    ensemble_rows.append(
        {
            "mlp_weight": mlp_weight,
            "random_forest_weight": rf_weight,
            **ensemble_metrics,
        }
    )

ensemble_validation_results = (
    pd.DataFrame(ensemble_rows)
    .sort_values(
        [
            "roc_auc",
            "average_precision",
            "log_loss",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Identify reference and best ensemble candidates
# ---------------------------------------------------------

best_auc_ensemble = (
    ensemble_validation_results.iloc[0].copy()
)

equal_weight_ensemble = (
    ensemble_validation_results.loc[
        np.isclose(
            ensemble_validation_results["mlp_weight"],
            0.50,
        )
    ]
    .iloc[0]
    .copy()
)

rf_only_result = (
    ensemble_validation_results.loc[
        np.isclose(
            ensemble_validation_results["mlp_weight"],
            0.0,
        )
    ]
    .iloc[0]
    .copy()
)

mlp_only_result = (
    ensemble_validation_results.loc[
        np.isclose(
            ensemble_validation_results["mlp_weight"],
            1.0,
        )
    ]
    .iloc[0]
    .copy()
)


# ---------------------------------------------------------
# Probability correlation
# ---------------------------------------------------------

probability_correlation = np.corrcoef(
    mlp_validation_probability,
    rf_validation_probability,
)[0, 1]


# ---------------------------------------------------------
# Save ensemble results
# ---------------------------------------------------------

ENSEMBLE_VALIDATION_PATH = (
    REPORT_DIR
    / "outperform_mlp_rf_ensemble_validation_v1.csv"
)

ensemble_validation_results.to_csv(
    ENSEMBLE_VALIDATION_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display comparison
# ---------------------------------------------------------

ensemble_candidate_summary = pd.DataFrame(
    [
        {
            "candidate": "random_forest_only",
            **rf_only_result.to_dict(),
        },
        {
            "candidate": "equal_weight_ensemble",
            **equal_weight_ensemble.to_dict(),
        },
        {
            "candidate": "best_validation_auc_ensemble",
            **best_auc_ensemble.to_dict(),
        },
        {
            "candidate": "mlp_only",
            **mlp_only_result.to_dict(),
        },
    ]
)

print(
    "MLP and Random Forest probability correlation:",
    round(float(probability_correlation), 6),
)

print(
    "\nBest validation ensemble weights:"
)

print(
    "MLP:",
    round(float(best_auc_ensemble["mlp_weight"]), 2),
)

print(
    "Random Forest:",
    round(
        float(best_auc_ensemble["random_forest_weight"]),
        2,
    ),
)

print("\nThe test set remains untouched.")

display(
    ensemble_candidate_summary[
        [
            "candidate",
            "mlp_weight",
            "random_forest_weight",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
        ]
    ]
)

print("\nTop 10 validation ensembles by ROC-AUC:")

display(
    ensemble_validation_results[
        [
            "mlp_weight",
            "random_forest_weight",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
        ]
    ].head(10)
)

MLP and Random Forest probability correlation: 0.489296

Best validation ensemble weights:
MLP: 0.3
Random Forest: 0.7

The test set remains untouched.


,candidate,mlp_weight,random_forest_weight,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1
0,random_forest_only,0.0,1.0,0.528908,0.530654,0.691974,0.249410,0.519090,0.525825,0.673715,0.590654
1,equal_weight_ensemble,0.5,0.5,0.533845,0.530611,0.691219,0.249032,0.526323,0.532007,0.659814,0.589058
2,best_validation_auc_ensemble,0.3,0.7,0.534681,0.531370,0.691136,0.248993,0.522344,0.528481,0.670289,0.590997
3,mlp_only,1.0,0.0,0.530320,0.528792,0.693764,0.250238,0.522063,0.529677,0.620264,0.571403



Top 10 validation ensembles by ROC-AUC:


,mlp_weight,random_forest_weight,roc_auc,average_precision,log_loss,brier_score
0,0.30,0.70,0.534681,0.531370,0.691136,0.248993
1,0.35,0.65,0.534626,0.531207,0.691108,0.248979
2,0.25,0.75,0.534526,0.531505,0.691196,0.249023
3,0.40,0.60,0.534417,0.530981,0.691113,0.248981
4,0.45,0.55,0.534161,0.530823,0.691150,0.248999
5,0.20,0.80,0.534123,0.531627,0.691288,0.249069
6,0.50,0.50,0.533845,0.530611,0.691219,0.249032
7,0.55,0.45,0.533473,0.530429,0.691321,0.249082
8,0.15,0.85,0.533412,0.531721,0.691411,0.249130
9,0.60,0.40,0.533117,0.530278,0.691456,0.249147


Compared with the Random Forest, the selected ensemble improves:

    ROC-AUC by 0.005773
    Average precision by 0.000716
    Log loss by 0.000838
    Brier score by 0.000417

The probability correlation of 0.4893 is moderate, which supports the idea that the two models learned partially different patterns.

The best weights are also inside a broad stable region between roughly 25% and 40% MLP. That is better than seeing a single isolated optimum.

However, we selected the weight using validation data. Before touching the test set, run a 20-trading-day moving-block bootstrap. This is more suitable than independently resampling rows because stocks on the same date share market conditions and the targets overlap across nearby dates.

In [11]:
# ---------------------------------------------------------
# Moving-block bootstrap: fixed ensemble versus RF
# ---------------------------------------------------------

BLOCK_BOOTSTRAP_ITERATIONS = 2000
BLOCK_LENGTH_TRADING_DAYS = 20
BLOCK_BOOTSTRAP_SEED = 42

SELECTED_MLP_WEIGHT = 0.30
SELECTED_RF_WEIGHT = 0.70

selected_ensemble_probability = (
    SELECTED_MLP_WEIGHT * mlp_validation_probability
    + SELECTED_RF_WEIGHT * rf_validation_probability
)


# ---------------------------------------------------------
# Validate probability and metadata alignment
# ---------------------------------------------------------

validation_metadata = (
    model_data.loc[
        valid_mask,
        ["date", "yf_ticker"],
    ]
    .reset_index(drop=True)
)

if not (
    len(validation_metadata)
    == len(validation_targets)
    == len(rf_validation_probability)
    == len(selected_ensemble_probability)
):
    raise ValueError(
        "Validation metadata, targets, and probabilities are not aligned."
    )

validation_dates = pd.DatetimeIndex(
    validation_metadata["date"]
    .drop_duplicates()
    .sort_values()
)

if len(validation_dates) < BLOCK_LENGTH_TRADING_DAYS:
    raise ValueError(
        "Validation period is shorter than the requested block length."
    )

date_to_row_indices = {
    date: group.index.to_numpy()
    for date, group in validation_metadata.groupby(
        "date",
        sort=True,
    )
}

maximum_block_start = (
    len(validation_dates)
    - BLOCK_LENGTH_TRADING_DAYS
)

blocks_needed = int(
    np.ceil(
        len(validation_dates)
        / BLOCK_LENGTH_TRADING_DAYS
    )
)


# ---------------------------------------------------------
# Observed validation differences
# ---------------------------------------------------------

observed_rf_metrics = calculate_probability_metrics(
    y_true=validation_targets,
    probability=rf_validation_probability,
)

observed_ensemble_metrics = calculate_probability_metrics(
    y_true=validation_targets,
    probability=selected_ensemble_probability,
)

observed_differences = {
    "roc_auc_difference_ensemble_minus_rf": (
        observed_ensemble_metrics["roc_auc"]
        - observed_rf_metrics["roc_auc"]
    ),
    "average_precision_difference_ensemble_minus_rf": (
        observed_ensemble_metrics["average_precision"]
        - observed_rf_metrics["average_precision"]
    ),
    "log_loss_difference_ensemble_minus_rf": (
        observed_ensemble_metrics["log_loss"]
        - observed_rf_metrics["log_loss"]
    ),
    "brier_difference_ensemble_minus_rf": (
        observed_ensemble_metrics["brier_score"]
        - observed_rf_metrics["brier_score"]
    ),
}


# ---------------------------------------------------------
# Moving-block bootstrap
# ---------------------------------------------------------

rng = np.random.default_rng(
    BLOCK_BOOTSTRAP_SEED
)

block_bootstrap_rows = []

for iteration in tqdm(
    range(BLOCK_BOOTSTRAP_ITERATIONS),
    desc="20-day block bootstrap",
    unit="sample",
    dynamic_ncols=True,
):
    block_starts = rng.integers(
        low=0,
        high=maximum_block_start + 1,
        size=blocks_needed,
    )

    sampled_date_positions = np.concatenate(
        [
            np.arange(
                start,
                start + BLOCK_LENGTH_TRADING_DAYS,
            )
            for start in block_starts
        ]
    )[:len(validation_dates)]

    sampled_dates = validation_dates[
        sampled_date_positions
    ]

    sampled_row_indices = np.concatenate(
        [
            date_to_row_indices[date]
            for date in sampled_dates
        ]
    )

    sampled_targets = validation_targets[
        sampled_row_indices
    ]

    if np.unique(sampled_targets).size < 2:
        continue

    sampled_rf_probability = (
        rf_validation_probability[
            sampled_row_indices
        ]
    )

    sampled_ensemble_probability = (
        selected_ensemble_probability[
            sampled_row_indices
        ]
    )

    rf_metrics = calculate_probability_metrics(
        y_true=sampled_targets,
        probability=sampled_rf_probability,
    )

    ensemble_metrics = calculate_probability_metrics(
        y_true=sampled_targets,
        probability=sampled_ensemble_probability,
    )

    block_bootstrap_rows.append(
        {
            "iteration": iteration + 1,
            "roc_auc_difference_ensemble_minus_rf": (
                ensemble_metrics["roc_auc"]
                - rf_metrics["roc_auc"]
            ),
            "average_precision_difference_ensemble_minus_rf": (
                ensemble_metrics["average_precision"]
                - rf_metrics["average_precision"]
            ),
            "log_loss_difference_ensemble_minus_rf": (
                ensemble_metrics["log_loss"]
                - rf_metrics["log_loss"]
            ),
            "brier_difference_ensemble_minus_rf": (
                ensemble_metrics["brier_score"]
                - rf_metrics["brier_score"]
            ),
        }
    )

block_bootstrap_results = pd.DataFrame(
    block_bootstrap_rows
)


# ---------------------------------------------------------
# Summarize confidence intervals
# ---------------------------------------------------------

metric_directions = {
    "roc_auc_difference_ensemble_minus_rf": "higher_is_better",
    "average_precision_difference_ensemble_minus_rf": "higher_is_better",
    "log_loss_difference_ensemble_minus_rf": "lower_is_better",
    "brier_difference_ensemble_minus_rf": "lower_is_better",
}

block_summary_rows = []

for metric, direction in metric_directions.items():
    differences = block_bootstrap_results[
        metric
    ].to_numpy()

    ci_lower, ci_upper = np.quantile(
        differences,
        [0.025, 0.975],
    )

    if direction == "higher_is_better":
        ensemble_win_rate = np.mean(
            differences > 0
        )
    else:
        ensemble_win_rate = np.mean(
            differences < 0
        )

    block_summary_rows.append(
        {
            "metric": metric,
            "observed_difference": observed_differences[
                metric
            ],
            "bootstrap_mean_difference": float(
                differences.mean()
            ),
            "ci_2_5_pct": float(ci_lower),
            "ci_97_5_pct": float(ci_upper),
            "ensemble_bootstrap_win_rate": float(
                ensemble_win_rate
            ),
            "ci_excludes_zero": bool(
                (ci_lower > 0)
                or (ci_upper < 0)
            ),
            "better_direction": direction,
        }
    )

ensemble_block_bootstrap_summary = pd.DataFrame(
    block_summary_rows
)


# ---------------------------------------------------------
# Save reports
# ---------------------------------------------------------

ENSEMBLE_BLOCK_DETAIL_PATH = (
    REPORT_DIR
    / "outperform_ensemble_vs_rf_validation_block_bootstrap_v1.csv"
)

ENSEMBLE_BLOCK_SUMMARY_PATH = (
    REPORT_DIR
    / "outperform_ensemble_vs_rf_validation_block_bootstrap_summary_v1.csv"
)

block_bootstrap_results.to_csv(
    ENSEMBLE_BLOCK_DETAIL_PATH,
    index=False,
)

ensemble_block_bootstrap_summary.to_csv(
    ENSEMBLE_BLOCK_SUMMARY_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print(
    "Completed block-bootstrap samples:",
    len(block_bootstrap_results),
)

print(
    "Block length:",
    BLOCK_LENGTH_TRADING_DAYS,
    "trading days",
)

print(
    "Fixed ensemble weights:",
    f"MLP={SELECTED_MLP_WEIGHT:.2f},",
    f"RF={SELECTED_RF_WEIGHT:.2f}",
)

print(
    "\nPositive AUC/AP differences favour the ensemble."
)

print(
    "Negative log-loss/Brier differences favour the ensemble."
)

display(ensemble_block_bootstrap_summary)

20-day block bootstrap:   0%|          | 0/2000 [00:00<?, ?sample/s]

Completed block-bootstrap samples: 2000
Block length: 20 trading days
Fixed ensemble weights: MLP=0.30, RF=0.70

Positive AUC/AP differences favour the ensemble.
Negative log-loss/Brier differences favour the ensemble.


,metric,observed_difference,bootstrap_mean_difference,ci_2_5_pct,ci_97_5_pct,ensemble_bootstrap_win_rate,ci_excludes_zero,better_direction
0,roc_auc_difference_ensemble_minus_rf,0.005774,0.005222,-0.011749,0.023113,0.7060,False,higher_is_better
1,average_precision_difference_ensemble_minus_rf,0.000715,0.002166,-0.014979,0.022752,0.5715,False,higher_is_better
2,log_loss_difference_ensemble_minus_rf,-0.000838,-0.000757,-0.003598,0.001678,0.6985,False,lower_is_better
3,brier_difference_ensemble_minus_rf,-0.000416,-0.000376,-0.001788,0.000833,0.6985,False,lower_is_better


## Moving-Block Bootstrap: Ensemble versus Random Forest

The fixed ensemble was evaluated against the purged Random Forest using a
20-trading-day moving-block bootstrap with 2,000 samples.

The ensemble weights were locked before this analysis:

- MLP weight: 0.30
- Random Forest weight: 0.70

### Why a Block Bootstrap Was Used

Stock-date rows are not fully independent.

Stocks observed on the same date share:

- Market conditions
- NIFTY movements
- India VIX conditions
- Macroeconomic information
- Sector-level shocks

Nearby observations are also related because the prediction target looks
forward 20 trading days.

The moving-block bootstrap therefore samples complete 20-trading-day periods
rather than sampling individual stock rows independently.

### Results

| Metric | Observed Ensemble − RF Difference | 95% Confidence Interval | Ensemble Win Rate |
|---|---:|---:|---:|
| ROC-AUC | +0.0058 | −0.0117 to +0.0231 | 70.60% |
| Average precision | +0.0007 | −0.0150 to +0.0228 | 57.15% |
| Log loss | −0.0008 | −0.0036 to +0.0017 | 69.85% |
| Brier score | −0.0004 | −0.0018 to +0.0008 | 69.85% |

Positive ROC-AUC and average-precision differences favour the ensemble.

Negative log-loss and Brier-score differences favour the ensemble.

### Interpretation

The ensemble performed better than the Random Forest on the observed validation
metrics and in a majority of bootstrap samples.

However, every 95% confidence interval includes zero.

Therefore, the validation evidence is suggestive but not conclusive. The
ensemble improvement may depend on the particular market periods included in
the validation year.

### Decision

The 30% MLP and 70% Random Forest ensemble remains a research candidate.

It is not eligible for production promotion based on the current evidence.

The untouched test set will remain unused while one additional,
pre-specified regularized MLP candidate is evaluated on validation data.

No ensemble-weight changes will be made after this point.

## Final Conclusion — Initial Tabular MLP Experiment

The first deep learning baseline for MarketGuard has now been completed.

The experiment evaluated a standard tabular multilayer perceptron for predicting:

`target_outperform_nifty50_20d`

The target identifies whether a stock outperformed the NIFTY 50 over the
following 20 trading days.

---

## Experiment Setup

The MLP used the same:

- 79 ordered model features
- Modeling population
- Target definition
- Purged chronological split
- Training and validation rows

as the retrained Random Forest research baseline.

### Purged Split

| Split | Rows | Period |
|---|---:|---|
| Train | 251,738 | Through 2023-11-30 |
| Validation | 19,980 | 2024-01-02 through 2024-12-02 |
| Test | 32,135 | 2025-01-01 through 2026-06-16 |

The final 20 trading dates before each split boundary were removed from the
preceding split to prevent forward 20-day target windows from crossing into the
next evaluation period.

The test period was not used during this experiment.

---

## Preprocessing

Preprocessing was fitted using training data only.

The process included:

1. Median imputation
2. Standard scaling
3. Conversion to 32-bit floating-point tensors

The transformed training features had:

- Mean approximately equal to zero
- Standard deviation approximately equal to one
- No missing or infinite values

Validation and test data did not influence preprocessing statistics.

---

## MLP Architecture

The first MLP architecture was:

```text
79 input features
        ↓
256-neuron hidden layer
        ↓
128-neuron hidden layer
        ↓
64-neuron hidden layer
        ↓
Single binary output logit
```

The network included:

- Batch normalization
- ReLU activation
- Dropout
- AdamW optimization
- Gradient clipping
- Learning-rate reduction
- Early stopping using validation ROC-AUC

Total trainable parameters:

`62,593`

Training was performed using the RTX 4070 Laptop GPU with PyTorch CUDA 12.6.

---

## Training Result

Validation performance reached its maximum at epoch 4.

| Split | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Train | 0.6184 | 0.6242 | 0.6720 | 0.2397 |
| Validation | 0.5303 | 0.5288 | 0.6938 | 0.2502 |

After epoch 4:

- Training loss continued decreasing
- Validation loss increased
- Validation ROC-AUC declined

This indicates overfitting.

The epoch-4 checkpoint was restored and retained as the best MLP model.

---

## Fair Random Forest Comparison

The Random Forest was retrained using the exact same purged training and
validation rows.

| Metric | MLP | Random Forest |
|---|---:|---:|
| ROC-AUC | 0.530320 | 0.528908 |
| Average precision | 0.528792 | 0.530654 |
| Log loss | 0.693764 | 0.691974 |
| Brier score | 0.250238 | 0.249410 |
| Balanced accuracy | 0.522063 | 0.519090 |
| F1 score | 0.571403 | 0.590654 |

The MLP produced a small ROC-AUC improvement of approximately 0.0014.

However, the Random Forest produced better:

- Average precision
- Log loss
- Brier score
- F1 score

The result was therefore mixed.

---

## Paired Bootstrap Comparison

A paired bootstrap with 2,000 validation samples was used to estimate the
uncertainty of the MLP-versus-Random-Forest differences.

| Metric | Mean MLP − RF Difference | 95% Confidence Interval |
|---|---:|---:|
| ROC-AUC | +0.0015 | −0.0066 to +0.0097 |
| Average precision | −0.0018 | −0.0096 to +0.0059 |
| Log loss | +0.0018 | +0.0002 to +0.0033 |
| Brier score | +0.0008 | +0.00005 to +0.0016 |

The ROC-AUC and average-precision confidence intervals included zero.

Therefore, the MLP did not demonstrate a reliable ranking improvement.

The log-loss and Brier-score intervals were entirely above zero. Because lower
values are better, this indicates that the Random Forest produced reliably
better probability estimates.

---

## Ensemble Evaluation

Although the MLP did not clearly outperform the Random Forest alone, their
validation probabilities had a correlation of only:

`0.4893`

This suggests that the models learned partially different patterns.

Several weighted probability ensembles were tested on validation data.

The best validation ROC-AUC was obtained using:

```text
30% MLP probability
+
70% Random Forest probability
```

### Ensemble Validation Results

| Model | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Random Forest | 0.528908 | 0.530654 | 0.691974 | 0.249410 |
| MLP | 0.530320 | 0.528792 | 0.693764 | 0.250238 |
| 30% MLP + 70% RF | 0.534681 | 0.531370 | 0.691136 | 0.248993 |

The ensemble improved all four observed validation metrics.

---

## Moving-Block Bootstrap

A 20-trading-day moving-block bootstrap with 2,000 samples was used to account
for:

- Stocks sharing the same market conditions
- Dependence between nearby trading dates
- Overlapping 20-day target windows
- Market-regime variation

The ensemble weights were fixed before running the bootstrap.

| Metric | Ensemble − RF Difference | 95% Confidence Interval | Ensemble Win Rate |
|---|---:|---:|---:|
| ROC-AUC | +0.0058 | −0.0117 to +0.0231 | 70.60% |
| Average precision | +0.0007 | −0.0150 to +0.0228 | 57.15% |
| Log loss | −0.0008 | −0.0036 to +0.0017 | 69.85% |
| Brier score | −0.0004 | −0.0018 to +0.0008 | 69.85% |

The ensemble performed better in a majority of sampled market periods.

However, all four confidence intervals included zero.

The improvement is therefore promising but not statistically reliable.

---

## Final MLP Decision

The initial MLP will not replace the Random Forest production model.

The experiment shows:

1. The MLP learned meaningful training patterns.
2. It overfit quickly.
3. It did not produce a reliable standalone validation improvement.
4. Its probability quality was worse than the Random Forest.
5. It learned partially different information.
6. Its predictions may still be useful as an ensemble component.

The MLP checkpoint and reports will be retained as a deep learning baseline.

The fixed 30% MLP and 70% Random Forest ensemble will remain a research
candidate, but it is not currently eligible for production promotion.

The untouched test period remains unused.

---

## Next Research Stage

The next model will be a Tabular ResNet.

The Tabular ResNet will use residual connections to support deeper feature
representations while improving gradient flow and model stability.

The planned deep learning sequence is:

```text
Standard MLP
    Completed

Tabular ResNet
    Next

FT-Transformer
    Planned

Optional TabNet
    Later

Model and ensemble comparison
    After all validation experiments

Untouched test evaluation
    Only after final candidate selection
```

The goal of the next stage is to determine whether a more structured neural
architecture can improve generalization without increasing the
train-validation overfitting gap.

# Tabular ResNet Experiment

## Purpose

The standard MLP experiment showed that a neural network can learn information
that is partially different from the Random Forest, but it also overfit quickly
and did not provide a reliable standalone validation improvement.

The next experiment uses a Tabular ResNet.

A Tabular ResNet is a feed-forward neural network with residual connections.
Instead of forcing every layer to completely replace the previous
representation, each residual block learns a refinement that is added back to
its input.

```text
Input representation
        ↓
Residual transformation
        ↓
Input + transformation
        ↓
Updated representation
```

This can make deeper tabular networks easier to train and can improve gradient
flow.

---

## Why Use a Tabular ResNet?

The first MLP used a simple sequential architecture:

```text
79 features
    ↓
256 neurons
    ↓
128 neurons
    ↓
64 neurons
    ↓
Prediction
```

Every hidden layer replaced the previous representation.

The Tabular ResNet will instead project the 79 features into a fixed-width
hidden representation and repeatedly refine it using residual blocks.

```text
79 features
    ↓
Input projection
    ↓
Residual block 1
    ↓
Residual block 2
    ↓
Residual block 3
    ↓
Prediction head
```

Each residual block will follow this structure:

```text
Input
  │
  ├───────────────────────────────┐
  │                               │
  ↓                               │
Normalization                     │
  ↓                               │
Linear layer                      │
  ↓                               │
Activation                        │
  ↓                               │
Dropout                           │
  ↓                               │
Linear layer                      │
  ↓                               │
Dropout                           │
  │                               │
  └──────── Add original input ───┘
                  ↓
          Updated representation
```

---

## Experimental Controls

The Tabular ResNet will use the same approved experiment contract as the MLP:

- Same 79 ordered features
- Same purged training rows
- Same purged validation rows
- Same untouched test rows
- Same training-only median imputation
- Same training-only standard scaling
- Same target definition
- Same evaluation metrics
- Same Random Forest benchmark

This ensures that any performance difference is caused by the model
architecture rather than a different dataset or preprocessing pipeline.

---

## Initial ResNet Configuration

The first configuration will be intentionally moderate:

| Setting | Value |
|---|---:|
| Input features | 79 |
| Hidden width | 192 |
| Residual blocks | 3 |
| Expansion width | 384 |
| Dropout | 0.20 |
| Activation | ReLU |
| Output | One binary logit |

The network will be larger and structurally more advanced than the initial MLP,
but it will remain small enough for controlled experimentation.

---

## Training Strategy

The Tabular ResNet will use:

- Binary cross-entropy with logits
- AdamW optimizer
- Weight decay
- Gradient clipping
- Learning-rate reduction
- Early stopping using validation ROC-AUC
- Best-checkpoint restoration
- CUDA training on the RTX 4070

The test set will remain untouched.

---

## Evaluation Plan

The Tabular ResNet will first be compared on the purged validation split against:

1. Random Forest
2. Initial MLP
3. The existing 30% MLP and 70% Random Forest ensemble

The main validation metrics will be:

- ROC-AUC
- Average precision
- Log loss
- Brier score
- Balanced accuracy
- Precision
- Recall
- F1 score

A paired bootstrap and moving-block bootstrap will only be used when the
Tabular ResNet produces a meaningful observed improvement.

---

## Decision Rule

The Tabular ResNet will not be considered better merely because it has:

- Lower training loss
- Higher training ROC-AUC
- More parameters
- A slightly higher single validation metric

It must demonstrate improved out-of-time validation behavior without materially
worsening probability quality.

The untouched test period will only be evaluated after the ResNet,
FT-Transformer, and final ensemble candidates have been selected.

In [12]:
# ---------------------------------------------------------
# Tabular ResNet configuration
# ---------------------------------------------------------

TABULAR_RESNET_CONFIG = {
    "input_dim": len(feature_cols),
    "hidden_dim": 192,
    "expansion_dim": 384,
    "num_blocks": 3,
    "dropout_rate": 0.20,
    "output_dim": 1,
}


class TabularResidualBlock(nn.Module):
    """
    Pre-normalized residual block for tabular data.

    The block learns a transformation of the current hidden
    representation and adds it back to the original input.
    """

    def __init__(
        self,
        hidden_dim: int,
        expansion_dim: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()

        if hidden_dim <= 0:
            raise ValueError(
                f"hidden_dim must be positive, found {hidden_dim}."
            )

        if expansion_dim < hidden_dim:
            raise ValueError(
                "expansion_dim must be greater than or equal "
                "to hidden_dim."
            )

        if not 0 <= dropout_rate < 1:
            raise ValueError(
                f"Invalid dropout rate: {dropout_rate}"
            )

        self.normalization = nn.LayerNorm(
            hidden_dim
        )

        self.residual_network = nn.Sequential(
            nn.Linear(
                hidden_dim,
                expansion_dim,
            ),
            nn.ReLU(),
            nn.Dropout(
                dropout_rate
            ),
            nn.Linear(
                expansion_dim,
                hidden_dim,
            ),
            nn.Dropout(
                dropout_rate
            ),
        )

    def forward(
        self,
        hidden: torch.Tensor,
    ) -> torch.Tensor:
        """Return the input plus its learned residual transformation."""

        residual = self.residual_network(
            self.normalization(hidden)
        )

        return hidden + residual


class TabularResNet(nn.Module):
    """
    Residual neural network for standardized tabular features.

    The network returns one raw binary-classification logit
    for each input row.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 192,
        expansion_dim: int = 384,
        num_blocks: int = 3,
        dropout_rate: float = 0.20,
    ) -> None:
        super().__init__()

        if input_dim <= 0:
            raise ValueError(
                f"input_dim must be positive, found {input_dim}."
            )

        if num_blocks <= 0:
            raise ValueError(
                f"num_blocks must be positive, found {num_blocks}."
            )

        self.input_projection = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.ReLU(),
        )

        self.residual_blocks = nn.Sequential(
            *[
                TabularResidualBlock(
                    hidden_dim=hidden_dim,
                    expansion_dim=expansion_dim,
                    dropout_rate=dropout_rate,
                )
                for _ in range(num_blocks)
            ]
        )

        self.output_head = nn.Sequential(
            nn.LayerNorm(
                hidden_dim
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                1,
            ),
        )

        self._initialize_weights()

    def _initialize_weights(self) -> None:
        """Initialize linear layers for ReLU-based training."""

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        features: torch.Tensor,
    ) -> torch.Tensor:
        """
        Return one raw logit per row.

        Output shape:
            [batch_size]
        """

        hidden = self.input_projection(
            features
        )

        hidden = self.residual_blocks(
            hidden
        )

        logits = self.output_head(
            hidden
        )

        return logits.squeeze(-1)


# ---------------------------------------------------------
# Reset seed before model initialization
# ---------------------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


# ---------------------------------------------------------
# Instantiate the ResNet
# ---------------------------------------------------------

outperform_resnet = TabularResNet(
    input_dim=TABULAR_RESNET_CONFIG["input_dim"],
    hidden_dim=TABULAR_RESNET_CONFIG["hidden_dim"],
    expansion_dim=TABULAR_RESNET_CONFIG["expansion_dim"],
    num_blocks=TABULAR_RESNET_CONFIG["num_blocks"],
    dropout_rate=TABULAR_RESNET_CONFIG["dropout_rate"],
).to(DEVICE)

resnet_parameter_counts = count_model_parameters(
    outperform_resnet
)


# ---------------------------------------------------------
# CUDA forward-pass validation
# ---------------------------------------------------------

resnet_dry_run_features = (
    sample_features[:32]
    .to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )
)

outperform_resnet.eval()

with torch.inference_mode():
    resnet_dry_run_logits = outperform_resnet(
        resnet_dry_run_features
    )

    resnet_dry_run_probabilities = torch.sigmoid(
        resnet_dry_run_logits
    )

if resnet_dry_run_logits.shape != torch.Size([32]):
    raise ValueError(
        "Unexpected ResNet output shape: "
        f"{resnet_dry_run_logits.shape}"
    )

if not torch.isfinite(
    resnet_dry_run_logits
).all():
    raise ValueError(
        "The ResNet forward pass produced non-finite logits."
    )

if not (
    (resnet_dry_run_probabilities >= 0)
    & (resnet_dry_run_probabilities <= 1)
).all():
    raise ValueError(
        "The ResNet probabilities are outside [0, 1]."
    )

outperform_resnet.train()


# ---------------------------------------------------------
# Display architecture audit
# ---------------------------------------------------------

print(outperform_resnet)

print(
    "\nInput features:",
    TABULAR_RESNET_CONFIG["input_dim"],
)

print(
    "Hidden width:",
    TABULAR_RESNET_CONFIG["hidden_dim"],
)

print(
    "Expansion width:",
    TABULAR_RESNET_CONFIG["expansion_dim"],
)

print(
    "Residual blocks:",
    TABULAR_RESNET_CONFIG["num_blocks"],
)

print(
    "Dropout rate:",
    TABULAR_RESNET_CONFIG["dropout_rate"],
)

print(
    "Total parameters:",
    f"{resnet_parameter_counts['total_parameters']:,}",
)

print(
    "Trainable parameters:",
    f"{resnet_parameter_counts['trainable_parameters']:,}",
)

print(
    "\nDry-run device:",
    resnet_dry_run_logits.device,
)

print(
    "Dry-run logits shape:",
    resnet_dry_run_logits.shape,
)

print(
    "Initial probability range:",
    round(
        float(
            resnet_dry_run_probabilities.min().item()
        ),
        6,
    ),
    "to",
    round(
        float(
            resnet_dry_run_probabilities.max().item()
        ),
        6,
    ),
)

if DEVICE.type == "cuda":
    print(
        "Allocated GPU memory:",
        round(
            torch.cuda.memory_allocated()
            / 1024**2,
            2,
        ),
        "MB",
    )

TabularResNet(
  (input_projection): Sequential(
    (0): Linear(in_features=79, out_features=192, bias=True)
    (1): LayerNorm((192,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): ReLU()
  )
  (residual_blocks): Sequential(
    (0): TabularResidualBlock(
      (normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True, bias=True)
      (residual_network): Sequential(
        (0): Linear(in_features=192, out_features=384, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=384, out_features=192, bias=True)
        (4): Dropout(p=0.2, inplace=False)
      )
    )
    (1): TabularResidualBlock(
      (normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True, bias=True)
      (residual_network): Sequential(
        (0): Linear(in_features=192, out_features=384, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=384, out_features=192, bias=True)
  

The ResNet forward pass is valid.

Compared with the MLP, this model has substantially more capacity:

    MLP parameters:       62,593
    ResNet parameters:   461,569

In [13]:
from tqdm.auto import tqdm


def train_one_epoch_with_progress(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_function,
    device: torch.device,
    epoch: int,
    max_gradient_norm: float = 5.0,
) -> float:
    """Train one epoch while displaying batch-level progress."""

    if max_gradient_norm <= 0:
        raise ValueError(
            "max_gradient_norm must be positive."
        )

    model.train()

    total_loss = 0.0
    total_rows = 0

    batch_progress = tqdm(
        data_loader,
        desc=f"Epoch {epoch:02d} batches",
        total=len(data_loader),
        unit="batch",
        leave=False,
        dynamic_ncols=True,
    )

    for batch_number, (
        feature_batch,
        target_batch,
    ) in enumerate(batch_progress, start=1):

        feature_batch = feature_batch.to(
            device,
            non_blocking=PIN_MEMORY,
        )

        target_batch = target_batch.to(
            device,
            non_blocking=PIN_MEMORY,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(feature_batch)

        if logits.shape != target_batch.shape:
            raise ValueError(
                "Logit and target shapes do not match: "
                f"{logits.shape} versus {target_batch.shape}"
            )

        loss = loss_function(
            logits,
            target_batch,
        )

        if not torch.isfinite(loss):
            raise ValueError(
                "A non-finite training loss was produced."
            )

        loss.backward()

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_gradient_norm,
            error_if_nonfinite=True,
        )

        optimizer.step()

        batch_rows = len(target_batch)

        total_loss += (
            float(loss.item()) * batch_rows
        )

        total_rows += batch_rows

        running_loss = (
            total_loss / total_rows
        )

        if (
            batch_number % 5 == 0
            or batch_number == len(data_loader)
        ):
            batch_progress.set_postfix(
                {
                    "loss": f"{running_loss:.5f}",
                    "grad": f"{float(gradient_norm):.3f}",
                    "lr": (
                        f"{optimizer.param_groups[0]['lr']:.2e}"
                    ),
                }
            )

    if total_rows == 0:
        raise ValueError(
            "The training data loader contains no rows."
        )

    return total_loss / total_rows


print("Progress-enabled training function is ready.")

Progress-enabled training function is ready.


In [14]:
# ---------------------------------------------------------
# Tabular ResNet training configuration
# ---------------------------------------------------------

RESNET_TRAINING_CONFIG = {
    "max_epochs": 50,
    "learning_rate": 5e-4,
    "weight_decay": 5e-4,
    "early_stopping_patience": 15,
    "minimum_auc_improvement": 1e-4,
    "gradient_clip_norm": 5.0,
    "scheduler_factor": 0.50,
    "scheduler_patience": 2,
    "minimum_learning_rate": 1e-5,
}

RESNET_CHECKPOINT_PATH = (
    MODEL_DIR
    / "tabular_resnet_outperform_purged_v1.pt"
)

RESNET_HISTORY_PATH = (
    REPORT_DIR
    / "tabular_resnet_outperform_training_history_v1.csv"
)


# ---------------------------------------------------------
# Reset reproducibility state
# ---------------------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# Recreate the shuffled loader so its random state starts fresh.
resnet_train_loader = create_data_loader(
    dataset=outperform_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED,
)

resnet_train_eval_loader = create_data_loader(
    dataset=outperform_train_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)


# ---------------------------------------------------------
# Create a fresh ResNet
# ---------------------------------------------------------

outperform_resnet = TabularResNet(
    input_dim=TABULAR_RESNET_CONFIG["input_dim"],
    hidden_dim=TABULAR_RESNET_CONFIG["hidden_dim"],
    expansion_dim=TABULAR_RESNET_CONFIG["expansion_dim"],
    num_blocks=TABULAR_RESNET_CONFIG["num_blocks"],
    dropout_rate=TABULAR_RESNET_CONFIG["dropout_rate"],
).to(DEVICE)

resnet_loss_function = nn.BCEWithLogitsLoss()

resnet_optimizer = torch.optim.AdamW(
    outperform_resnet.parameters(),
    lr=RESNET_TRAINING_CONFIG["learning_rate"],
    weight_decay=RESNET_TRAINING_CONFIG["weight_decay"],
)

resnet_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    resnet_optimizer,
    mode="max",
    factor=RESNET_TRAINING_CONFIG["scheduler_factor"],
    patience=RESNET_TRAINING_CONFIG["scheduler_patience"],
    threshold=RESNET_TRAINING_CONFIG[
        "minimum_auc_improvement"
    ],
    min_lr=RESNET_TRAINING_CONFIG[
        "minimum_learning_rate"
    ],
)


# ---------------------------------------------------------
# Train using validation ROC-AUC
# ---------------------------------------------------------

resnet_training_history = []

resnet_best_epoch = 0
resnet_best_valid_roc_auc = -np.inf
resnet_best_valid_metrics = None
resnet_best_model_state = None

resnet_epochs_without_improvement = 0

resnet_training_start = time.perf_counter()

resnet_epoch_progress = tqdm(
    range(
        1,
        RESNET_TRAINING_CONFIG["max_epochs"] + 1,
    ),
    desc="Outperformance ResNet",
    unit="epoch",
    dynamic_ncols=True,
)

for epoch in resnet_epoch_progress:
    epoch_start = time.perf_counter()

    current_learning_rate = (
        resnet_optimizer.param_groups[0]["lr"]
    )

    train_loss = train_one_epoch_with_progress(
        model=outperform_resnet,
        data_loader=resnet_train_loader,
        optimizer=resnet_optimizer,
        loss_function=resnet_loss_function,
        device=DEVICE,
        epoch=epoch,
        max_gradient_norm=RESNET_TRAINING_CONFIG[
            "gradient_clip_norm"
        ],
    )

    valid_metrics, _, _ = evaluate_binary_model(
        model=outperform_resnet,
        data_loader=outperform_valid_loader,
        loss_function=resnet_loss_function,
        device=DEVICE,
        threshold=0.50,
    )

    valid_roc_auc = valid_metrics["roc_auc"]

    resnet_scheduler.step(valid_roc_auc)

    next_learning_rate = (
        resnet_optimizer.param_groups[0]["lr"]
    )

    epoch_seconds = (
        time.perf_counter() - epoch_start
    )

    resnet_training_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_metrics["loss"],
            "valid_roc_auc": valid_roc_auc,
            "valid_average_precision": valid_metrics[
                "average_precision"
            ],
            "valid_log_loss": valid_metrics["log_loss"],
            "valid_brier_score": valid_metrics[
                "brier_score"
            ],
            "valid_accuracy": valid_metrics["accuracy"],
            "valid_balanced_accuracy": valid_metrics[
                "balanced_accuracy"
            ],
            "valid_precision": valid_metrics["precision"],
            "valid_recall": valid_metrics["recall"],
            "valid_f1": valid_metrics["f1"],
            "average_valid_probability": valid_metrics[
                "average_probability"
            ],
            "learning_rate": current_learning_rate,
            "next_learning_rate": next_learning_rate,
            "epoch_seconds": epoch_seconds,
        }
    )

    improved = (
        valid_roc_auc
        > resnet_best_valid_roc_auc
        + RESNET_TRAINING_CONFIG[
            "minimum_auc_improvement"
        ]
    )

    if improved:
        resnet_best_epoch = epoch
        resnet_best_valid_roc_auc = valid_roc_auc
        resnet_best_valid_metrics = valid_metrics.copy()

        resnet_best_model_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor
            in outperform_resnet.state_dict().items()
        }

        resnet_epochs_without_improvement = 0

        tqdm.write(
            f"New best ResNet epoch {epoch:02d}: "
            f"validation AUC={valid_roc_auc:.6f}, "
            f"AP={valid_metrics['average_precision']:.6f}"
        )

    else:
        resnet_epochs_without_improvement += 1

    resnet_epoch_progress.set_postfix(
        {
            "train": f"{train_loss:.5f}",
            "valid": f"{valid_metrics['loss']:.5f}",
            "auc": f"{valid_roc_auc:.5f}",
            "best": f"{resnet_best_valid_roc_auc:.5f}",
            "AP": f"{valid_metrics['average_precision']:.5f}",
            "patience": (
                f"{resnet_epochs_without_improvement}/"
                f"{RESNET_TRAINING_CONFIG['early_stopping_patience']}"
            ),
            "lr": f"{next_learning_rate:.1e}",
            "sec": f"{epoch_seconds:.1f}",
        }
    )

    if (
        resnet_epochs_without_improvement
        >= RESNET_TRAINING_CONFIG[
            "early_stopping_patience"
        ]
    ):
        tqdm.write(
            "ResNet early stopping triggered after "
            f"{resnet_epochs_without_improvement} epochs "
            "without sufficient validation AUC improvement."
        )
        break


resnet_training_seconds = (
    time.perf_counter() - resnet_training_start
)

if resnet_best_model_state is None:
    raise RuntimeError(
        "ResNet training did not produce a valid checkpoint."
    )


# ---------------------------------------------------------
# Restore the best validation checkpoint
# ---------------------------------------------------------

outperform_resnet.load_state_dict(
    resnet_best_model_state
)

outperform_resnet.to(DEVICE)
outperform_resnet.eval()


# ---------------------------------------------------------
# Evaluate train and validation only
# ---------------------------------------------------------

(
    resnet_train_metrics,
    _,
    resnet_train_probabilities,
) = evaluate_binary_model(
    model=outperform_resnet,
    data_loader=resnet_train_eval_loader,
    loss_function=resnet_loss_function,
    device=DEVICE,
    threshold=0.50,
)

(
    resnet_valid_metrics,
    resnet_valid_targets,
    resnet_valid_probabilities,
) = evaluate_binary_model(
    model=outperform_resnet,
    data_loader=outperform_valid_loader,
    loss_function=resnet_loss_function,
    device=DEVICE,
    threshold=0.50,
)

if not np.isclose(
    resnet_valid_metrics["roc_auc"],
    resnet_best_valid_roc_auc,
    atol=1e-10,
):
    raise ValueError(
        "Restored ResNet checkpoint does not match "
        "the recorded best validation ROC-AUC."
    )


# ---------------------------------------------------------
# Save training history and checkpoint
# ---------------------------------------------------------

resnet_training_history_df = pd.DataFrame(
    resnet_training_history
)

resnet_training_history_df.to_csv(
    RESNET_HISTORY_PATH,
    index=False,
)

resnet_checkpoint = {
    "model_name": "tabular_resnet_outperform",
    "model_version": "v1",
    "target": OUTPERFORM_TARGET,
    "split_method": PRIMARY_SPLIT_METHOD,
    "random_seed": RANDOM_SEED,
    "model_config": TABULAR_RESNET_CONFIG,
    "training_config": RESNET_TRAINING_CONFIG,
    "feature_cols": feature_cols,
    "best_epoch": resnet_best_epoch,
    "best_valid_metrics": resnet_valid_metrics,
    "training_duration_seconds": resnet_training_seconds,
    "state_dict": resnet_best_model_state,
}

torch.save(
    resnet_checkpoint,
    RESNET_CHECKPOINT_PATH,
)


# ---------------------------------------------------------
# Compare validation results
# ---------------------------------------------------------

resnet_training_summary = pd.DataFrame(
    [
        {
            "model": "tabular_resnet",
            "split": "train",
            **resnet_train_metrics,
        },
        {
            "model": "tabular_resnet",
            "split": "valid",
            **resnet_valid_metrics,
        },
    ]
)

print("\nResNet training completed.")
print("Best epoch:", resnet_best_epoch)

print(
    "Best validation ROC-AUC:",
    round(resnet_best_valid_roc_auc, 6),
)

print(
    "Total training time:",
    round(resnet_training_seconds, 2),
    "seconds",
)

print("Checkpoint:", RESNET_CHECKPOINT_PATH)
print("Training history:", RESNET_HISTORY_PATH)

print("\nThe test set has not been evaluated.")

display(
    resnet_training_summary[
        [
            "model",
            "split",
            "rows",
            "loss",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
        ]
    ]
)

Outperformance ResNet:   0%|          | 0/50 [00:00<?, ?epoch/s]

Epoch 01 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

New best ResNet epoch 01: validation AUC=0.531590, AP=0.530559


Epoch 02 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

New best ResNet epoch 02: validation AUC=0.538011, AP=0.533929


...

Epoch 17 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 18 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 19 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

ResNet early stopping triggered after 15 epochs without sufficient validation AUC improvement.

ResNet training completed.
Best epoch: 4
Best validation ROC-AUC: 0.540066
Total training time: 76.12 seconds
Checkpoint: E:\Projects\marketguard-india\models\deep_learning\tabular_resnet_outperform_purged_v1.pt
Training history: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\tabular_resnet_outperform_training_history_v1.csv

The test set has not been evaluated.


,model,split,rows,loss,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1
0,tabular_resnet,train,251738,0.672662,0.612011,0.616611,0.672662,0.240014,0.577616,0.575060,0.573937,0.680269,0.622595
1,tabular_resnet,valid,19980,0.692683,0.540066,0.538873,0.692683,0.249710,0.531431,0.529274,0.535789,0.625061,0.576993


In [15]:
# ---------------------------------------------------------
# Complete validation comparison
# ---------------------------------------------------------

resnet_valid_probability = np.asarray(
    resnet_valid_probabilities,
    dtype=np.float64,
)

resnet_valid_target = np.asarray(
    resnet_valid_targets,
    dtype=np.int64,
)

if not np.array_equal(
    resnet_valid_target,
    validation_targets,
):
    raise ValueError(
        "ResNet validation targets are not aligned with "
        "the MLP and Random Forest validation rows."
    )


# Existing fixed MLP-RF ensemble
mlp_rf_ensemble_probability = (
    0.30 * mlp_validation_probability
    + 0.70 * rf_validation_probability
)


def build_validation_model_row(
    model_name: str,
    probability: np.ndarray,
) -> dict:
    metrics = calculate_binary_metrics(
        y_true=validation_targets,
        y_probability=probability,
        threshold=0.50,
    )

    return {
        "model": model_name,
        **metrics,
    }


resnet_validation_comparison = pd.DataFrame(
    [
        build_validation_model_row(
            model_name="random_forest",
            probability=rf_validation_probability,
        ),
        build_validation_model_row(
            model_name="tabular_mlp",
            probability=mlp_validation_probability,
        ),
        build_validation_model_row(
            model_name="mlp_30_rf_70",
            probability=mlp_rf_ensemble_probability,
        ),
        build_validation_model_row(
            model_name="tabular_resnet",
            probability=resnet_valid_probability,
        ),
    ]
).sort_values(
    by="roc_auc",
    ascending=False,
).reset_index(drop=True)


# ---------------------------------------------------------
# Probability correlations
# ---------------------------------------------------------

resnet_probability_correlations = pd.DataFrame(
    {
        "comparison": [
            "resnet_vs_random_forest",
            "resnet_vs_mlp",
            "resnet_vs_mlp_rf_ensemble",
        ],
        "probability_correlation": [
            np.corrcoef(
                resnet_valid_probability,
                rf_validation_probability,
            )[0, 1],
            np.corrcoef(
                resnet_valid_probability,
                mlp_validation_probability,
            )[0, 1],
            np.corrcoef(
                resnet_valid_probability,
                mlp_rf_ensemble_probability,
            )[0, 1],
        ],
    }
)


# ---------------------------------------------------------
# Save reports
# ---------------------------------------------------------

RESNET_COMPARISON_PATH = (
    REPORT_DIR
    / "tabular_resnet_validation_comparison_v1.csv"
)

RESNET_CORRELATION_PATH = (
    REPORT_DIR
    / "tabular_resnet_probability_correlations_v1.csv"
)

resnet_validation_comparison.to_csv(
    RESNET_COMPARISON_PATH,
    index=False,
)

resnet_probability_correlations.to_csv(
    RESNET_CORRELATION_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display actual saved training configuration
# ---------------------------------------------------------

print("Actual ResNet training configuration:")

for key, value in RESNET_TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

print("\nValidation comparison:")

display(
    resnet_validation_comparison[
        [
            "model",
            "rows",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "average_probability",
        ]
    ]
)

print("\nProbability correlations:")

display(resnet_probability_correlations)

print("\nThe test set remains untouched.")

Actual ResNet training configuration:
  max_epochs: 50
  learning_rate: 0.0005
  weight_decay: 0.0005
  early_stopping_patience: 15
  minimum_auc_improvement: 0.0001
  gradient_clip_norm: 5.0
  scheduler_factor: 0.5
  scheduler_patience: 2
  minimum_learning_rate: 1e-05

Validation comparison:


,model,rows,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1,average_probability
0,tabular_resnet,19980,0.540066,0.538873,0.692683,0.249710,0.531431,0.529274,0.535789,0.625061,0.576993,0.511354
1,mlp_30_rf_70,19980,0.534681,0.531370,0.691136,0.248993,0.525676,0.522344,0.528481,0.670289,0.590997,0.511467
2,tabular_mlp,19980,0.530320,0.528792,0.693764,0.250238,0.524274,0.522063,0.529677,0.620264,0.571403,0.510942
3,random_forest,19980,0.528908,0.530654,0.691974,0.249410,0.522573,0.519090,0.525825,0.673715,0.590654,0.511692



Probability correlations:


,comparison,probability_correlation
0,resnet_vs_random_forest,0.504541
1,resnet_vs_mlp,0.748599
2,resnet_vs_mlp_rf_ensemble,0.722477



The test set remains untouched.


## Tabular ResNet Result

The Tabular ResNet achieved the best validation ranking performance so far.

### Validation Comparison

| Model | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Tabular ResNet | **0.5401** | **0.5389** | 0.6927 | 0.2497 |
| 30% MLP + 70% RF | 0.5347 | 0.5314 | **0.6911** | **0.2490** |
| Tabular MLP | 0.5303 | 0.5288 | 0.6938 | 0.2502 |
| Random Forest | 0.5289 | 0.5307 | 0.6920 | 0.2494 |

### Interpretation

The ResNet produced:

- The highest ROC-AUC
- The highest average precision
- Better ranking performance than the MLP and Random Forest
- Better ranking performance than the existing MLP-RF ensemble

The best checkpoint was reached at epoch 4. Performance declined during later
epochs, showing that the model started overfitting. Early stopping restored the
best epoch.

The ResNet was slightly worse than the MLP-RF ensemble on log loss and Brier
score. This means it ranked stocks better, but its probability values were
slightly less accurate.

The ResNet predictions had a correlation of approximately `0.50` with the
Random Forest predictions. This suggests that the two models learned partly
different information and may work well together in an ensemble.

### Decision

The Tabular ResNet is currently the strongest standalone model.

A paired bootstrap comparison will be used to check whether its improvement
over the Random Forest and the existing ensemble is reliable.

The test set remains untouched.

### check whether the ResNet’s ranking advantage is reliable.

In [16]:
# ---------------------------------------------------------
# Paired bootstrap:
# ResNet versus RF and existing MLP-RF ensemble
# ---------------------------------------------------------

RESNET_BOOTSTRAP_ITERATIONS = 2000
RESNET_BOOTSTRAP_SEED = 42

comparison_probabilities = {
    "random_forest": np.asarray(
        rf_validation_probability,
        dtype=np.float64,
    ),
    "mlp_30_rf_70": np.asarray(
        mlp_rf_ensemble_probability,
        dtype=np.float64,
    ),
}

resnet_probability = np.asarray(
    resnet_valid_probability,
    dtype=np.float64,
)

bootstrap_targets = np.asarray(
    validation_targets,
    dtype=np.int64,
)

for name, probability in comparison_probabilities.items():
    if len(probability) != len(bootstrap_targets):
        raise ValueError(
            f"{name} probability length does not match targets."
        )

if len(resnet_probability) != len(bootstrap_targets):
    raise ValueError(
        "ResNet probability length does not match targets."
    )


# ---------------------------------------------------------
# Calculate observed differences
# ---------------------------------------------------------

resnet_observed_metrics = calculate_probability_metrics(
    y_true=bootstrap_targets,
    probability=resnet_probability,
)

observed_comparison_differences = {}

for baseline_name, baseline_probability in (
    comparison_probabilities.items()
):
    baseline_metrics = calculate_probability_metrics(
        y_true=bootstrap_targets,
        probability=baseline_probability,
    )

    observed_comparison_differences[baseline_name] = {
        "roc_auc_difference_resnet_minus_baseline": (
            resnet_observed_metrics["roc_auc"]
            - baseline_metrics["roc_auc"]
        ),
        "average_precision_difference_resnet_minus_baseline": (
            resnet_observed_metrics["average_precision"]
            - baseline_metrics["average_precision"]
        ),
        "log_loss_difference_resnet_minus_baseline": (
            resnet_observed_metrics["log_loss"]
            - baseline_metrics["log_loss"]
        ),
        "brier_difference_resnet_minus_baseline": (
            resnet_observed_metrics["brier_score"]
            - baseline_metrics["brier_score"]
        ),
    }


# ---------------------------------------------------------
# Run paired bootstrap
# ---------------------------------------------------------

rng = np.random.default_rng(
    RESNET_BOOTSTRAP_SEED
)

validation_row_count = len(
    bootstrap_targets
)

resnet_bootstrap_rows = []

for iteration in tqdm(
    range(RESNET_BOOTSTRAP_ITERATIONS),
    desc="ResNet paired bootstrap",
    unit="sample",
    dynamic_ncols=True,
):
    sampled_indices = rng.integers(
        low=0,
        high=validation_row_count,
        size=validation_row_count,
    )

    sampled_targets = bootstrap_targets[
        sampled_indices
    ]

    if np.unique(sampled_targets).size < 2:
        continue

    sampled_resnet_probability = (
        resnet_probability[
            sampled_indices
        ]
    )

    sampled_resnet_metrics = (
        calculate_probability_metrics(
            y_true=sampled_targets,
            probability=sampled_resnet_probability,
        )
    )

    for (
        baseline_name,
        baseline_probability,
    ) in comparison_probabilities.items():

        sampled_baseline_probability = (
            baseline_probability[
                sampled_indices
            ]
        )

        sampled_baseline_metrics = (
            calculate_probability_metrics(
                y_true=sampled_targets,
                probability=sampled_baseline_probability,
            )
        )

        resnet_bootstrap_rows.append(
            {
                "iteration": iteration + 1,
                "baseline": baseline_name,
                "roc_auc_difference_resnet_minus_baseline": (
                    sampled_resnet_metrics["roc_auc"]
                    - sampled_baseline_metrics["roc_auc"]
                ),
                "average_precision_difference_resnet_minus_baseline": (
                    sampled_resnet_metrics[
                        "average_precision"
                    ]
                    - sampled_baseline_metrics[
                        "average_precision"
                    ]
                ),
                "log_loss_difference_resnet_minus_baseline": (
                    sampled_resnet_metrics["log_loss"]
                    - sampled_baseline_metrics["log_loss"]
                ),
                "brier_difference_resnet_minus_baseline": (
                    sampled_resnet_metrics["brier_score"]
                    - sampled_baseline_metrics["brier_score"]
                ),
            }
        )

resnet_bootstrap_results = pd.DataFrame(
    resnet_bootstrap_rows
)


# ---------------------------------------------------------
# Summarize confidence intervals
# ---------------------------------------------------------

resnet_metric_directions = {
    "roc_auc_difference_resnet_minus_baseline": (
        "higher_is_better"
    ),
    "average_precision_difference_resnet_minus_baseline": (
        "higher_is_better"
    ),
    "log_loss_difference_resnet_minus_baseline": (
        "lower_is_better"
    ),
    "brier_difference_resnet_minus_baseline": (
        "lower_is_better"
    ),
}

resnet_bootstrap_summary_rows = []

for baseline_name in comparison_probabilities:
    baseline_results = (
        resnet_bootstrap_results.loc[
            resnet_bootstrap_results["baseline"]
            == baseline_name
        ]
    )

    for (
        metric,
        better_direction,
    ) in resnet_metric_directions.items():

        differences = baseline_results[
            metric
        ].to_numpy()

        ci_lower, ci_upper = np.quantile(
            differences,
            [0.025, 0.975],
        )

        if better_direction == "higher_is_better":
            resnet_win_rate = np.mean(
                differences > 0
            )
        else:
            resnet_win_rate = np.mean(
                differences < 0
            )

        resnet_bootstrap_summary_rows.append(
            {
                "comparison": (
                    f"resnet_vs_{baseline_name}"
                ),
                "metric": metric,
                "observed_difference": (
                    observed_comparison_differences[
                        baseline_name
                    ][metric]
                ),
                "bootstrap_mean_difference": float(
                    differences.mean()
                ),
                "ci_2_5_pct": float(ci_lower),
                "ci_97_5_pct": float(ci_upper),
                "resnet_bootstrap_win_rate": float(
                    resnet_win_rate
                ),
                "ci_excludes_zero": bool(
                    (ci_lower > 0)
                    or (ci_upper < 0)
                ),
                "better_direction": better_direction,
            }
        )

resnet_bootstrap_summary = pd.DataFrame(
    resnet_bootstrap_summary_rows
)


# ---------------------------------------------------------
# Save reports
# ---------------------------------------------------------

RESNET_BOOTSTRAP_DETAIL_PATH = (
    REPORT_DIR
    / "tabular_resnet_validation_bootstrap_v1.csv"
)

RESNET_BOOTSTRAP_SUMMARY_PATH = (
    REPORT_DIR
    / "tabular_resnet_validation_bootstrap_summary_v1.csv"
)

resnet_bootstrap_results.to_csv(
    RESNET_BOOTSTRAP_DETAIL_PATH,
    index=False,
)

resnet_bootstrap_summary.to_csv(
    RESNET_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print(
    "Completed paired bootstrap samples:",
    resnet_bootstrap_results[
        "iteration"
    ].nunique(),
)

print(
    "\nPositive ROC-AUC/AP differences favour the ResNet."
)

print(
    "Negative log-loss/Brier differences favour the ResNet."
)

print(
    "\nThe test set remains untouched."
)

display(resnet_bootstrap_summary)

ResNet paired bootstrap:   0%|          | 0/2000 [00:00<?, ?sample/s]

Completed paired bootstrap samples: 2000

Positive ROC-AUC/AP differences favour the ResNet.
Negative log-loss/Brier differences favour the ResNet.

The test set remains untouched.


,comparison,metric,observed_difference,bootstrap_mean_difference,ci_2_5_pct,ci_97_5_pct,resnet_bootstrap_win_rate,ci_excludes_zero,better_direction
0,resnet_vs_random_forest,roc_auc_difference_resnet_minus_baseline,0.011159,0.011224,0.002822,0.018850,0.9975,True,higher_is_better
1,resnet_vs_random_forest,average_precision_difference_resnet_minus_base...,0.008219,0.008277,0.000208,0.015863,0.9780,True,higher_is_better
2,resnet_vs_random_forest,log_loss_difference_resnet_minus_baseline,0.000709,0.000691,-0.001004,0.002409,0.2225,False,lower_is_better
3,resnet_vs_random_forest,brier_difference_resnet_minus_baseline,0.000300,0.000292,-0.000523,0.001132,0.2545,False,lower_is_better
4,resnet_vs_mlp_30_rf_70,roc_auc_difference_resnet_minus_baseline,0.005385,0.005398,-0.000887,0.011568,0.9505,False,higher_is_better
5,resnet_vs_mlp_30_rf_70,average_precision_difference_resnet_minus_base...,0.007504,0.007543,0.000916,0.013570,0.9895,True,higher_is_better
6,resnet_vs_mlp_30_rf_70,log_loss_difference_resnet_minus_baseline,0.001547,0.001533,0.000094,0.003002,0.0170,True,lower_is_better
7,resnet_vs_mlp_30_rf_70,brier_difference_resnet_minus_baseline,0.000717,0.000710,0.000013,0.001417,0.0205,True,lower_is_better


## ResNet Bootstrap Result

The paired bootstrap used 2,000 validation samples to compare the ResNet with
the Random Forest and the existing MLP-RF ensemble.

### ResNet versus Random Forest

| Metric | Result |
|---|---|
| ROC-AUC | ResNet reliably better |
| Average precision | ResNet reliably better |
| Log loss | No reliable difference |
| Brier score | No reliable difference |

The ResNet ROC-AUC improvement was approximately `+0.0112`.

Its 95% confidence interval was entirely above zero:

`+0.0028 to +0.0189`

The ResNet also beat the Random Forest in `99.75%` of ROC-AUC bootstrap samples.

This provides strong evidence that the ResNet ranks stocks better than the
Random Forest on the validation data.

### ResNet versus MLP-RF Ensemble

| Metric | Result |
|---|---|
| ROC-AUC | ResNet higher, but not reliably |
| Average precision | ResNet reliably better |
| Log loss | Ensemble reliably better |
| Brier score | Ensemble reliably better |

The ResNet has stronger ranking performance, while the MLP-RF ensemble produces
better probability estimates.

### Decision

The Tabular ResNet is now the strongest standalone ranking model.

However, the existing ensemble remains better calibrated.

A moving-block bootstrap is still required because stock-market rows and nearby
trading dates are not fully independent.

The test set remains untouched.

### moving-block bootstrap - 20 day market periods

In [17]:
# ---------------------------------------------------------
# Moving-block bootstrap:
# ResNet versus RF and existing MLP-RF ensemble
# ---------------------------------------------------------

RESNET_BLOCK_ITERATIONS = 2000
RESNET_BLOCK_LENGTH = 20
RESNET_BLOCK_SEED = 42

resnet_block_targets = np.asarray(
    validation_targets,
    dtype=np.int64,
)

resnet_block_probability = np.asarray(
    resnet_valid_probability,
    dtype=np.float64,
)

resnet_block_baselines = {
    "random_forest": np.asarray(
        rf_validation_probability,
        dtype=np.float64,
    ),
    "mlp_30_rf_70": np.asarray(
        mlp_rf_ensemble_probability,
        dtype=np.float64,
    ),
}


# ---------------------------------------------------------
# Check validation alignment
# ---------------------------------------------------------

required_length = len(resnet_block_targets)

if len(validation_metadata) != required_length:
    raise ValueError(
        "Validation metadata does not align with targets."
    )

if len(resnet_block_probability) != required_length:
    raise ValueError(
        "ResNet probabilities do not align with targets."
    )

for name, probability in resnet_block_baselines.items():
    if len(probability) != required_length:
        raise ValueError(
            f"{name} probabilities do not align with targets."
        )


# ---------------------------------------------------------
# Prepare trading-date blocks
# ---------------------------------------------------------

block_validation_dates = pd.DatetimeIndex(
    validation_metadata["date"]
    .drop_duplicates()
    .sort_values()
)

if len(block_validation_dates) < RESNET_BLOCK_LENGTH:
    raise ValueError(
        "Validation period is shorter than the block length."
    )

block_date_to_indices = {
    date: group.index.to_numpy()
    for date, group in validation_metadata.groupby(
        "date",
        sort=True,
    )
}

maximum_start_position = (
    len(block_validation_dates)
    - RESNET_BLOCK_LENGTH
)

number_of_blocks = int(
    np.ceil(
        len(block_validation_dates)
        / RESNET_BLOCK_LENGTH
    )
)


# ---------------------------------------------------------
# Calculate observed differences
# ---------------------------------------------------------

observed_resnet_metrics = calculate_probability_metrics(
    y_true=resnet_block_targets,
    probability=resnet_block_probability,
)

observed_block_differences = {}

for baseline_name, baseline_probability in (
    resnet_block_baselines.items()
):
    baseline_metrics = calculate_probability_metrics(
        y_true=resnet_block_targets,
        probability=baseline_probability,
    )

    observed_block_differences[baseline_name] = {
        "roc_auc_difference": (
            observed_resnet_metrics["roc_auc"]
            - baseline_metrics["roc_auc"]
        ),
        "average_precision_difference": (
            observed_resnet_metrics["average_precision"]
            - baseline_metrics["average_precision"]
        ),
        "log_loss_difference": (
            observed_resnet_metrics["log_loss"]
            - baseline_metrics["log_loss"]
        ),
        "brier_difference": (
            observed_resnet_metrics["brier_score"]
            - baseline_metrics["brier_score"]
        ),
    }


# ---------------------------------------------------------
# Run moving-block bootstrap
# ---------------------------------------------------------

rng = np.random.default_rng(
    RESNET_BLOCK_SEED
)

resnet_block_rows = []

for iteration in tqdm(
    range(RESNET_BLOCK_ITERATIONS),
    desc="ResNet 20-day block bootstrap",
    unit="sample",
    dynamic_ncols=True,
):
    sampled_block_starts = rng.integers(
        low=0,
        high=maximum_start_position + 1,
        size=number_of_blocks,
    )

    sampled_date_positions = np.concatenate(
        [
            np.arange(
                start,
                start + RESNET_BLOCK_LENGTH,
            )
            for start in sampled_block_starts
        ]
    )[:len(block_validation_dates)]

    sampled_dates = block_validation_dates[
        sampled_date_positions
    ]

    sampled_indices = np.concatenate(
        [
            block_date_to_indices[date]
            for date in sampled_dates
        ]
    )

    sampled_targets = resnet_block_targets[
        sampled_indices
    ]

    if np.unique(sampled_targets).size < 2:
        continue

    sampled_resnet_probability = (
        resnet_block_probability[
            sampled_indices
        ]
    )

    sampled_resnet_metrics = (
        calculate_probability_metrics(
            y_true=sampled_targets,
            probability=sampled_resnet_probability,
        )
    )

    for baseline_name, baseline_probability in (
        resnet_block_baselines.items()
    ):
        sampled_baseline_probability = (
            baseline_probability[
                sampled_indices
            ]
        )

        sampled_baseline_metrics = (
            calculate_probability_metrics(
                y_true=sampled_targets,
                probability=sampled_baseline_probability,
            )
        )

        resnet_block_rows.append(
            {
                "iteration": iteration + 1,
                "baseline": baseline_name,
                "roc_auc_difference": (
                    sampled_resnet_metrics["roc_auc"]
                    - sampled_baseline_metrics["roc_auc"]
                ),
                "average_precision_difference": (
                    sampled_resnet_metrics[
                        "average_precision"
                    ]
                    - sampled_baseline_metrics[
                        "average_precision"
                    ]
                ),
                "log_loss_difference": (
                    sampled_resnet_metrics["log_loss"]
                    - sampled_baseline_metrics["log_loss"]
                ),
                "brier_difference": (
                    sampled_resnet_metrics["brier_score"]
                    - sampled_baseline_metrics["brier_score"]
                ),
            }
        )

resnet_block_results = pd.DataFrame(
    resnet_block_rows
)


# ---------------------------------------------------------
# Summarize results
# ---------------------------------------------------------

metric_directions = {
    "roc_auc_difference": "higher_is_better",
    "average_precision_difference": "higher_is_better",
    "log_loss_difference": "lower_is_better",
    "brier_difference": "lower_is_better",
}

resnet_block_summary_rows = []

for baseline_name in resnet_block_baselines:
    baseline_results = resnet_block_results.loc[
        resnet_block_results["baseline"]
        == baseline_name
    ]

    for metric, direction in metric_directions.items():
        differences = baseline_results[
            metric
        ].to_numpy()

        ci_lower, ci_upper = np.quantile(
            differences,
            [0.025, 0.975],
        )

        if direction == "higher_is_better":
            resnet_win_rate = np.mean(
                differences > 0
            )
        else:
            resnet_win_rate = np.mean(
                differences < 0
            )

        resnet_block_summary_rows.append(
            {
                "comparison": (
                    f"resnet_vs_{baseline_name}"
                ),
                "metric": metric,
                "observed_difference": (
                    observed_block_differences[
                        baseline_name
                    ][metric]
                ),
                "bootstrap_mean_difference": float(
                    differences.mean()
                ),
                "ci_2_5_pct": float(ci_lower),
                "ci_97_5_pct": float(ci_upper),
                "resnet_win_rate": float(
                    resnet_win_rate
                ),
                "ci_excludes_zero": bool(
                    (ci_lower > 0)
                    or (ci_upper < 0)
                ),
                "better_direction": direction,
            }
        )

resnet_block_summary = pd.DataFrame(
    resnet_block_summary_rows
)


# ---------------------------------------------------------
# Save reports
# ---------------------------------------------------------

RESNET_BLOCK_DETAIL_PATH = (
    REPORT_DIR
    / "tabular_resnet_validation_block_bootstrap_v1.csv"
)

RESNET_BLOCK_SUMMARY_PATH = (
    REPORT_DIR
    / "tabular_resnet_validation_block_bootstrap_summary_v1.csv"
)

resnet_block_results.to_csv(
    RESNET_BLOCK_DETAIL_PATH,
    index=False,
)

resnet_block_summary.to_csv(
    RESNET_BLOCK_SUMMARY_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print(
    "Completed block-bootstrap samples:",
    resnet_block_results[
        "iteration"
    ].nunique(),
)

print(
    "Block length:",
    RESNET_BLOCK_LENGTH,
    "trading days",
)

print(
    "\nPositive ROC-AUC/AP differences favour the ResNet."
)

print(
    "Negative log-loss/Brier differences favour the ResNet."
)

print("\nThe test set remains untouched.")

display(resnet_block_summary)

ResNet 20-day block bootstrap:   0%|          | 0/2000 [00:00<?, ?sample/s]

Completed block-bootstrap samples: 2000
Block length: 20 trading days

Positive ROC-AUC/AP differences favour the ResNet.
Negative log-loss/Brier differences favour the ResNet.

The test set remains untouched.


,comparison,metric,observed_difference,bootstrap_mean_difference,ci_2_5_pct,ci_97_5_pct,resnet_win_rate,ci_excludes_zero,better_direction
0,resnet_vs_random_forest,roc_auc_difference,0.011159,0.009586,-0.016505,0.035751,0.7490,False,higher_is_better
1,resnet_vs_random_forest,average_precision_difference,0.008219,0.007814,-0.021445,0.039303,0.6760,False,higher_is_better
2,resnet_vs_random_forest,log_loss_difference,0.000709,0.001578,-0.006054,0.009061,0.3395,False,lower_is_better
3,resnet_vs_random_forest,brier_difference,0.000300,0.000727,-0.002981,0.004339,0.3455,False,lower_is_better
4,resnet_vs_mlp_30_rf_70,roc_auc_difference,0.005385,0.004364,-0.016189,0.024747,0.6750,False,higher_is_better
5,resnet_vs_mlp_30_rf_70,average_precision_difference,0.007504,0.005648,-0.013879,0.022157,0.7455,False,higher_is_better
6,resnet_vs_mlp_30_rf_70,log_loss_difference,0.001547,0.002335,-0.003206,0.007964,0.2165,False,lower_is_better
7,resnet_vs_mlp_30_rf_70,brier_difference,0.000717,0.001102,-0.001531,0.003788,0.2185,False,lower_is_better


## ResNet Moving-Block Bootstrap Result

The ResNet was compared with the Random Forest and the existing MLP-RF
ensemble using 2,000 moving-block bootstrap samples.

Each sample used complete 20-trading-day blocks to preserve dependence between:

- Stocks observed during the same market period
- Nearby trading dates
- Overlapping 20-day prediction targets

### ResNet versus Random Forest

| Metric | Observed Difference | 95% Confidence Interval | ResNet Win Rate |
|---|---:|---:|---:|
| ROC-AUC | +0.0112 | −0.0165 to +0.0358 | 74.90% |
| Average precision | +0.0082 | −0.0214 to +0.0393 | 67.60% |
| Log loss | +0.0007 | −0.0061 to +0.0091 | 33.95% |
| Brier score | +0.0003 | −0.0030 to +0.0043 | 34.55% |

### Interpretation

The ResNet achieved higher observed ROC-AUC and average precision than the
Random Forest.

It also beat the Random Forest in most sampled market periods for the ranking
metrics.

However, every confidence interval includes zero.

Therefore, the ResNet improvement is promising but not reliably consistent
across different validation market periods.

### Decision

The ResNet remains the strongest standalone validation candidate.

It is not yet eligible to replace the Random Forest because its advantage may
depend on the particular market period.

The next experiment will test a ResNet-Random-Forest ensemble because their
prediction correlation is only approximately `0.50`.

The test set remains untouched.

In [18]:
# ---------------------------------------------------------
# Validation-only ResNet and Random Forest ensemble search
# ---------------------------------------------------------

RESNET_RF_WEIGHTS = np.arange(
    0.0,
    1.01,
    0.05,
)

resnet_ensemble_probability = np.asarray(
    resnet_valid_probability,
    dtype=np.float64,
)

rf_ensemble_probability = np.asarray(
    rf_validation_probability,
    dtype=np.float64,
)

ensemble_targets = np.asarray(
    validation_targets,
    dtype=np.int64,
)

if not (
    len(ensemble_targets)
    == len(resnet_ensemble_probability)
    == len(rf_ensemble_probability)
):
    raise ValueError(
        "Targets and model probabilities are not aligned."
    )


# ---------------------------------------------------------
# Evaluate every weight combination
# ---------------------------------------------------------

resnet_rf_ensemble_rows = []

for resnet_weight in RESNET_RF_WEIGHTS:
    rf_weight = 1.0 - resnet_weight

    combined_probability = (
        resnet_weight * resnet_ensemble_probability
        + rf_weight * rf_ensemble_probability
    )

    metrics = calculate_binary_metrics(
        y_true=ensemble_targets,
        y_probability=combined_probability,
        threshold=0.50,
    )

    resnet_rf_ensemble_rows.append(
        {
            "resnet_weight": float(resnet_weight),
            "random_forest_weight": float(rf_weight),
            **metrics,
        }
    )

resnet_rf_ensemble_results = (
    pd.DataFrame(resnet_rf_ensemble_rows)
    .sort_values(
        by=[
            "roc_auc",
            "average_precision",
            "log_loss",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Identify important candidates
# ---------------------------------------------------------

best_resnet_rf_ensemble = (
    resnet_rf_ensemble_results.iloc[0].copy()
)

rf_only_result = (
    resnet_rf_ensemble_results.loc[
        np.isclose(
            resnet_rf_ensemble_results[
                "resnet_weight"
            ],
            0.0,
        )
    ]
    .iloc[0]
    .copy()
)

resnet_only_result = (
    resnet_rf_ensemble_results.loc[
        np.isclose(
            resnet_rf_ensemble_results[
                "resnet_weight"
            ],
            1.0,
        )
    ]
    .iloc[0]
    .copy()
)

equal_weight_result = (
    resnet_rf_ensemble_results.loc[
        np.isclose(
            resnet_rf_ensemble_results[
                "resnet_weight"
            ],
            0.50,
        )
    ]
    .iloc[0]
    .copy()
)


# ---------------------------------------------------------
# Existing MLP-RF ensemble reference
# ---------------------------------------------------------

existing_mlp_rf_metrics = calculate_binary_metrics(
    y_true=ensemble_targets,
    y_probability=mlp_rf_ensemble_probability,
    threshold=0.50,
)


# ---------------------------------------------------------
# Save results
# ---------------------------------------------------------

RESNET_RF_ENSEMBLE_PATH = (
    REPORT_DIR
    / "tabular_resnet_rf_ensemble_validation_v1.csv"
)

resnet_rf_ensemble_results.to_csv(
    RESNET_RF_ENSEMBLE_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display summary
# ---------------------------------------------------------

resnet_rf_candidate_summary = pd.DataFrame(
    [
        {
            "candidate": "random_forest_only",
            **rf_only_result.to_dict(),
        },
        {
            "candidate": "resnet_only",
            **resnet_only_result.to_dict(),
        },
        {
            "candidate": "equal_resnet_rf",
            **equal_weight_result.to_dict(),
        },
        {
            "candidate": "best_resnet_rf_ensemble",
            **best_resnet_rf_ensemble.to_dict(),
        },
        {
            "candidate": "existing_mlp_30_rf_70",
            "resnet_weight": np.nan,
            "random_forest_weight": 0.70,
            **existing_mlp_rf_metrics,
        },
    ]
)

print(
    "Best validation ResNet weight:",
    round(
        float(
            best_resnet_rf_ensemble[
                "resnet_weight"
            ]
        ),
        2,
    ),
)

print(
    "Best validation Random Forest weight:",
    round(
        float(
            best_resnet_rf_ensemble[
                "random_forest_weight"
            ]
        ),
        2,
    ),
)

print(
    "\nResNet and Random Forest probability correlation:",
    round(
        float(
            np.corrcoef(
                resnet_ensemble_probability,
                rf_ensemble_probability,
            )[0, 1]
        ),
        6,
    ),
)

print("\nThe test set remains untouched.")

display(
    resnet_rf_candidate_summary[
        [
            "candidate",
            "resnet_weight",
            "random_forest_weight",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
        ]
    ]
)

print("\nTop 10 ResNet-RF combinations by validation ROC-AUC:")

display(
    resnet_rf_ensemble_results[
        [
            "resnet_weight",
            "random_forest_weight",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
        ]
    ].head(10)
)

Best validation ResNet weight: 0.75
Best validation Random Forest weight: 0.25

ResNet and Random Forest probability correlation: 0.504541

The test set remains untouched.


,candidate,resnet_weight,random_forest_weight,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1
0,random_forest_only,0.00,1.00,0.528908,0.530654,0.691974,0.249410,0.519090,0.525825,0.673715,0.590654
1,resnet_only,1.00,0.00,0.540066,0.538873,0.692683,0.249710,0.529274,0.535789,0.625061,0.576993
2,equal_resnet_rf,0.50,0.50,0.539653,0.538903,0.690407,0.248631,0.528389,0.533974,0.652374,0.587266
3,best_resnet_rf_ensemble,0.75,0.25,0.540278,0.539083,0.691047,0.248938,0.530514,0.536247,0.640137,0.583605
4,existing_mlp_30_rf_70,NaN,0.70,0.534681,0.531370,0.691136,0.248993,0.522344,0.528481,0.670289,0.590997



Top 10 ResNet-RF combinations by validation ROC-AUC:


,resnet_weight,random_forest_weight,roc_auc,average_precision,log_loss,brier_score
0,0.75,0.25,0.540278,0.539083,0.691047,0.248938
1,0.80,0.20,0.540278,0.539070,0.691293,0.249055
2,0.70,0.30,0.540253,0.539112,0.690841,0.248840
3,0.85,0.15,0.540251,0.539024,0.691579,0.249191
4,0.90,0.10,0.540207,0.538985,0.691906,0.249346
5,0.65,0.35,0.540181,0.539123,0.690674,0.248760
6,0.95,0.05,0.540146,0.538944,0.692273,0.249519
7,1.00,0.00,0.540066,0.538873,0.692683,0.249710
8,0.60,0.40,0.540066,0.539116,0.690547,0.248698
9,0.55,0.45,0.539899,0.539024,0.690458,0.248655


In [19]:
# ---------------------------------------------------------
# Fixed ResNet-RF ensemble block bootstrap
# ---------------------------------------------------------

FIXED_RESNET_WEIGHT = 0.75
FIXED_RF_WEIGHT = 0.25
ENSEMBLE_BLOCK_ITERATIONS = 2000
ENSEMBLE_BLOCK_SEED = 42

fixed_resnet_rf_probability = FIXED_RESNET_WEIGHT * resnet_valid_probability + FIXED_RF_WEIGHT * rf_validation_probability

baseline_probabilities = {
    "resnet": np.asarray(resnet_valid_probability, dtype=np.float64),
    "random_forest": np.asarray(rf_validation_probability, dtype=np.float64),
}

targets = np.asarray(validation_targets, dtype=np.int64)
ensemble_probability = np.asarray(fixed_resnet_rf_probability, dtype=np.float64)

observed_ensemble_metrics = calculate_probability_metrics(targets, ensemble_probability)
observed_differences = {}

for name, probability in baseline_probabilities.items():
    baseline_metrics = calculate_probability_metrics(targets, probability)
    observed_differences[name] = {
        "roc_auc_difference": observed_ensemble_metrics["roc_auc"] - baseline_metrics["roc_auc"],
        "average_precision_difference": observed_ensemble_metrics["average_precision"] - baseline_metrics["average_precision"],
        "log_loss_difference": observed_ensemble_metrics["log_loss"] - baseline_metrics["log_loss"],
        "brier_difference": observed_ensemble_metrics["brier_score"] - baseline_metrics["brier_score"],
    }

rng = np.random.default_rng(ENSEMBLE_BLOCK_SEED)
bootstrap_rows = []

for iteration in tqdm(range(ENSEMBLE_BLOCK_ITERATIONS), desc="ResNet-RF block bootstrap", unit="sample", dynamic_ncols=True):
    starts = rng.integers(0, maximum_start_position + 1, size=number_of_blocks)
    date_positions = np.concatenate([np.arange(start, start + RESNET_BLOCK_LENGTH) for start in starts])[:len(block_validation_dates)]
    sampled_dates = block_validation_dates[date_positions]
    indices = np.concatenate([block_date_to_indices[date] for date in sampled_dates])
    sampled_targets = targets[indices]

    if np.unique(sampled_targets).size < 2:
        continue

    ensemble_metrics = calculate_probability_metrics(sampled_targets, ensemble_probability[indices])

    for name, probability in baseline_probabilities.items():
        baseline_metrics = calculate_probability_metrics(sampled_targets, probability[indices])

        bootstrap_rows.append({
            "iteration": iteration + 1,
            "baseline": name,
            "roc_auc_difference": ensemble_metrics["roc_auc"] - baseline_metrics["roc_auc"],
            "average_precision_difference": ensemble_metrics["average_precision"] - baseline_metrics["average_precision"],
            "log_loss_difference": ensemble_metrics["log_loss"] - baseline_metrics["log_loss"],
            "brier_difference": ensemble_metrics["brier_score"] - baseline_metrics["brier_score"],
        })

ensemble_block_results = pd.DataFrame(bootstrap_rows)

metric_directions = {
    "roc_auc_difference": "higher_is_better",
    "average_precision_difference": "higher_is_better",
    "log_loss_difference": "lower_is_better",
    "brier_difference": "lower_is_better",
}

summary_rows = []

for baseline_name in baseline_probabilities:
    baseline_results = ensemble_block_results[ensemble_block_results["baseline"] == baseline_name]

    for metric, direction in metric_directions.items():
        values = baseline_results[metric].to_numpy()
        ci_lower, ci_upper = np.quantile(values, [0.025, 0.975])
        win_rate = np.mean(values > 0) if direction == "higher_is_better" else np.mean(values < 0)

        summary_rows.append({
            "comparison": f"ensemble_vs_{baseline_name}",
            "metric": metric,
            "observed_difference": observed_differences[baseline_name][metric],
            "bootstrap_mean_difference": values.mean(),
            "ci_2_5_pct": ci_lower,
            "ci_97_5_pct": ci_upper,
            "ensemble_win_rate": win_rate,
            "ci_excludes_zero": bool(ci_lower > 0 or ci_upper < 0),
            "better_direction": direction,
        })

ensemble_block_summary = pd.DataFrame(summary_rows)

ENSEMBLE_BLOCK_DETAIL_PATH = REPORT_DIR / "resnet_rf_ensemble_validation_block_bootstrap_v1.csv"
ENSEMBLE_BLOCK_SUMMARY_PATH = REPORT_DIR / "resnet_rf_ensemble_validation_block_bootstrap_summary_v1.csv"

ensemble_block_results.to_csv(ENSEMBLE_BLOCK_DETAIL_PATH, index=False)
ensemble_block_summary.to_csv(ENSEMBLE_BLOCK_SUMMARY_PATH, index=False)

print("Completed samples:", ensemble_block_results["iteration"].nunique())
print("Fixed weights: ResNet=0.75, Random Forest=0.25")
print("Positive ROC-AUC/AP differences favour the ensemble.")
print("Negative log-loss/Brier differences favour the ensemble.")
print("\nThe test set remains untouched.")

display(ensemble_block_summary)

ResNet-RF block bootstrap:   0%|          | 0/2000 [00:00<?, ?sample/s]

Completed samples: 2000
Fixed weights: ResNet=0.75, Random Forest=0.25
Positive ROC-AUC/AP differences favour the ensemble.
Negative log-loss/Brier differences favour the ensemble.

The test set remains untouched.


,comparison,metric,observed_difference,bootstrap_mean_difference,ci_2_5_pct,ci_97_5_pct,ensemble_win_rate,ci_excludes_zero,better_direction
0,ensemble_vs_resnet,roc_auc_difference,0.000212,-0.000060,-0.004000,0.003933,0.4905,False,higher_is_better
1,ensemble_vs_resnet,average_precision_difference,0.000210,0.000189,-0.003266,0.004031,0.5330,False,higher_is_better
2,ensemble_vs_resnet,log_loss_difference,-0.001636,-0.001901,-0.003927,-0.000082,0.9805,True,lower_is_better
3,ensemble_vs_resnet,brier_difference,-0.000772,-0.000899,-0.001824,-0.000056,0.9845,True,lower_is_better
4,ensemble_vs_random_forest,roc_auc_difference,0.011370,0.009525,-0.014009,0.032766,0.7770,False,higher_is_better
5,ensemble_vs_random_forest,average_precision_difference,0.008429,0.008003,-0.018311,0.037129,0.6910,False,higher_is_better
6,ensemble_vs_random_forest,log_loss_difference,-0.000927,-0.000324,-0.006257,0.005266,0.5345,False,lower_is_better
7,ensemble_vs_random_forest,brier_difference,-0.000471,-0.000173,-0.003072,0.002556,0.5395,False,lower_is_better


## ResNet-Random Forest Ensemble Result

The selected ensemble used:

```text
75% Tabular ResNet
25% Random Forest
```

### Ensemble versus ResNet

The ensemble did not reliably improve ranking performance.

| Metric | Result |
|---|---|
| ROC-AUC | No reliable difference |
| Average precision | No reliable difference |
| Log loss | Ensemble reliably better |
| Brier score | Ensemble reliably better |

The ensemble and ResNet therefore rank stocks similarly, but the ensemble
produces more reliable probability estimates.

### Ensemble versus Random Forest

The ensemble achieved higher observed ROC-AUC and average precision than the
Random Forest.

However, the 20-day block-bootstrap confidence intervals included zero.

This means the ranking advantage was not consistent across all sampled market
periods.

### Decision

The 75% ResNet and 25% Random Forest ensemble is the leading validation
candidate because it:

- Retains the ResNet's stronger ranking performance
- Improves probability calibration
- Combines two moderately different models

The ensemble weights are now fixed and will not be changed.

The next model to evaluate is the FT-Transformer.

The test set remains untouched.

# FT-Transformer Experiment

## Purpose

The Tabular ResNet improved validation ranking performance, and the
75% ResNet plus 25% Random Forest ensemble improved probability quality.

The next experiment evaluates an FT-Transformer.

Unlike the MLP and ResNet, the FT-Transformer represents each numerical feature
as a separate learned token and uses attention to model relationships between
features.

```text
79 numerical features
        ↓
79 learned feature tokens
        ↓
Transformer attention blocks
        ↓
Classification token
        ↓
Outperformance probability
```

The model may learn interactions such as:

```text
Stock momentum
× market volatility
× sector strength
× broader market regime
```

## Experimental Controls

The FT-Transformer will use the same:

- 79 ordered features
- Purged training and validation rows
- Training-only preprocessing
- Outperformance target
- Evaluation metrics
- Untouched test period

## Evaluation

The FT-Transformer will be compared against:

1. Random Forest
2. Tabular MLP
3. Tabular ResNet
4. 75% ResNet + 25% Random Forest ensemble

The test set will remain untouched until the final model is selected.

In [20]:
# ---------------------------------------------------------
# FT-Transformer configuration
# ---------------------------------------------------------

FT_TRANSFORMER_CONFIG = {
    "input_dim": len(feature_cols),
    "token_dim": 96,
    "num_heads": 8,
    "num_blocks": 3,
    "ffn_dim": 192,
    "attention_dropout": 0.15,
    "ffn_dropout": 0.15,
    "residual_dropout": 0.10,
}


class NumericalFeatureTokenizer(nn.Module):
    """Convert every numerical feature into its own learned token."""

    def __init__(self, input_dim: int, token_dim: int) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.weight = nn.Parameter(torch.empty(input_dim, token_dim))
        self.bias = nn.Parameter(torch.empty(input_dim, token_dim))
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if features.ndim != 2 or features.shape[1] != self.input_dim:
            raise ValueError(f"Expected [batch, {self.input_dim}], found {tuple(features.shape)}.")

        return features.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class FTTransformerBlock(nn.Module):
    """Pre-normalized Transformer block with attention and feed-forward layers."""

    def __init__(
        self,
        token_dim: int,
        num_heads: int,
        ffn_dim: int,
        attention_dropout: float,
        ffn_dropout: float,
        residual_dropout: float,
    ) -> None:
        super().__init__()

        if token_dim % num_heads != 0:
            raise ValueError("token_dim must be divisible by num_heads.")

        self.attention_norm = nn.LayerNorm(token_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=token_dim,
            num_heads=num_heads,
            dropout=attention_dropout,
            batch_first=True,
        )
        self.attention_residual_dropout = nn.Dropout(residual_dropout)

        self.ffn_norm = nn.LayerNorm(token_dim)
        self.ffn = nn.Sequential(
            nn.Linear(token_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(ffn_dropout),
            nn.Linear(ffn_dim, token_dim),
        )
        self.ffn_residual_dropout = nn.Dropout(residual_dropout)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        normalized_tokens = self.attention_norm(tokens)
        attention_output, _ = self.attention(
            normalized_tokens,
            normalized_tokens,
            normalized_tokens,
            need_weights=False,
        )
        tokens = tokens + self.attention_residual_dropout(attention_output)
        tokens = tokens + self.ffn_residual_dropout(self.ffn(self.ffn_norm(tokens)))
        return tokens


class FTTransformer(nn.Module):
    """FT-Transformer for numerical tabular binary classification."""

    def __init__(
        self,
        input_dim: int,
        token_dim: int,
        num_heads: int,
        num_blocks: int,
        ffn_dim: int,
        attention_dropout: float,
        ffn_dropout: float,
        residual_dropout: float,
    ) -> None:
        super().__init__()

        self.tokenizer = NumericalFeatureTokenizer(input_dim, token_dim)
        self.classification_token = nn.Parameter(torch.zeros(1, 1, token_dim))

        self.blocks = nn.ModuleList([
            FTTransformerBlock(
                token_dim=token_dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                attention_dropout=attention_dropout,
                ffn_dropout=ffn_dropout,
                residual_dropout=residual_dropout,
            )
            for _ in range(num_blocks)
        ])

        self.output_head = nn.Sequential(
            nn.LayerNorm(token_dim),
            nn.GELU(),
            nn.Linear(token_dim, 1),
        )

        nn.init.normal_(self.classification_token, mean=0.0, std=0.02)
        self._initialize_linear_layers()

    def _initialize_linear_layers(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        feature_tokens = self.tokenizer(features)
        cls_token = self.classification_token.expand(features.shape[0], -1, -1)
        tokens = torch.cat([cls_token, feature_tokens], dim=1)

        for block in self.blocks:
            tokens = block(tokens)

        return self.output_head(tokens[:, 0]).squeeze(-1)


# ---------------------------------------------------------
# Create and audit the model
# ---------------------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

outperform_ft_transformer = FTTransformer(**FT_TRANSFORMER_CONFIG).to(DEVICE)
ft_parameter_counts = count_model_parameters(outperform_ft_transformer)

ft_dry_run_features = sample_features[:32].to(DEVICE, non_blocking=PIN_MEMORY)

outperform_ft_transformer.eval()

with torch.inference_mode():
    ft_dry_run_logits = outperform_ft_transformer(ft_dry_run_features)
    ft_dry_run_probabilities = torch.sigmoid(ft_dry_run_logits)

if ft_dry_run_logits.shape != torch.Size([32]):
    raise ValueError(f"Unexpected output shape: {ft_dry_run_logits.shape}")

if not torch.isfinite(ft_dry_run_logits).all():
    raise ValueError("FT-Transformer produced non-finite logits.")

outperform_ft_transformer.train()

print(outperform_ft_transformer)
print("\nInput features:", FT_TRANSFORMER_CONFIG["input_dim"])
print("Tokens per row:", FT_TRANSFORMER_CONFIG["input_dim"] + 1)
print("Token dimension:", FT_TRANSFORMER_CONFIG["token_dim"])
print("Attention heads:", FT_TRANSFORMER_CONFIG["num_heads"])
print("Transformer blocks:", FT_TRANSFORMER_CONFIG["num_blocks"])
print("Total parameters:", f"{ft_parameter_counts['total_parameters']:,}")
print("Trainable parameters:", f"{ft_parameter_counts['trainable_parameters']:,}")
print("\nDry-run device:", ft_dry_run_logits.device)
print("Dry-run output shape:", ft_dry_run_logits.shape)
print(
    "Initial probability range:",
    round(float(ft_dry_run_probabilities.min()), 6),
    "to",
    round(float(ft_dry_run_probabilities.max()), 6),
)

if DEVICE.type == "cuda":
    print("Allocated GPU memory:", round(torch.cuda.memory_allocated() / 1024**2, 2), "MB")

FTTransformer(
  (tokenizer): NumericalFeatureTokenizer()
  (blocks): ModuleList(
    (0-2): 3 x FTTransformerBlock(
      (attention_norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
      )
      (attention_residual_dropout): Dropout(p=0.1, inplace=False)
      (ffn_norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): Sequential(
        (0): Linear(in_features=96, out_features=192, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.15, inplace=False)
        (3): Linear(in_features=192, out_features=96, bias=True)
      )
      (ffn_residual_dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (output_head): Sequential(
    (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=96, out_features=1, bias=

In [21]:
# ---------------------------------------------------------
# FT-Transformer training configuration
# ---------------------------------------------------------

FT_TRAINING_CONFIG = {
    "max_epochs": 40,
    "learning_rate": 2e-4,
    "weight_decay": 1e-5,
    "early_stopping_patience": 10,
    "minimum_auc_improvement": 1e-4,
    "gradient_clip_norm": 1.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 2,
    "minimum_learning_rate": 1e-6,
}

FT_TRAIN_BATCH_SIZE = 1024
FT_EVAL_BATCH_SIZE = 2048

FT_CHECKPOINT_PATH = MODEL_DIR / "ft_transformer_outperform_purged_v1.pt"
FT_HISTORY_PATH = REPORT_DIR / "ft_transformer_outperform_training_history_v1.csv"


# ---------------------------------------------------------
# Reproducibility and data loaders
# ---------------------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.cuda.empty_cache()

ft_train_loader = create_data_loader(outperform_train_dataset, FT_TRAIN_BATCH_SIZE, True, RANDOM_SEED)
ft_train_eval_loader = create_data_loader(outperform_train_dataset, FT_EVAL_BATCH_SIZE, False, RANDOM_SEED)
ft_valid_loader = create_data_loader(outperform_valid_dataset, FT_EVAL_BATCH_SIZE, False, RANDOM_SEED)


# ---------------------------------------------------------
# Fresh model, loss, optimizer and scheduler
# ---------------------------------------------------------

outperform_ft_transformer = FTTransformer(**FT_TRANSFORMER_CONFIG).to(DEVICE)
ft_loss_function = nn.BCEWithLogitsLoss()

ft_optimizer = torch.optim.AdamW(
    outperform_ft_transformer.parameters(),
    lr=FT_TRAINING_CONFIG["learning_rate"],
    weight_decay=FT_TRAINING_CONFIG["weight_decay"],
)

ft_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    ft_optimizer,
    mode="max",
    factor=FT_TRAINING_CONFIG["scheduler_factor"],
    patience=FT_TRAINING_CONFIG["scheduler_patience"],
    threshold=FT_TRAINING_CONFIG["minimum_auc_improvement"],
    min_lr=FT_TRAINING_CONFIG["minimum_learning_rate"],
)


# ---------------------------------------------------------
# Training loop
# ---------------------------------------------------------

ft_history = []
ft_best_epoch = 0
ft_best_valid_auc = -np.inf
ft_best_state = None
ft_epochs_without_improvement = 0
ft_training_start = time.perf_counter()

epoch_progress = tqdm(
    range(1, FT_TRAINING_CONFIG["max_epochs"] + 1),
    desc="Outperformance FT-Transformer",
    unit="epoch",
    dynamic_ncols=True,
)

for epoch in epoch_progress:
    epoch_start = time.perf_counter()
    current_lr = ft_optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch_with_progress(
        model=outperform_ft_transformer,
        data_loader=ft_train_loader,
        optimizer=ft_optimizer,
        loss_function=ft_loss_function,
        device=DEVICE,
        epoch=epoch,
        max_gradient_norm=FT_TRAINING_CONFIG["gradient_clip_norm"],
    )

    valid_metrics, _, _ = evaluate_binary_model(
        model=outperform_ft_transformer,
        data_loader=ft_valid_loader,
        loss_function=ft_loss_function,
        device=DEVICE,
        threshold=0.50,
    )

    valid_auc = valid_metrics["roc_auc"]
    ft_scheduler.step(valid_auc)
    next_lr = ft_optimizer.param_groups[0]["lr"]
    epoch_seconds = time.perf_counter() - epoch_start

    ft_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_metrics["loss"],
        "valid_roc_auc": valid_auc,
        "valid_average_precision": valid_metrics["average_precision"],
        "valid_log_loss": valid_metrics["log_loss"],
        "valid_brier_score": valid_metrics["brier_score"],
        "valid_accuracy": valid_metrics["accuracy"],
        "valid_balanced_accuracy": valid_metrics["balanced_accuracy"],
        "valid_precision": valid_metrics["precision"],
        "valid_recall": valid_metrics["recall"],
        "valid_f1": valid_metrics["f1"],
        "learning_rate": current_lr,
        "next_learning_rate": next_lr,
        "epoch_seconds": epoch_seconds,
    })

    improved = valid_auc > ft_best_valid_auc + FT_TRAINING_CONFIG["minimum_auc_improvement"]

    if improved:
        ft_best_epoch = epoch
        ft_best_valid_auc = valid_auc
        ft_best_state = {name: tensor.detach().cpu().clone() for name, tensor in outperform_ft_transformer.state_dict().items()}
        ft_epochs_without_improvement = 0
        tqdm.write(f"New best FT epoch {epoch:02d}: validation AUC={valid_auc:.6f}, AP={valid_metrics['average_precision']:.6f}")
    else:
        ft_epochs_without_improvement += 1

    epoch_progress.set_postfix({
        "train": f"{train_loss:.5f}",
        "valid": f"{valid_metrics['loss']:.5f}",
        "auc": f"{valid_auc:.5f}",
        "best": f"{ft_best_valid_auc:.5f}",
        "AP": f"{valid_metrics['average_precision']:.5f}",
        "patience": f"{ft_epochs_without_improvement}/{FT_TRAINING_CONFIG['early_stopping_patience']}",
        "lr": f"{next_lr:.1e}",
        "sec": f"{epoch_seconds:.1f}",
    })

    if ft_epochs_without_improvement >= FT_TRAINING_CONFIG["early_stopping_patience"]:
        tqdm.write(f"FT-Transformer early stopping after {ft_epochs_without_improvement} epochs without improvement.")
        break

ft_training_seconds = time.perf_counter() - ft_training_start

if ft_best_state is None:
    raise RuntimeError("FT-Transformer training did not produce a valid checkpoint.")


# ---------------------------------------------------------
# Restore and evaluate best checkpoint
# ---------------------------------------------------------

outperform_ft_transformer.load_state_dict(ft_best_state)
outperform_ft_transformer.to(DEVICE)
outperform_ft_transformer.eval()

ft_train_metrics, _, ft_train_probabilities = evaluate_binary_model(
    outperform_ft_transformer, ft_train_eval_loader, ft_loss_function, DEVICE, 0.50
)

ft_valid_metrics, ft_valid_targets, ft_valid_probabilities = evaluate_binary_model(
    outperform_ft_transformer, ft_valid_loader, ft_loss_function, DEVICE, 0.50
)

if not np.isclose(ft_valid_metrics["roc_auc"], ft_best_valid_auc, atol=1e-10):
    raise ValueError("Restored FT-Transformer checkpoint does not match the best validation AUC.")


# ---------------------------------------------------------
# Save checkpoint and history
# ---------------------------------------------------------

ft_history_df = pd.DataFrame(ft_history)
ft_history_df.to_csv(FT_HISTORY_PATH, index=False)

torch.save({
    "model_name": "ft_transformer_outperform",
    "model_version": "v1",
    "target": OUTPERFORM_TARGET,
    "split_method": PRIMARY_SPLIT_METHOD,
    "random_seed": RANDOM_SEED,
    "model_config": FT_TRANSFORMER_CONFIG,
    "training_config": FT_TRAINING_CONFIG,
    "feature_cols": feature_cols,
    "best_epoch": ft_best_epoch,
    "best_valid_metrics": ft_valid_metrics,
    "training_duration_seconds": ft_training_seconds,
    "state_dict": ft_best_state,
}, FT_CHECKPOINT_PATH)


# ---------------------------------------------------------
# Display result
# ---------------------------------------------------------

ft_summary = pd.DataFrame([
    {"model": "ft_transformer", "split": "train", **ft_train_metrics},
    {"model": "ft_transformer", "split": "valid", **ft_valid_metrics},
])

print("\nFT-Transformer training completed.")
print("Best epoch:", ft_best_epoch)
print("Best validation ROC-AUC:", round(ft_best_valid_auc, 6))
print("Total training time:", round(ft_training_seconds, 2), "seconds")
print("Checkpoint:", FT_CHECKPOINT_PATH)
print("Training history:", FT_HISTORY_PATH)
print("\nThe test set has not been evaluated.")

display(ft_summary[[
    "model",
    "split",
    "rows",
    "loss",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
]])

Outperformance FT-Transformer:   0%|          | 0/40 [00:00<?, ?epoch/s]

Epoch 01 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

New best FT epoch 01: validation AUC=0.509422, AP=0.515528


Epoch 02 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Epoch 03 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

...

Epoch 20 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Epoch 21 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Epoch 22 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

FT-Transformer early stopping after 10 epochs without improvement.

FT-Transformer training completed.
Best epoch: 12
Best validation ROC-AUC: 0.524744
Total training time: 798.31 seconds
Checkpoint: E:\Projects\marketguard-india\models\deep_learning\ft_transformer_outperform_purged_v1.pt
Training history: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\ft_transformer_outperform_training_history_v1.csv

The test set has not been evaluated.


,model,split,rows,loss,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1
0,ft_transformer,train,251738,0.643719,0.672072,0.679202,0.643719,0.226689,0.619013,0.617999,0.620421,0.659722,0.639468
1,ft_transformer,valid,19980,0.745740,0.524744,0.516713,0.745740,0.270616,0.516767,0.518102,0.531768,0.458835,0.492617


## FT-Transformer Result

The FT-Transformer reached its best validation result at epoch 12.

| Split | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Train | 0.6721 | 0.6792 | 0.6437 | 0.2267 |
| Validation | 0.5247 | 0.5167 | 0.7457 | 0.2706 |

The large difference between training and validation performance shows strong
overfitting.

The FT-Transformer performed worse than:

- Random Forest
- Tabular MLP
- Tabular ResNet
- ResNet-Random Forest ensemble

It was also substantially worse on log loss and Brier score, meaning its
probability estimates were unreliable.

### Decision

The FT-Transformer will not be considered for production or ensemble selection.

No bootstrap analysis is required because it did not produce an observed
validation improvement.

The leading candidate remains:

```text
75% Tabular ResNet
25% Random Forest
```

The test set remains untouched.

In [22]:
import pytorch_tabnet
from pytorch_tabnet.tab_model import TabNetClassifier

print("pytorch-tabnet imported successfully")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

pytorch-tabnet imported successfully
PyTorch version: 2.13.0+cu126
CUDA available: True


# TabNet Experiment

## Purpose

The next model is TabNet.

TabNet is a deep learning model designed specifically for tabular data. It uses
attention to select the most useful features during each decision step.

```text
79 input features
        ↓
Feature selection
        ↓
Decision step 1
        ↓
Feature selection
        ↓
Decision step 2
        ↓
Additional decision steps
        ↓
Outperformance probability
```

Unlike a normal MLP, TabNet does not treat all features equally at every stage.
It learns which features should receive more attention for each prediction.

## Why Test TabNet?

TabNet may help because MarketGuard contains many related features such as:

- Momentum
- Volatility
- Volume
- Market conditions
- Sector performance
- Technical indicators

The model may learn to focus on different feature groups under different market
conditions.

TabNet also provides feature-importance information, which can help explain
which inputs influence its predictions.

## Experimental Controls

TabNet will use the same:

- 79 ordered features
- Purged training split
- Purged validation split
- Training-only preprocessing
- Outperformance target
- Evaluation metrics
- Untouched test period

This allows a fair comparison with:

1. Random Forest
2. Tabular MLP
3. Tabular ResNet
4. FT-Transformer
5. ResNet-Random Forest ensemble

## Evaluation

TabNet will be evaluated using:

- ROC-AUC
- Average precision
- Log loss
- Brier score
- Balanced accuracy
- Precision
- Recall
- F1 score

The model will use early stopping based on validation ROC-AUC.

The test set will remain untouched until the final candidate is selected.

In [24]:
import math
from tqdm.auto import tqdm
from pytorch_tabnet.callbacks import Callback


class TabNetTQDMCallback(Callback):
    """Display epoch-level and batch-level TabNet progress."""

    def __init__(self, max_epochs, steps_per_epoch, metric_name="valid_auc"):
        super().__init__()
        self.max_epochs = max_epochs
        self.steps_per_epoch = steps_per_epoch
        self.metric_name = metric_name
        self.best_metric = -np.inf
        self.epoch_bar = None
        self.batch_bar = None

    def on_train_begin(self, logs=None):
        self.epoch_bar = tqdm(total=self.max_epochs, desc="Outperformance TabNet", unit="epoch", dynamic_ncols=True)

    def on_epoch_begin(self, epoch, logs=None):
        self.batch_bar = tqdm(total=self.steps_per_epoch, desc=f"Epoch {epoch + 1:03d}", unit="batch", leave=False, dynamic_ncols=True)

    def on_batch_end(self, batch, logs=None):
        logs = logs or {}
        self.batch_bar.update(1)

        if "loss" in logs:
            self.batch_bar.set_postfix(loss=f"{logs['loss']:.5f}")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        if self.batch_bar is not None:
            self.batch_bar.close()

        current_metric = logs.get(self.metric_name)

        if current_metric is not None:
            self.best_metric = max(self.best_metric, current_metric)

        postfix = {}

        if "loss" in logs:
            postfix["train"] = f"{logs['loss']:.5f}"

        if current_metric is not None:
            postfix["valid_auc"] = f"{current_metric:.5f}"
            postfix["best"] = f"{self.best_metric:.5f}"

        if "lr" in logs:
            postfix["lr"] = f"{logs['lr']:.2e}"

        self.epoch_bar.update(1)
        self.epoch_bar.set_postfix(postfix)

    def on_train_end(self, logs=None):
        if self.batch_bar is not None:
            self.batch_bar.close()

        if self.epoch_bar is not None:
            self.epoch_bar.close()


# TabNet data
X_tabnet_train = np.ascontiguousarray(X_train, dtype=np.float32)
X_tabnet_valid = np.ascontiguousarray(X_valid, dtype=np.float32)
y_tabnet_train = np.asarray(y_train_outperform, dtype=np.int64).reshape(-1)
y_tabnet_valid = np.asarray(y_valid_outperform, dtype=np.int64).reshape(-1)

if not np.isfinite(X_tabnet_train).all() or not np.isfinite(X_tabnet_valid).all():
    raise ValueError("TabNet features contain missing or infinite values.")

print("Train:", X_tabnet_train.shape, y_tabnet_train.shape)
print("Valid:", X_tabnet_valid.shape, y_tabnet_valid.shape)


# Configuration
TABNET_MAX_EPOCHS = 100
TABNET_PATIENCE = 15
TABNET_BATCH_SIZE = 4096
TABNET_VIRTUAL_BATCH_SIZE = 256

TABNET_CONFIG = {
    "n_d": 32,
    "n_a": 32,
    "n_steps": 5,
    "gamma": 1.5,
    "n_independent": 2,
    "n_shared": 2,
    "lambda_sparse": 1e-4,
    "momentum": 0.02,
    "clip_value": 1.0,
    "mask_type": "sparsemax",
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": {"lr": 2e-2, "weight_decay": 1e-5},
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {"step_size": 10, "gamma": 0.5},
    "seed": RANDOM_SEED,
    "verbose": 0,
    "device_name": "cuda",
}

TABNET_MODEL_PATH = MODEL_DIR / "tabnet_outperform_purged_v1"
TABNET_HISTORY_PATH = REPORT_DIR / "tabnet_outperform_training_history_v1.csv"

steps_per_epoch = math.ceil(len(X_tabnet_train) / TABNET_BATCH_SIZE)
tabnet_progress = TabNetTQDMCallback(TABNET_MAX_EPOCHS, steps_per_epoch)


# Train
tabnet_model = TabNetClassifier(**TABNET_CONFIG)
tabnet_training_start = time.perf_counter()

tabnet_model.fit(
    X_train=X_tabnet_train,
    y_train=y_tabnet_train,
    eval_set=[(X_tabnet_valid, y_tabnet_valid)],
    eval_name=["valid"],
    eval_metric=["auc"],
    max_epochs=TABNET_MAX_EPOCHS,
    patience=TABNET_PATIENCE,
    batch_size=TABNET_BATCH_SIZE,
    virtual_batch_size=TABNET_VIRTUAL_BATCH_SIZE,
    num_workers=0,
    drop_last=False,
    pin_memory=True,
    weights=0,
    callbacks=[tabnet_progress],
)

tabnet_training_seconds = time.perf_counter() - tabnet_training_start


# Evaluate
tabnet_train_probability = tabnet_model.predict_proba(X_tabnet_train)[:, 1]
tabnet_valid_probability = tabnet_model.predict_proba(X_tabnet_valid)[:, 1]

tabnet_train_metrics = calculate_binary_metrics(y_tabnet_train, tabnet_train_probability, 0.50)
tabnet_valid_metrics = calculate_binary_metrics(y_tabnet_valid, tabnet_valid_probability, 0.50)


# Save
saved_tabnet_path = tabnet_model.save_model(str(TABNET_MODEL_PATH))
tabnet_history_data = getattr(tabnet_model.history, "history", tabnet_model.history)
tabnet_history_df = pd.DataFrame(tabnet_history_data)
tabnet_history_df.to_csv(TABNET_HISTORY_PATH, index=False)

tabnet_summary = pd.DataFrame([
    {"model": "tabnet", "split": "train", **tabnet_train_metrics},
    {"model": "tabnet", "split": "valid", **tabnet_valid_metrics},
])


# Results
print("\nTabNet training completed.")
print("Best epoch:", getattr(tabnet_model, "best_epoch", "not available"))
print("Best validation AUC:", getattr(tabnet_model, "best_cost", "not available"))
print("Training time:", round(tabnet_training_seconds, 2), "seconds")
print("Saved model:", saved_tabnet_path)
print("Training history:", TABNET_HISTORY_PATH)
print("\nThe test set has not been evaluated.")

display(tabnet_summary[[
    "model", "split", "rows", "roc_auc", "average_precision", "log_loss",
    "brier_score", "accuracy", "balanced_accuracy", "precision", "recall", "f1",
]])

Train: (251738, 79) (251738,)
Valid: (19980, 79) (19980,)


Outperformance TabNet:   0%|          | 0/100 [00:00<?, ?epoch/s]

Epoch 001:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 002:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 003:   0%|          | 0/62 [00:00<?, ?batch/s]

...

Epoch 024:   0%|          | 0/62 [00:00<?, ?batch/s]


Early stopping occurred at epoch 23 with best_epoch = 8 and best_valid_auc = 0.51675


c:\Users\ppava\anaconda3\envs\marketguard\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Successfully saved model at E:\Projects\marketguard-india\models\deep_learning\tabnet_outperform_purged_v1.zip

TabNet training completed.
Best epoch: 8
Best validation AUC: 0.5167502435476478
Training time: 244.59 seconds
Saved model: E:\Projects\marketguard-india\models\deep_learning\tabnet_outperform_purged_v1.zip
Training history: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\tabnet_outperform_training_history_v1.csv

The test set has not been evaluated.


,model,split,rows,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1
0,tabnet,train,251738,0.537716,0.545732,0.690779,0.248807,0.526671,0.526068,0.536939,0.55087,0.543815
1,tabnet,valid,19980,0.516750,0.524671,0.693347,0.250088,0.510711,0.509479,0.519798,0.56417,0.541076


## TabNet Result

TabNet reached its best validation result at epoch 8.

| Split | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Train | 0.5377 | 0.5457 | 0.6908 | 0.2488 |
| Validation | 0.5168 | 0.5247 | 0.6933 | 0.2501 |

TabNet performed worse than:

- Random Forest
- Tabular MLP
- Tabular ResNet
- FT-Transformer
- ResNet-Random Forest ensemble

The training ROC-AUC was also low, which suggests that this TabNet configuration
did not learn enough useful structure from the data.

### Decision

TabNet will not be considered for final model selection or ensemble testing.

No bootstrap analysis is needed because it did not improve the observed
validation results.

The leading candidate remains:

```text
75% Tabular ResNet
25% Random Forest
```

The test set remains untouched.

### final validation comparison and formally lock the winning model.

In [25]:
import json

# Final validation probabilities
final_validation_probabilities = {
    "random_forest": np.asarray(rf_validation_probability, dtype=np.float64),
    "tabular_mlp": np.asarray(mlp_validation_probability, dtype=np.float64),
    "mlp_30_rf_70": np.asarray(mlp_rf_ensemble_probability, dtype=np.float64),
    "tabular_resnet": np.asarray(resnet_valid_probability, dtype=np.float64),
    "resnet_75_rf_25": 0.75 * np.asarray(resnet_valid_probability) + 0.25 * np.asarray(rf_validation_probability),
    "ft_transformer": np.asarray(ft_valid_probabilities, dtype=np.float64),
    "tabnet": np.asarray(tabnet_valid_probability, dtype=np.float64),
}

final_validation_rows = []

for model_name, probability in final_validation_probabilities.items():
    metrics = calculate_binary_metrics(validation_targets, probability, 0.50)
    final_validation_rows.append({"model": model_name, **metrics})

final_validation_comparison = pd.DataFrame(final_validation_rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

FINAL_VALIDATION_PATH = REPORT_DIR / "outperform_final_validation_comparison_v1.csv"
FINAL_SELECTION_PATH = REPORT_DIR / "outperform_final_model_selection_v1.json"

final_validation_comparison.to_csv(FINAL_VALIDATION_PATH, index=False)

final_model_selection = {
    "target": OUTPERFORM_TARGET,
    "selected_candidate": "resnet_75_rf_25",
    "resnet_weight": 0.75,
    "random_forest_weight": 0.25,
    "selection_split": "purged_validation",
    "selection_reason": "Best overall validation ranking with reliably better calibration than ResNet alone.",
    "test_used_for_selection": False,
}

with open(FINAL_SELECTION_PATH, "w", encoding="utf-8") as file:
    json.dump(final_model_selection, file, indent=2)

print("Final candidate locked: 75% Tabular ResNet + 25% Random Forest")
print("Validation report:", FINAL_VALIDATION_PATH)
print("Selection record:", FINAL_SELECTION_PATH)
print("\nThe test set remains untouched.")

display(final_validation_comparison[[
    "model", "rows", "roc_auc", "average_precision", "log_loss", "brier_score",
    "balanced_accuracy", "precision", "recall", "f1", "average_probability",
]])

Final candidate locked: 75% Tabular ResNet + 25% Random Forest
Validation report: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\outperform_final_validation_comparison_v1.csv
Selection record: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\outperform_final_model_selection_v1.json

The test set remains untouched.


,model,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,resnet_75_rf_25,19980,0.540278,0.539083,0.691047,0.248938,0.530514,0.536247,0.640137,0.583605,0.511438
1,tabular_resnet,19980,0.540066,0.538873,0.692683,0.249710,0.529274,0.535789,0.625061,0.576993,0.511354
2,mlp_30_rf_70,19980,0.534681,0.531370,0.691136,0.248993,0.522344,0.528481,0.670289,0.590997,0.511467
3,tabular_mlp,19980,0.530320,0.528792,0.693764,0.250238,0.522063,0.529677,0.620264,0.571403,0.510942
4,random_forest,19980,0.528908,0.530654,0.691974,0.249410,0.519090,0.525825,0.673715,0.590654,0.511692
5,ft_transformer,19980,0.524744,0.516713,0.745740,0.270616,0.518102,0.531768,0.458835,0.492617,0.469132
6,tabnet,19980,0.516750,0.524671,0.693347,0.250088,0.509479,0.519798,0.564170,0.541076,0.505939


### Validation on Test Set

In [26]:
# ---------------------------------------------------------
# One-time final test evaluation
# ---------------------------------------------------------

FINAL_RESNET_WEIGHT = 0.75
FINAL_RF_WEIGHT = 0.25

with open(FINAL_SELECTION_PATH, "r", encoding="utf-8") as file:
    locked_selection = json.load(file)

if locked_selection["selected_candidate"] != "resnet_75_rf_25":
    raise ValueError("The locked candidate does not match the planned final ensemble.")

if locked_selection["test_used_for_selection"]:
    raise ValueError("The selection record incorrectly indicates that test data influenced selection.")

test_targets = np.asarray(y_test_outperform, dtype=np.int64)
X_test_rf = model_data.loc[test_mask, feature_cols]

rf_test_probability = purged_rf_outperform.predict_proba(X_test_rf)[:, 1]

final_test_loader = create_data_loader(
    dataset=outperform_test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED,
)

_, resnet_test_targets, resnet_test_probability = evaluate_binary_model(
    model=outperform_resnet,
    data_loader=final_test_loader,
    loss_function=resnet_loss_function,
    device=DEVICE,
    threshold=0.50,
)

resnet_test_targets = np.asarray(resnet_test_targets, dtype=np.int64)
resnet_test_probability = np.asarray(resnet_test_probability, dtype=np.float64)

if not np.array_equal(test_targets, resnet_test_targets):
    raise ValueError("ResNet test targets are not aligned with the Random Forest test rows.")

if len(rf_test_probability) != len(test_targets):
    raise ValueError("Random Forest test probabilities are not aligned with the test targets.")

final_ensemble_test_probability = FINAL_RESNET_WEIGHT * resnet_test_probability + FINAL_RF_WEIGHT * rf_test_probability

final_test_probabilities = {
    "random_forest": rf_test_probability,
    "tabular_resnet": resnet_test_probability,
    "resnet_75_rf_25": final_ensemble_test_probability,
}

final_test_rows = []

for model_name, probability in final_test_probabilities.items():
    metrics = calculate_binary_metrics(test_targets, probability, 0.50)
    final_test_rows.append({"model": model_name, **metrics})

final_test_comparison = pd.DataFrame(final_test_rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

FINAL_TEST_PATH = REPORT_DIR / "outperform_final_test_comparison_v1.csv"
final_test_comparison.to_csv(FINAL_TEST_PATH, index=False)

print("Final test evaluation completed.")
print("Locked candidate: 75% Tabular ResNet + 25% Random Forest")
print("Test period:", model_data.loc[test_mask, "date"].min().date(), "to", model_data.loc[test_mask, "date"].max().date())
print("Test rows:", len(test_targets))
print("Test report:", FINAL_TEST_PATH)
print("\nNo model or ensemble weights will be changed using these results.")

display(final_test_comparison[[
    "model", "rows", "roc_auc", "average_precision", "log_loss", "brier_score",
    "accuracy", "balanced_accuracy", "precision", "recall", "f1", "average_probability",
]])

Final test evaluation completed.
Locked candidate: 75% Tabular ResNet + 25% Random Forest
Test period: 2025-01-01 to 2026-06-16
Test rows: 32135
Test report: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\outperform_final_test_comparison_v1.csv

No model or ensemble weights will be changed using these results.


,model,rows,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1,average_probability
0,random_forest,32135,0.514088,0.529567,0.693999,0.250413,0.518189,0.512633,0.526829,0.673344,0.591143,0.520193
1,resnet_75_rf_25,32135,0.505105,0.522364,0.698421,0.252572,0.509942,0.503649,0.519958,0.685676,0.591428,0.524448
2,tabular_resnet,32135,0.502376,0.521298,0.701652,0.254122,0.508884,0.503120,0.519623,0.669855,0.585252,0.525866


## Final Untouched Test Result

The final locked candidate was:

```text
75% Tabular ResNet
25% Random Forest
```

It was selected using the purged validation period without using the test data.

The candidate was then evaluated once on the untouched test period from
2025-01-01 through 2026-06-16.

### Test Results

| Model | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Random Forest | **0.5141** | **0.5296** | **0.6940** | **0.2504** |
| ResNet-RF ensemble | 0.5051 | 0.5224 | 0.6984 | 0.2526 |
| Tabular ResNet | 0.5024 | 0.5213 | 0.7017 | 0.2541 |

### Interpretation

The ResNet validation improvement did not generalize to the future test period.

The standalone ResNet performed close to random ranking, and adding it to the
Random Forest reduced both ranking and probability quality.

The Random Forest produced the strongest test result among the evaluated final
candidates.

### Final Decision

The ResNet and ResNet-Random-Forest ensemble will not replace the existing
Random Forest outperformance model.

The Random Forest remains the preferred model for the current project version.

No model configuration or ensemble weight will be changed using the test
results.

The deep-learning experiment is considered complete and unsuccessful for model
promotion, but successful as a controlled research experiment.

Possible future improvements include:

- Point-in-time stock-universe data
- Cross-sectional ranking targets
- Walk-forward tuning
- Gradient-boosted tree models
- Improved market-regime and sector-relative features

# Downside-Risk Deep Learning Experiment

## Purpose

This experiment predicts:

`target_big_downside_10pct_20d`

The target equals `1` when a stock falls by at least 10% at any point during the
next 20 trading days.

```text
Current stock information
        ↓
Model
        ↓
Probability of a 10% downside event
```

## Why This Target Is Different

The outperformance target was close to balanced.

The downside target is imbalanced:

| Split | Downside-event rate |
|---|---:|
| Purged train | 14.85% |
| Purged validation | 10.49% |
| Test | 11.20% |

Therefore, accuracy alone is not useful. A model predicting no downside event
for every row would still achieve high accuracy.

The main metrics will be:

- ROC-AUC
- Average precision
- Recall
- Precision
- Log loss
- Brier score

## Experimental Plan

The downside experiment will compare:

1. Purged Random Forest baseline
2. Tabular ResNet
3. ResNet-Random Forest ensemble

The existing downside Random Forest previously showed useful predictive
performance, so the neural model must improve it without reducing downside-event
detection.

The same controls will be used:

- Same 79 ordered features
- Same purged chronological split
- Same training-only preprocessing
- Same untouched test period
- Early stopping using validation ROC-AUC

The test period will remain untouched until the final downside candidate is
selected.

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib

DOWNSIDE_RF_PATH = MODEL_DIR / "random_forest_downside_purged_research_v1.joblib"
DOWNSIDE_RF_REPORT_PATH = REPORT_DIR / "random_forest_downside_purged_validation_v1.csv"

purged_rf_downside = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=200,
        min_samples_split=200,
        max_features="sqrt",
        class_weight="balanced_subsample",
        bootstrap=True,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

downside_rf_start = time.perf_counter()
purged_rf_downside.fit(X_train_raw, np.asarray(y_train_downside, dtype=np.int64))
downside_rf_seconds = time.perf_counter() - downside_rf_start

downside_rf_train_probability = purged_rf_downside.predict_proba(X_train_raw)[:, 1]
downside_rf_valid_probability = purged_rf_downside.predict_proba(X_valid_raw)[:, 1]

downside_rf_train_metrics = calculate_binary_metrics(y_train_downside, downside_rf_train_probability, 0.50)
downside_rf_valid_metrics = calculate_binary_metrics(y_valid_downside, downside_rf_valid_probability, 0.50)

downside_rf_summary = pd.DataFrame([
    {"model": "random_forest_downside", "split": "train", **downside_rf_train_metrics},
    {"model": "random_forest_downside", "split": "valid", **downside_rf_valid_metrics},
])

joblib.dump(purged_rf_downside, DOWNSIDE_RF_PATH)
downside_rf_summary.to_csv(DOWNSIDE_RF_REPORT_PATH, index=False)

print("Purged downside Random Forest training time:", round(downside_rf_seconds, 2), "seconds")
print("Validation ROC-AUC:", round(downside_rf_valid_metrics["roc_auc"], 6))
print("Validation average precision:", round(downside_rf_valid_metrics["average_precision"], 6))
print("Saved model:", DOWNSIDE_RF_PATH)
print("\nThe test set remains untouched.")

display_columns = ["model", "split", "rows", "roc_auc", "average_precision", "log_loss", "brier_score", "balanced_accuracy", "precision", "recall", "f1", "average_probability"]
display(downside_rf_summary[display_columns].round(6))

Purged downside Random Forest training time: 22.52 seconds
Validation ROC-AUC: 0.659263
Validation average precision: 0.202046
Saved model: E:\Projects\marketguard-india\models\deep_learning\random_forest_downside_purged_research_v1.joblib

The test set remains untouched.


,model,split,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,random_forest_downside,train,251738,0.804856,0.448937,0.577802,0.196260,0.728014,0.303364,0.760719,0.433753,0.445305
1,random_forest_downside,valid,19980,0.659263,0.202046,0.594743,0.202433,0.625919,0.191630,0.498092,0.276776,0.435097


## Purged Downside Random Forest Result

The Random Forest was retrained using the purged training and validation splits.

| Split | ROC-AUC | Average Precision | Precision | Recall |
|---|---:|---:|---:|---:|
| Train | 0.8049 | 0.4489 | 0.3034 | 0.7607 |
| Validation | 0.6593 | 0.2020 | 0.1916 | 0.4981 |

The downside-event rate in validation is approximately `10.49%`.

An average precision of `0.2020` is therefore meaningfully better than the
random baseline of approximately `0.1049`.

At the default threshold of `0.50`, the model detected approximately half of
the future 10% downside events.

The difference between training and validation performance indicates some
overfitting, but the validation result still shows useful predictive ability.

The average predicted probability was higher than the actual event rate because
the Random Forest used balanced class weights. Its raw probabilities should
therefore be treated mainly as risk scores unless they are calibrated later.

### Decision

The purged Random Forest is the downside-risk benchmark that the neural model
must beat.

The test set remains untouched.

### Downside Tabular ResNet

In [31]:
# ---------------------------------------------------------
# Downside Tabular ResNet
# ---------------------------------------------------------

DOWNSIDE_RESNET_CONFIG = TABULAR_RESNET_CONFIG.copy()
DOWNSIDE_RESNET_CONFIG.pop("output_dim", None)

DOWNSIDE_TRAINING_CONFIG = {
    "max_epochs": 50,
    "learning_rate": 5e-4,
    "weight_decay": 5e-4,
    "early_stopping_patience": 15,
    "minimum_auc_improvement": 1e-4,
    "gradient_clip_norm": 5.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 2,
    "minimum_learning_rate": 1e-5,
}

DOWNSIDE_RESNET_PATH = MODEL_DIR / "tabular_resnet_downside_purged_v1.pt"
DOWNSIDE_RESNET_HISTORY_PATH = REPORT_DIR / "tabular_resnet_downside_training_history_v1.csv"

y_downside_train_array = np.asarray(y_train_downside, dtype=np.float32)
negative_count = np.sum(y_downside_train_array == 0)
positive_count = np.sum(y_downside_train_array == 1)
downside_pos_weight = negative_count / positive_count

print("Training positive rate:", round(float(y_downside_train_array.mean()), 6))
print("Positive-class weight:", round(float(downside_pos_weight), 4))


# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.cuda.empty_cache()


# Data loaders
downside_resnet_train_loader = create_data_loader(downside_train_dataset, TRAIN_BATCH_SIZE, True, RANDOM_SEED)
downside_resnet_train_eval_loader = create_data_loader(downside_train_dataset, EVAL_BATCH_SIZE, False, RANDOM_SEED)
downside_resnet_valid_loader = create_data_loader(downside_valid_dataset, EVAL_BATCH_SIZE, False, RANDOM_SEED)


# Model and optimizer
downside_resnet = TabularResNet(**DOWNSIDE_RESNET_CONFIG).to(DEVICE)
downside_loss_function = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(downside_pos_weight, dtype=torch.float32, device=DEVICE)
)

downside_optimizer = torch.optim.AdamW(
    downside_resnet.parameters(),
    lr=DOWNSIDE_TRAINING_CONFIG["learning_rate"],
    weight_decay=DOWNSIDE_TRAINING_CONFIG["weight_decay"],
)

downside_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    downside_optimizer,
    mode="max",
    factor=DOWNSIDE_TRAINING_CONFIG["scheduler_factor"],
    patience=DOWNSIDE_TRAINING_CONFIG["scheduler_patience"],
    threshold=DOWNSIDE_TRAINING_CONFIG["minimum_auc_improvement"],
    min_lr=DOWNSIDE_TRAINING_CONFIG["minimum_learning_rate"],
)


# Training
downside_history = []
downside_best_epoch = 0
downside_best_valid_auc = -np.inf
downside_best_state = None
downside_epochs_without_improvement = 0
downside_training_start = time.perf_counter()

epoch_progress = tqdm(
    range(1, DOWNSIDE_TRAINING_CONFIG["max_epochs"] + 1),
    desc="Downside ResNet",
    unit="epoch",
    dynamic_ncols=True,
)

for epoch in epoch_progress:
    epoch_start = time.perf_counter()
    current_lr = downside_optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch_with_progress(
        downside_resnet,
        downside_resnet_train_loader,
        downside_optimizer,
        downside_loss_function,
        DEVICE,
        epoch,
        DOWNSIDE_TRAINING_CONFIG["gradient_clip_norm"],
    )

    valid_metrics, _, _ = evaluate_binary_model(
        downside_resnet,
        downside_resnet_valid_loader,
        downside_loss_function,
        DEVICE,
        0.50,
    )

    valid_auc = valid_metrics["roc_auc"]
    downside_scheduler.step(valid_auc)
    next_lr = downside_optimizer.param_groups[0]["lr"]
    epoch_seconds = time.perf_counter() - epoch_start

    downside_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_metrics["loss"],
        "valid_roc_auc": valid_auc,
        "valid_average_precision": valid_metrics["average_precision"],
        "valid_log_loss": valid_metrics["log_loss"],
        "valid_brier_score": valid_metrics["brier_score"],
        "valid_balanced_accuracy": valid_metrics["balanced_accuracy"],
        "valid_precision": valid_metrics["precision"],
        "valid_recall": valid_metrics["recall"],
        "valid_f1": valid_metrics["f1"],
        "learning_rate": current_lr,
        "next_learning_rate": next_lr,
        "epoch_seconds": epoch_seconds,
    })

    improved = valid_auc > downside_best_valid_auc + DOWNSIDE_TRAINING_CONFIG["minimum_auc_improvement"]

    if improved:
        downside_best_epoch = epoch
        downside_best_valid_auc = valid_auc
        downside_best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in downside_resnet.state_dict().items()
        }
        downside_epochs_without_improvement = 0
        tqdm.write(
            f"New best downside ResNet epoch {epoch:02d}: "
            f"AUC={valid_auc:.6f}, AP={valid_metrics['average_precision']:.6f}"
        )
    else:
        downside_epochs_without_improvement += 1

    epoch_progress.set_postfix({
        "train": f"{train_loss:.5f}",
        "valid": f"{valid_metrics['loss']:.5f}",
        "auc": f"{valid_auc:.5f}",
        "best": f"{downside_best_valid_auc:.5f}",
        "AP": f"{valid_metrics['average_precision']:.5f}",
        "patience": f"{downside_epochs_without_improvement}/{DOWNSIDE_TRAINING_CONFIG['early_stopping_patience']}",
        "lr": f"{next_lr:.1e}",
        "sec": f"{epoch_seconds:.1f}",
    })

    if downside_epochs_without_improvement >= DOWNSIDE_TRAINING_CONFIG["early_stopping_patience"]:
        tqdm.write("Downside ResNet early stopping triggered.")
        break


# Restore best checkpoint
downside_training_seconds = time.perf_counter() - downside_training_start

if downside_best_state is None:
    raise RuntimeError("Downside ResNet did not produce a valid checkpoint.")

downside_resnet.load_state_dict(downside_best_state)
downside_resnet.to(DEVICE)
downside_resnet.eval()


# Train and validation evaluation
downside_resnet_train_metrics, _, downside_resnet_train_probability = evaluate_binary_model(
    downside_resnet, downside_resnet_train_eval_loader, downside_loss_function, DEVICE, 0.50
)

downside_resnet_valid_metrics, downside_resnet_valid_targets, downside_resnet_valid_probability = evaluate_binary_model(
    downside_resnet, downside_resnet_valid_loader, downside_loss_function, DEVICE, 0.50
)

if not np.isclose(downside_resnet_valid_metrics["roc_auc"], downside_best_valid_auc, atol=1e-10):
    raise ValueError("Restored downside ResNet does not match the best validation checkpoint.")


# Save
downside_history_df = pd.DataFrame(downside_history)
downside_history_df.to_csv(DOWNSIDE_RESNET_HISTORY_PATH, index=False)

torch.save({
    "model_name": "tabular_resnet_downside",
    "model_version": "v1",
    "target": DOWNSIDE_TARGET,
    "split_method": PRIMARY_SPLIT_METHOD,
    "random_seed": RANDOM_SEED,
    "model_config": DOWNSIDE_RESNET_CONFIG,
    "training_config": DOWNSIDE_TRAINING_CONFIG,
    "positive_class_weight": float(downside_pos_weight),
    "feature_cols": feature_cols,
    "best_epoch": downside_best_epoch,
    "best_valid_metrics": downside_resnet_valid_metrics,
    "training_duration_seconds": downside_training_seconds,
    "state_dict": downside_best_state,
}, DOWNSIDE_RESNET_PATH)

downside_resnet_summary = pd.DataFrame([
    {"model": "tabular_resnet_downside", "split": "train", **downside_resnet_train_metrics},
    {"model": "tabular_resnet_downside", "split": "valid", **downside_resnet_valid_metrics},
])

print("\nDownside ResNet training completed.")
print("Best epoch:", downside_best_epoch)
print("Best validation ROC-AUC:", round(downside_best_valid_auc, 6))
print("Training time:", round(downside_training_seconds, 2), "seconds")
print("Checkpoint:", DOWNSIDE_RESNET_PATH)
print("History:", DOWNSIDE_RESNET_HISTORY_PATH)
print("\nThe test set remains untouched.")

display(downside_resnet_summary[[
    "model", "split", "rows", "roc_auc", "average_precision", "log_loss",
    "brier_score", "balanced_accuracy", "precision", "recall", "f1",
    "average_probability",
]])

Training positive rate: 0.148516
Positive-class weight: 5.7333


Downside ResNet:   0%|          | 0/50 [00:00<?, ?epoch/s]

Epoch 01 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

New best downside ResNet epoch 01: AUC=0.589107, AP=0.136206


Epoch 02 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

...

Epoch 14 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 15 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 16 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Downside ResNet early stopping triggered.

Downside ResNet training completed.
Best epoch: 1
Best validation ROC-AUC: 0.589107
Training time: 66.59 seconds
Checkpoint: E:\Projects\marketguard-india\models\deep_learning\tabular_resnet_downside_purged_v1.pt
History: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\tabular_resnet_downside_training_history_v1.csv

The test set remains untouched.


,model,split,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,tabular_resnet_downside,train,251738,0.726946,0.326213,0.636285,0.223900,0.664716,0.243878,0.717362,0.364007,0.459308
1,tabular_resnet_downside,valid,19980,0.589107,0.136206,0.763536,0.280617,0.569277,0.128845,0.667462,0.215995,0.509023


## Downside ResNet Result

The downside ResNet reached its best validation result at epoch 1.

| Model | ROC-AUC | Average Precision | Precision | Recall |
|---|---:|---:|---:|---:|
| Random Forest | **0.6593** | **0.2020** | **0.1916** | 0.4981 |
| Tabular ResNet | 0.5891 | 0.1362 | 0.1288 | **0.6675** |

The ResNet detected more downside events, but its precision was much lower.
This means it produced substantially more false downside alerts.

The model also overfit quickly. Training ROC-AUC reached `0.7269`, while
validation ROC-AUC was only `0.5891`.

### Decision

The Tabular ResNet will not replace the downside Random Forest.

A validation-only ensemble test will be performed to check whether the ResNet
contains any complementary information.

The test set remains untouched.

In [32]:
downside_targets = np.asarray(y_valid_downside, dtype=np.int64)
downside_rf_probability = np.asarray(downside_rf_valid_probability, dtype=np.float64)
downside_resnet_probability = np.asarray(downside_resnet_valid_probability, dtype=np.float64)

if not (len(downside_targets) == len(downside_rf_probability) == len(downside_resnet_probability)):
    raise ValueError("Downside validation arrays are not aligned.")

rows = []

for resnet_weight in np.arange(0.0, 1.01, 0.05):
    rf_weight = 1.0 - resnet_weight
    probability = resnet_weight * downside_resnet_probability + rf_weight * downside_rf_probability
    metrics = calculate_binary_metrics(downside_targets, probability, 0.50)
    rows.append({"resnet_weight": resnet_weight, "random_forest_weight": rf_weight, **metrics})

downside_ensemble_results = pd.DataFrame(rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

best_downside_ensemble = downside_ensemble_results.iloc[0]

DOWNSIDE_ENSEMBLE_PATH = REPORT_DIR / "downside_resnet_rf_ensemble_validation_v1.csv"
downside_ensemble_results.to_csv(DOWNSIDE_ENSEMBLE_PATH, index=False)

print("ResNet-RF probability correlation:", round(np.corrcoef(downside_resnet_probability, downside_rf_probability)[0, 1], 6))
print("Best ResNet weight:", round(float(best_downside_ensemble["resnet_weight"]), 2))
print("Best Random Forest weight:", round(float(best_downside_ensemble["random_forest_weight"]), 2))
print("\nThe test set remains untouched.")

display(downside_ensemble_results[[
    "resnet_weight", "random_forest_weight", "roc_auc", "average_precision",
    "log_loss", "brier_score", "precision", "recall", "f1",
]].head(10))

ResNet-RF probability correlation: 0.738657
Best ResNet weight: 0.0
Best Random Forest weight: 1.0

The test set remains untouched.


,resnet_weight,random_forest_weight,roc_auc,average_precision,log_loss,brier_score,precision,recall,f1
0,0.00,1.00,0.659263,0.202046,0.594743,0.202433,0.191630,0.498092,0.276776
1,0.05,0.95,0.657123,0.201901,0.600581,0.205296,0.185907,0.518607,0.273700
2,0.10,0.90,0.654607,0.200964,0.606620,0.208269,0.177407,0.531966,0.266078
3,0.15,0.85,0.651706,0.199137,0.612869,0.211352,0.171509,0.549141,0.261383
4,0.20,0.80,0.648452,0.196701,0.619335,0.214545,0.167325,0.565840,0.258275
5,0.25,0.75,0.644859,0.193594,0.626025,0.217848,0.162046,0.577290,0.253059
6,0.30,0.70,0.640961,0.189792,0.632951,0.221262,0.159219,0.594943,0.251209
7,0.35,0.65,0.636897,0.185774,0.640121,0.224785,0.155198,0.606870,0.247182
8,0.40,0.60,0.632717,0.181568,0.647548,0.228419,0.151323,0.614027,0.242807
9,0.45,0.55,0.628513,0.177236,0.655244,0.232163,0.148216,0.622137,0.239398


## Downside Ensemble Result

The downside ResNet and Random Forest probabilities had a correlation of
approximately `0.74`.

The validation ensemble search selected:

```text
0% Tabular ResNet
100% Random Forest
```

Adding even 5% ResNet reduced ROC-AUC and worsened probability quality.

Although the ResNet increased recall, it also reduced precision and produced
more false downside alerts.

### Decision

The ResNet does not add useful complementary information to the downside Random
Forest.

No bootstrap analysis is required because the best ensemble is simply the
Random Forest alone.

The downside Random Forest is now the locked final candidate.

The next step is one-time evaluation on the untouched test period.

In [33]:
# One-time downside test evaluation
DOWNSIDE_FINAL_TEST_PATH = REPORT_DIR / "downside_final_test_comparison_v1.csv"

downside_test_targets = np.asarray(y_test_downside, dtype=np.int64)
downside_test_raw = model_data.loc[test_mask, feature_cols]

downside_rf_test_probability = purged_rf_downside.predict_proba(downside_test_raw)[:, 1]

downside_test_loader = create_data_loader(downside_test_dataset, EVAL_BATCH_SIZE, False, RANDOM_SEED)

_, downside_resnet_test_targets, downside_resnet_test_probability = evaluate_binary_model(
    downside_resnet,
    downside_test_loader,
    downside_loss_function,
    DEVICE,
    0.50,
)

downside_resnet_test_targets = np.asarray(downside_resnet_test_targets, dtype=np.int64)
downside_resnet_test_probability = np.asarray(downside_resnet_test_probability, dtype=np.float64)

if not np.array_equal(downside_test_targets, downside_resnet_test_targets):
    raise ValueError("Downside test targets are not aligned.")

test_probabilities = {
    "random_forest_downside": downside_rf_test_probability,
    "tabular_resnet_downside": downside_resnet_test_probability,
}

test_rows = []

for model_name, probability in test_probabilities.items():
    metrics = calculate_binary_metrics(downside_test_targets, probability, 0.50)
    test_rows.append({"model": model_name, **metrics})

downside_final_test_comparison = pd.DataFrame(test_rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

downside_final_test_comparison.to_csv(DOWNSIDE_FINAL_TEST_PATH, index=False)

print("Final downside test evaluation completed.")
print("Locked candidate: Random Forest")
print("Test period:", model_data.loc[test_mask, "date"].min().date(), "to", model_data.loc[test_mask, "date"].max().date())
print("Test rows:", len(downside_test_targets))
print("Actual downside-event rate:", round(float(downside_test_targets.mean()), 6))
print("Test report:", DOWNSIDE_FINAL_TEST_PATH)
print("\nNo model changes will be made using these results.")

display(downside_final_test_comparison[[
    "model", "rows", "roc_auc", "average_precision", "log_loss", "brier_score",
    "accuracy", "balanced_accuracy", "precision", "recall", "f1", "average_probability",
]])

Final downside test evaluation completed.
Locked candidate: Random Forest
Test period: 2025-01-01 to 2026-06-16
Test rows: 32135
Actual downside-event rate: 0.111965
Test report: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\downside_final_test_comparison_v1.csv

No model changes will be made using these results.


,model,rows,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision,recall,f1,average_probability
0,random_forest_downside,32135,0.684487,0.205591,0.545953,0.184022,0.706084,0.624656,0.195056,0.519733,0.283656,0.393105
1,tabular_resnet_downside,32135,0.589823,0.137422,0.674754,0.240626,0.599720,0.561854,0.142471,0.513063,0.223014,0.434681


# Downside Weighted MLP Experiment

## Purpose

This experiment evaluates a standard tabular MLP for predicting:

`target_big_downside_10pct_20d`

The target identifies whether a stock experiences a decline of at least 10%
during the following 20 trading days.

The downside target is imbalanced, so the MLP will use a positive-class weight
to give greater importance to rare downside events.

## Model Structure

```text
79 input features
        ↓
256 neurons
        ↓
128 neurons
        ↓
64 neurons
        ↓
Downside-risk logit
```

The model will use:

- Batch normalization
- ReLU activations
- Dropout
- Weighted binary cross-entropy
- AdamW optimization
- Gradient clipping
- Early stopping using validation ROC-AUC

## Evaluation

The weighted MLP will be compared with:

1. Downside Random Forest
2. Downside Tabular ResNet

The main metrics are:

- ROC-AUC
- Average precision
- Precision
- Recall
- Log loss
- Brier score

Model selection will continue to use the purged validation period.

The previously evaluated test period will not be used for model tuning.

In [34]:
# ---------------------------------------------------------
# Weighted downside MLP
# ---------------------------------------------------------

DOWNSIDE_MLP_CONFIG = {
    "input_dim": len(feature_cols),
    "hidden_dims": [256, 128, 64],
    "dropout_rates": [0.20, 0.15, 0.10],
}

DOWNSIDE_MLP_TRAINING_CONFIG = {
    "max_epochs": 50,
    "learning_rate": 5e-4,
    "weight_decay": 5e-4,
    "early_stopping_patience": 15,
    "minimum_auc_improvement": 1e-4,
    "gradient_clip_norm": 5.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 2,
    "minimum_learning_rate": 1e-5,
}

DOWNSIDE_MLP_PATH = MODEL_DIR / "tabular_mlp_downside_purged_v1.pt"
DOWNSIDE_MLP_HISTORY_PATH = REPORT_DIR / "tabular_mlp_downside_training_history_v1.csv"


class DownsideMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rates):
        super().__init__()

        layers = []
        previous_dim = input_dim

        for hidden_dim, dropout_rate in zip(hidden_dims, dropout_rates):
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
            ])
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, features):
        return self.network(features).squeeze(-1)


# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.cuda.empty_cache()


# Data loaders
downside_mlp_train_loader = create_data_loader(downside_train_dataset, TRAIN_BATCH_SIZE, True, RANDOM_SEED)
downside_mlp_train_eval_loader = create_data_loader(downside_train_dataset, EVAL_BATCH_SIZE, False, RANDOM_SEED)
downside_mlp_valid_loader = create_data_loader(downside_valid_dataset, EVAL_BATCH_SIZE, False, RANDOM_SEED)


# Model, loss and optimizer
downside_mlp = DownsideMLP(**DOWNSIDE_MLP_CONFIG).to(DEVICE)

downside_mlp_loss = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(downside_pos_weight, dtype=torch.float32, device=DEVICE)
)

downside_mlp_optimizer = torch.optim.AdamW(
    downside_mlp.parameters(),
    lr=DOWNSIDE_MLP_TRAINING_CONFIG["learning_rate"],
    weight_decay=DOWNSIDE_MLP_TRAINING_CONFIG["weight_decay"],
)

downside_mlp_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    downside_mlp_optimizer,
    mode="max",
    factor=DOWNSIDE_MLP_TRAINING_CONFIG["scheduler_factor"],
    patience=DOWNSIDE_MLP_TRAINING_CONFIG["scheduler_patience"],
    threshold=DOWNSIDE_MLP_TRAINING_CONFIG["minimum_auc_improvement"],
    min_lr=DOWNSIDE_MLP_TRAINING_CONFIG["minimum_learning_rate"],
)


# Training
downside_mlp_history = []
downside_mlp_best_epoch = 0
downside_mlp_best_auc = -np.inf
downside_mlp_best_state = None
downside_mlp_epochs_without_improvement = 0
downside_mlp_start = time.perf_counter()

epoch_progress = tqdm(
    range(1, DOWNSIDE_MLP_TRAINING_CONFIG["max_epochs"] + 1),
    desc="Downside weighted MLP",
    unit="epoch",
    dynamic_ncols=True,
)

for epoch in epoch_progress:
    epoch_start = time.perf_counter()
    current_lr = downside_mlp_optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch_with_progress(
        downside_mlp,
        downside_mlp_train_loader,
        downside_mlp_optimizer,
        downside_mlp_loss,
        DEVICE,
        epoch,
        DOWNSIDE_MLP_TRAINING_CONFIG["gradient_clip_norm"],
    )

    valid_metrics, _, _ = evaluate_binary_model(
        downside_mlp,
        downside_mlp_valid_loader,
        downside_mlp_loss,
        DEVICE,
        0.50,
    )

    valid_auc = valid_metrics["roc_auc"]
    downside_mlp_scheduler.step(valid_auc)
    next_lr = downside_mlp_optimizer.param_groups[0]["lr"]
    epoch_seconds = time.perf_counter() - epoch_start

    downside_mlp_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_metrics["loss"],
        "valid_roc_auc": valid_auc,
        "valid_average_precision": valid_metrics["average_precision"],
        "valid_log_loss": valid_metrics["log_loss"],
        "valid_brier_score": valid_metrics["brier_score"],
        "valid_precision": valid_metrics["precision"],
        "valid_recall": valid_metrics["recall"],
        "valid_f1": valid_metrics["f1"],
        "learning_rate": current_lr,
        "next_learning_rate": next_lr,
        "epoch_seconds": epoch_seconds,
    })

    improved = valid_auc > downside_mlp_best_auc + DOWNSIDE_MLP_TRAINING_CONFIG["minimum_auc_improvement"]

    if improved:
        downside_mlp_best_epoch = epoch
        downside_mlp_best_auc = valid_auc
        downside_mlp_best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in downside_mlp.state_dict().items()
        }
        downside_mlp_epochs_without_improvement = 0
        tqdm.write(
            f"New best downside MLP epoch {epoch:02d}: "
            f"AUC={valid_auc:.6f}, AP={valid_metrics['average_precision']:.6f}"
        )
    else:
        downside_mlp_epochs_without_improvement += 1

    epoch_progress.set_postfix({
        "train": f"{train_loss:.5f}",
        "valid": f"{valid_metrics['loss']:.5f}",
        "auc": f"{valid_auc:.5f}",
        "best": f"{downside_mlp_best_auc:.5f}",
        "AP": f"{valid_metrics['average_precision']:.5f}",
        "patience": f"{downside_mlp_epochs_without_improvement}/{DOWNSIDE_MLP_TRAINING_CONFIG['early_stopping_patience']}",
        "lr": f"{next_lr:.1e}",
        "sec": f"{epoch_seconds:.1f}",
    })

    if downside_mlp_epochs_without_improvement >= DOWNSIDE_MLP_TRAINING_CONFIG["early_stopping_patience"]:
        tqdm.write("Downside MLP early stopping triggered.")
        break


# Restore best checkpoint
downside_mlp_training_seconds = time.perf_counter() - downside_mlp_start

if downside_mlp_best_state is None:
    raise RuntimeError("Downside MLP did not produce a valid checkpoint.")

downside_mlp.load_state_dict(downside_mlp_best_state)
downside_mlp.to(DEVICE)
downside_mlp.eval()


# Evaluate train and validation
downside_mlp_train_metrics, _, downside_mlp_train_probability = evaluate_binary_model(
    downside_mlp, downside_mlp_train_eval_loader, downside_mlp_loss, DEVICE, 0.50
)

downside_mlp_valid_metrics, downside_mlp_valid_targets, downside_mlp_valid_probability = evaluate_binary_model(
    downside_mlp, downside_mlp_valid_loader, downside_mlp_loss, DEVICE, 0.50
)

if not np.isclose(downside_mlp_valid_metrics["roc_auc"], downside_mlp_best_auc, atol=1e-10):
    raise ValueError("Restored downside MLP does not match the best validation checkpoint.")


# Save model and history
pd.DataFrame(downside_mlp_history).to_csv(DOWNSIDE_MLP_HISTORY_PATH, index=False)

torch.save({
    "model_name": "tabular_mlp_downside",
    "model_version": "v1",
    "target": DOWNSIDE_TARGET,
    "split_method": PRIMARY_SPLIT_METHOD,
    "random_seed": RANDOM_SEED,
    "model_config": DOWNSIDE_MLP_CONFIG,
    "training_config": DOWNSIDE_MLP_TRAINING_CONFIG,
    "positive_class_weight": float(downside_pos_weight),
    "feature_cols": feature_cols,
    "best_epoch": downside_mlp_best_epoch,
    "best_valid_metrics": downside_mlp_valid_metrics,
    "training_duration_seconds": downside_mlp_training_seconds,
    "state_dict": downside_mlp_best_state,
}, DOWNSIDE_MLP_PATH)

downside_mlp_summary = pd.DataFrame([
    {"model": "weighted_mlp_downside", "split": "train", **downside_mlp_train_metrics},
    {"model": "weighted_mlp_downside", "split": "valid", **downside_mlp_valid_metrics},
])

print("\nDownside MLP training completed.")
print("Best epoch:", downside_mlp_best_epoch)
print("Best validation ROC-AUC:", round(downside_mlp_best_auc, 6))
print("Training time:", round(downside_mlp_training_seconds, 2), "seconds")
print("Checkpoint:", DOWNSIDE_MLP_PATH)
print("History:", DOWNSIDE_MLP_HISTORY_PATH)
print("\nThe previously viewed test period was not used for this training.")

display(downside_mlp_summary[[
    "model", "split", "rows", "roc_auc", "average_precision", "log_loss",
    "brier_score", "balanced_accuracy", "precision", "recall", "f1",
    "average_probability",
]])

Downside weighted MLP:   0%|          | 0/50 [00:00<?, ?epoch/s]

Epoch 01 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

New best downside MLP epoch 01: AUC=0.589598, AP=0.149716


Epoch 02 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

New best downside MLP epoch 02: AUC=0.596831, AP=0.156669


...

Epoch 18 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 19 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 20 batches:   0%|          | 0/62 [00:00<?, ?batch/s]

Downside MLP early stopping triggered.

Downside MLP training completed.
Best epoch: 5
Best validation ROC-AUC: 0.617987
Training time: 73.27 seconds
Checkpoint: E:\Projects\marketguard-india\models\deep_learning\tabular_mlp_downside_purged_v1.pt
History: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\tabular_mlp_downside_training_history_v1.csv

The previously viewed test period was not used for this training.


,model,split,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,weighted_mlp_downside,train,251738,0.780518,0.403568,0.592658,0.204929,0.707589,0.275788,0.766042,0.405565,0.443875
1,weighted_mlp_downside,valid,19980,0.617987,0.194551,0.709208,0.257413,0.581996,0.135373,0.652195,0.224209,0.489839


## Downside Weighted MLP Result

The weighted MLP reached its best validation result at epoch 5.

| Model | ROC-AUC | Average Precision | Precision | Recall |
|---|---:|---:|---:|---:|
| Random Forest | **0.6593** | **0.2020** | **0.1916** | 0.4981 |
| Weighted MLP | 0.6180 | 0.1946 | 0.1354 | **0.6522** |
| Tabular ResNet | 0.5891 | 0.1362 | 0.1288 | 0.6675 |

The MLP achieved higher recall than the Random Forest, meaning it detected more
downside events.

However, its lower precision shows that it also generated more false downside
warnings.

The model overfit, with training ROC-AUC of `0.7805` and validation ROC-AUC of
`0.6180`.

Its average predicted probability was much higher than the actual downside-event
rate because of the weighted loss.

### Decision

The weighted MLP will not replace the downside Random Forest.

It will be retained as a secondary research candidate for a later ensemble
comparison.

The next downside model will be the FT-Transformer.

In [35]:
# ---------------------------------------------------------
# Weighted downside FT-Transformer
# ---------------------------------------------------------

DOWNSIDE_FT_TRAINING_CONFIG = {
    "max_epochs": 40,
    "learning_rate": 2e-4,
    "weight_decay": 1e-5,
    "early_stopping_patience": 10,
    "minimum_auc_improvement": 1e-4,
    "gradient_clip_norm": 1.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 2,
    "minimum_learning_rate": 1e-6,
}

DOWNSIDE_FT_BATCH_SIZE = 1024
DOWNSIDE_FT_EVAL_BATCH_SIZE = 2048

DOWNSIDE_FT_PATH = MODEL_DIR / "ft_transformer_downside_purged_v1.pt"
DOWNSIDE_FT_HISTORY_PATH = REPORT_DIR / "ft_transformer_downside_training_history_v1.csv"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.cuda.empty_cache()

downside_ft_train_loader = create_data_loader(downside_train_dataset, DOWNSIDE_FT_BATCH_SIZE, True, RANDOM_SEED)
downside_ft_train_eval_loader = create_data_loader(downside_train_dataset, DOWNSIDE_FT_EVAL_BATCH_SIZE, False, RANDOM_SEED)
downside_ft_valid_loader = create_data_loader(downside_valid_dataset, DOWNSIDE_FT_EVAL_BATCH_SIZE, False, RANDOM_SEED)

downside_ft = FTTransformer(**FT_TRANSFORMER_CONFIG).to(DEVICE)

downside_ft_loss = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(downside_pos_weight, dtype=torch.float32, device=DEVICE)
)

downside_ft_optimizer = torch.optim.AdamW(
    downside_ft.parameters(),
    lr=DOWNSIDE_FT_TRAINING_CONFIG["learning_rate"],
    weight_decay=DOWNSIDE_FT_TRAINING_CONFIG["weight_decay"],
)

downside_ft_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    downside_ft_optimizer,
    mode="max",
    factor=DOWNSIDE_FT_TRAINING_CONFIG["scheduler_factor"],
    patience=DOWNSIDE_FT_TRAINING_CONFIG["scheduler_patience"],
    threshold=DOWNSIDE_FT_TRAINING_CONFIG["minimum_auc_improvement"],
    min_lr=DOWNSIDE_FT_TRAINING_CONFIG["minimum_learning_rate"],
)

downside_ft_history = []
downside_ft_best_epoch = 0
downside_ft_best_auc = -np.inf
downside_ft_best_state = None
downside_ft_epochs_without_improvement = 0
downside_ft_start = time.perf_counter()

epoch_progress = tqdm(
    range(1, DOWNSIDE_FT_TRAINING_CONFIG["max_epochs"] + 1),
    desc="Downside FT-Transformer",
    unit="epoch",
    dynamic_ncols=True,
)

for epoch in epoch_progress:
    epoch_start = time.perf_counter()
    current_lr = downside_ft_optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch_with_progress(
        downside_ft,
        downside_ft_train_loader,
        downside_ft_optimizer,
        downside_ft_loss,
        DEVICE,
        epoch,
        DOWNSIDE_FT_TRAINING_CONFIG["gradient_clip_norm"],
    )

    valid_metrics, _, _ = evaluate_binary_model(
        downside_ft,
        downside_ft_valid_loader,
        downside_ft_loss,
        DEVICE,
        0.50,
    )

    valid_auc = valid_metrics["roc_auc"]
    downside_ft_scheduler.step(valid_auc)
    next_lr = downside_ft_optimizer.param_groups[0]["lr"]
    epoch_seconds = time.perf_counter() - epoch_start

    downside_ft_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_metrics["loss"],
        "valid_roc_auc": valid_auc,
        "valid_average_precision": valid_metrics["average_precision"],
        "valid_log_loss": valid_metrics["log_loss"],
        "valid_brier_score": valid_metrics["brier_score"],
        "valid_precision": valid_metrics["precision"],
        "valid_recall": valid_metrics["recall"],
        "valid_f1": valid_metrics["f1"],
        "learning_rate": current_lr,
        "next_learning_rate": next_lr,
        "epoch_seconds": epoch_seconds,
    })

    improved = valid_auc > downside_ft_best_auc + DOWNSIDE_FT_TRAINING_CONFIG["minimum_auc_improvement"]

    if improved:
        downside_ft_best_epoch = epoch
        downside_ft_best_auc = valid_auc
        downside_ft_best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in downside_ft.state_dict().items()
        }
        downside_ft_epochs_without_improvement = 0
        tqdm.write(
            f"New best downside FT epoch {epoch:02d}: "
            f"AUC={valid_auc:.6f}, AP={valid_metrics['average_precision']:.6f}"
        )
    else:
        downside_ft_epochs_without_improvement += 1

    epoch_progress.set_postfix({
        "train": f"{train_loss:.5f}",
        "valid": f"{valid_metrics['loss']:.5f}",
        "auc": f"{valid_auc:.5f}",
        "best": f"{downside_ft_best_auc:.5f}",
        "AP": f"{valid_metrics['average_precision']:.5f}",
        "patience": f"{downside_ft_epochs_without_improvement}/{DOWNSIDE_FT_TRAINING_CONFIG['early_stopping_patience']}",
        "lr": f"{next_lr:.1e}",
        "sec": f"{epoch_seconds:.1f}",
    })

    if downside_ft_epochs_without_improvement >= DOWNSIDE_FT_TRAINING_CONFIG["early_stopping_patience"]:
        tqdm.write("Downside FT-Transformer early stopping triggered.")
        break

downside_ft_training_seconds = time.perf_counter() - downside_ft_start

if downside_ft_best_state is None:
    raise RuntimeError("Downside FT-Transformer did not produce a valid checkpoint.")

downside_ft.load_state_dict(downside_ft_best_state)
downside_ft.to(DEVICE)
downside_ft.eval()

downside_ft_train_metrics, _, downside_ft_train_probability = evaluate_binary_model(
    downside_ft, downside_ft_train_eval_loader, downside_ft_loss, DEVICE, 0.50
)

downside_ft_valid_metrics, downside_ft_valid_targets, downside_ft_valid_probability = evaluate_binary_model(
    downside_ft, downside_ft_valid_loader, downside_ft_loss, DEVICE, 0.50
)

if not np.isclose(downside_ft_valid_metrics["roc_auc"], downside_ft_best_auc, atol=1e-10):
    raise ValueError("Restored downside FT checkpoint does not match the best validation AUC.")

pd.DataFrame(downside_ft_history).to_csv(DOWNSIDE_FT_HISTORY_PATH, index=False)

torch.save({
    "model_name": "ft_transformer_downside",
    "model_version": "v1",
    "target": DOWNSIDE_TARGET,
    "split_method": PRIMARY_SPLIT_METHOD,
    "random_seed": RANDOM_SEED,
    "model_config": FT_TRANSFORMER_CONFIG,
    "training_config": DOWNSIDE_FT_TRAINING_CONFIG,
    "positive_class_weight": float(downside_pos_weight),
    "feature_cols": feature_cols,
    "best_epoch": downside_ft_best_epoch,
    "best_valid_metrics": downside_ft_valid_metrics,
    "training_duration_seconds": downside_ft_training_seconds,
    "state_dict": downside_ft_best_state,
}, DOWNSIDE_FT_PATH)

downside_ft_summary = pd.DataFrame([
    {"model": "ft_transformer_downside", "split": "train", **downside_ft_train_metrics},
    {"model": "ft_transformer_downside", "split": "valid", **downside_ft_valid_metrics},
])

print("\nDownside FT-Transformer training completed.")
print("Best epoch:", downside_ft_best_epoch)
print("Best validation ROC-AUC:", round(downside_ft_best_auc, 6))
print("Training time:", round(downside_ft_training_seconds, 2), "seconds")
print("Checkpoint:", DOWNSIDE_FT_PATH)
print("History:", DOWNSIDE_FT_HISTORY_PATH)
print("\nThe previously viewed test period was not used for training or selection.")

display(downside_ft_summary[[
    "model", "split", "rows", "roc_auc", "average_precision", "log_loss",
    "brier_score", "balanced_accuracy", "precision", "recall", "f1",
    "average_probability",
]])

Downside FT-Transformer:   0%|          | 0/40 [00:00<?, ?epoch/s]

Epoch 01 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

New best downside FT epoch 01: AUC=0.623550, AP=0.185838


Epoch 02 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Epoch 03 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

...

Epoch 15 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Epoch 16 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Epoch 17 batches:   0%|          | 0/246 [00:00<?, ?batch/s]

Downside FT-Transformer early stopping triggered.

Downside FT-Transformer training completed.
Best epoch: 7
Best validation ROC-AUC: 0.62366
Training time: 604.41 seconds
Checkpoint: E:\Projects\marketguard-india\models\deep_learning\ft_transformer_downside_purged_v1.pt
History: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\ft_transformer_downside_training_history_v1.csv

The previously viewed test period was not used for training or selection.


,model,split,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,ft_transformer_downside,train,251738,0.853754,0.545108,0.509075,0.167481,0.773673,0.359463,0.794180,0.494916,0.376388
1,ft_transformer_downside,valid,19980,0.623660,0.170670,0.825156,0.289719,0.583704,0.138597,0.616412,0.226309,0.478920


## Downside FT-Transformer Result

The weighted FT-Transformer reached its best validation ROC-AUC at epoch 7.

| Split | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Train | 0.8538 | 0.5451 | 0.5091 | 0.1675 |
| Validation | 0.6237 | 0.1707 | 0.8252 | 0.2897 |

The large difference between training and validation performance shows strong
overfitting.

The FT-Transformer achieved slightly higher ROC-AUC than the weighted MLP, but
its average precision, log loss, and Brier score were worse.

It remained clearly behind the Random Forest.

### Decision

The FT-Transformer will not replace the downside Random Forest.

It will not be included in an ensemble because its probability quality is poor
and it does not provide a validation improvement over the Random Forest.

The next downside model will be TabNet.

# Downside TabNet Experiment

## Purpose

This experiment evaluates TabNet for predicting:

`target_big_downside_10pct_20d`

TabNet uses sequential attention to select the most useful features during each
decision step.

```text
79 input features
        ↓
Feature selection
        ↓
Multiple decision steps
        ↓
Downside-risk score
```

## Class Imbalance

Only about `14.85%` of training rows contain a future 10% downside event.

TabNet will therefore use balanced class weights so downside events receive
greater importance during training.

## Evaluation

TabNet will be compared with:

| Model | Validation ROC-AUC | Average Precision |
|---|---:|---:|
| Random Forest | **0.6593** | **0.2020** |
| FT-Transformer | 0.6237 | 0.1707 |
| Weighted MLP | 0.6180 | 0.1946 |
| Tabular ResNet | 0.5891 | 0.1362 |

The main metrics are:

- ROC-AUC
- Average precision
- Precision
- Recall
- Log loss
- Brier score

Model selection will use the purged validation period.

The previously evaluated test period will not be used for training or model
selection.

In [38]:
# Downside TabNet data
X_downside_tabnet_train = np.ascontiguousarray(X_train, dtype=np.float32)
X_downside_tabnet_valid = np.ascontiguousarray(X_valid, dtype=np.float32)

y_downside_tabnet_train = np.asarray(y_train_downside, dtype=np.int64).reshape(-1)
y_downside_tabnet_valid = np.asarray(y_valid_downside, dtype=np.int64).reshape(-1)

print("Train:", X_downside_tabnet_train.shape, y_downside_tabnet_train.shape)
print("Valid:", X_downside_tabnet_valid.shape, y_downside_tabnet_valid.shape)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

if not np.isfinite(X_downside_tabnet_train).all() or not np.isfinite(X_downside_tabnet_valid).all():
    raise ValueError("TabNet features contain missing or infinite values.")

TABNET_MAX_EPOCHS = 100
TABNET_PATIENCE = 15
TABNET_BATCH_SIZE = 4096
TABNET_VIRTUAL_BATCH_SIZE = 256

DOWNSIDE_TABNET_CONFIG = {
    "n_d": 32,
    "n_a": 32,
    "n_steps": 5,
    "gamma": 1.5,
    "n_independent": 2,
    "n_shared": 2,
    "lambda_sparse": 1e-4,
    "momentum": 0.02,
    "clip_value": 1.0,
    "mask_type": "sparsemax",
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": {"lr": 2e-2, "weight_decay": 1e-5},
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {"step_size": 10, "gamma": 0.5},
    "seed": RANDOM_SEED,
    "verbose": 0,
    "device_name": "cuda",
}

DOWNSIDE_TABNET_PATH = MODEL_DIR / "tabnet_downside_purged_v1"
DOWNSIDE_TABNET_HISTORY_PATH = REPORT_DIR / "tabnet_downside_training_history_v1.csv"

steps_per_epoch = math.ceil(len(X_downside_tabnet_train) / TABNET_BATCH_SIZE)
downside_tabnet_progress = TabNetTQDMCallback(TABNET_MAX_EPOCHS, steps_per_epoch)

downside_tabnet = TabNetClassifier(**DOWNSIDE_TABNET_CONFIG)
downside_tabnet_start = time.perf_counter()

downside_tabnet.fit(
    X_train=X_downside_tabnet_train,
    y_train=y_downside_tabnet_train,
    eval_set=[(X_downside_tabnet_valid, y_downside_tabnet_valid)],
    eval_name=["valid"],
    eval_metric=["auc"],
    max_epochs=TABNET_MAX_EPOCHS,
    patience=TABNET_PATIENCE,
    batch_size=TABNET_BATCH_SIZE,
    virtual_batch_size=TABNET_VIRTUAL_BATCH_SIZE,
    num_workers=0,
    drop_last=False,
    pin_memory=True,
    weights=1,
    callbacks=[downside_tabnet_progress],
)

downside_tabnet_seconds = time.perf_counter() - downside_tabnet_start

# Validation and training predictions
downside_tabnet_train_probability = downside_tabnet.predict_proba(X_downside_tabnet_train)[:, 1]
downside_tabnet_valid_probability = downside_tabnet.predict_proba(X_downside_tabnet_valid)[:, 1]

downside_tabnet_train_metrics = calculate_binary_metrics(
    y_downside_tabnet_train, downside_tabnet_train_probability, 0.50
)

downside_tabnet_valid_metrics = calculate_binary_metrics(
    y_downside_tabnet_valid, downside_tabnet_valid_probability, 0.50
)

# Save model and history
saved_downside_tabnet_path = downside_tabnet.save_model(str(DOWNSIDE_TABNET_PATH))
downside_tabnet_history = getattr(downside_tabnet.history, "history", downside_tabnet.history)
pd.DataFrame(downside_tabnet_history).to_csv(DOWNSIDE_TABNET_HISTORY_PATH, index=False)

downside_tabnet_summary = pd.DataFrame([
    {"model": "tabnet_downside", "split": "train", **downside_tabnet_train_metrics},
    {"model": "tabnet_downside", "split": "valid", **downside_tabnet_valid_metrics},
])

print("\nDownside TabNet training completed.")
print("Device:", downside_tabnet.device)
print("Best epoch:", downside_tabnet.best_epoch)
print("Best validation AUC:", downside_tabnet.best_cost)
print("Training time:", round(downside_tabnet_seconds, 2), "seconds")
print("Saved model:", saved_downside_tabnet_path)
print("History:", DOWNSIDE_TABNET_HISTORY_PATH)
print("\nThe previously viewed test period was not used for training or selection.")

display(downside_tabnet_summary[[
    "model", "split", "rows", "roc_auc", "average_precision", "log_loss",
    "brier_score", "balanced_accuracy", "precision", "recall", "f1",
    "average_probability",
]])

Train: (251738, 79) (251738,)
Valid: (19980, 79) (19980,)


Outperformance TabNet:   0%|          | 0/100 [00:00<?, ?epoch/s]

Epoch 001:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 002:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 003:   0%|          | 0/62 [00:00<?, ?batch/s]

...

Epoch 030:   0%|          | 0/62 [00:00<?, ?batch/s]


Early stopping occurred at epoch 29 with best_epoch = 14 and best_valid_auc = 0.63432


c:\Users\ppava\anaconda3\envs\marketguard\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Successfully saved model at E:\Projects\marketguard-india\models\deep_learning\tabnet_downside_purged_v1.zip

Downside TabNet training completed.
Device: cuda
Best epoch: 14
Best validation AUC: 0.6343173740739729
Training time: 302.84 seconds
Saved model: E:\Projects\marketguard-india\models\deep_learning\tabnet_downside_purged_v1.zip
History: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\tabnet_downside_training_history_v1.csv

The previously viewed test period was not used for training or selection.


,model,split,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,tabnet_downside,train,251738,0.744092,0.346161,0.601642,0.208582,0.674395,0.256040,0.707198,0.375963,0.442480
1,tabnet_downside,valid,19980,0.634317,0.182942,0.821576,0.306999,0.568830,0.124206,0.792939,0.214770,0.550291


In [39]:
# ---------------------------------------------------------
# Downside TabNet V2: simplified and unweighted
# ---------------------------------------------------------

DOWNSIDE_TABNET_V2_CONFIG = {
    "n_d": 24,
    "n_a": 24,
    "n_steps": 3,
    "gamma": 1.3,
    "n_independent": 2,
    "n_shared": 2,
    "lambda_sparse": 1e-3,
    "momentum": 0.02,
    "clip_value": 1.0,
    "mask_type": "sparsemax",
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": {"lr": 5e-3, "weight_decay": 1e-4},
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {"step_size": 10, "gamma": 0.5},
    "seed": RANDOM_SEED,
    "verbose": 0,
    "device_name": "cuda",
}

TABNET_V2_MAX_EPOCHS = 100
TABNET_V2_PATIENCE = 15
TABNET_V2_BATCH_SIZE = 4096
TABNET_V2_VIRTUAL_BATCH_SIZE = 256

DOWNSIDE_TABNET_V2_PATH = MODEL_DIR / "tabnet_downside_purged_v2"
DOWNSIDE_TABNET_V2_HISTORY_PATH = REPORT_DIR / "tabnet_downside_training_history_v2.csv"
DOWNSIDE_TABNET_V2_COMPARISON_PATH = REPORT_DIR / "tabnet_downside_v1_v2_validation_comparison.csv"


class NamedTabNetTQDMCallback(TabNetTQDMCallback):
    def __init__(self, max_epochs, steps_per_epoch, description):
        super().__init__(max_epochs, steps_per_epoch)
        self.description = description

    def on_train_begin(self, logs=None):
        self.epoch_bar = tqdm(total=self.max_epochs, desc=self.description, unit="epoch", dynamic_ncols=True)


if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

steps_per_epoch = math.ceil(len(X_downside_tabnet_train) / TABNET_V2_BATCH_SIZE)
tabnet_v2_progress = NamedTabNetTQDMCallback(
    TABNET_V2_MAX_EPOCHS,
    steps_per_epoch,
    "Downside TabNet V2",
)

downside_tabnet_v2 = TabNetClassifier(**DOWNSIDE_TABNET_V2_CONFIG)
tabnet_v2_start = time.perf_counter()

downside_tabnet_v2.fit(
    X_train=X_downside_tabnet_train,
    y_train=y_downside_tabnet_train,
    eval_set=[(X_downside_tabnet_valid, y_downside_tabnet_valid)],
    eval_name=["valid"],
    eval_metric=["auc"],
    max_epochs=TABNET_V2_MAX_EPOCHS,
    patience=TABNET_V2_PATIENCE,
    batch_size=TABNET_V2_BATCH_SIZE,
    virtual_batch_size=TABNET_V2_VIRTUAL_BATCH_SIZE,
    num_workers=0,
    drop_last=False,
    pin_memory=True,
    weights=0,
    callbacks=[tabnet_v2_progress],
)

tabnet_v2_seconds = time.perf_counter() - tabnet_v2_start

# Predictions
downside_tabnet_v2_train_probability = downside_tabnet_v2.predict_proba(X_downside_tabnet_train)[:, 1]
downside_tabnet_v2_valid_probability = downside_tabnet_v2.predict_proba(X_downside_tabnet_valid)[:, 1]

# Metrics
downside_tabnet_v2_train_metrics = calculate_binary_metrics(
    y_downside_tabnet_train,
    downside_tabnet_v2_train_probability,
    0.50,
)

downside_tabnet_v2_valid_metrics = calculate_binary_metrics(
    y_downside_tabnet_valid,
    downside_tabnet_v2_valid_probability,
    0.50,
)

# Save model and history
saved_tabnet_v2_path = downside_tabnet_v2.save_model(str(DOWNSIDE_TABNET_V2_PATH))
tabnet_v2_history = getattr(downside_tabnet_v2.history, "history", downside_tabnet_v2.history)
pd.DataFrame(tabnet_v2_history).to_csv(DOWNSIDE_TABNET_V2_HISTORY_PATH, index=False)

# Validation comparison
downside_tabnet_v2_comparison = pd.DataFrame([
    {"model": "random_forest", **downside_rf_valid_metrics},
    {"model": "tabnet_v1_balanced", **downside_tabnet_valid_metrics},
    {"model": "tabnet_v2_unweighted", **downside_tabnet_v2_valid_metrics},
]).sort_values(["roc_auc", "average_precision"], ascending=False).reset_index(drop=True)

downside_tabnet_v2_comparison.to_csv(DOWNSIDE_TABNET_V2_COMPARISON_PATH, index=False)

print("\nDownside TabNet V2 training completed.")
print("Device:", downside_tabnet_v2.device)
print("Best epoch:", downside_tabnet_v2.best_epoch)
print("Best validation ROC-AUC:", round(float(downside_tabnet_v2.best_cost), 6))
print("Training time:", round(tabnet_v2_seconds, 2), "seconds")
print("Saved model:", saved_tabnet_v2_path)
print("History:", DOWNSIDE_TABNET_V2_HISTORY_PATH)
print("\nNatural class distribution was used: weights=0")
print("The previously viewed test period was not used.")

display(downside_tabnet_v2_comparison[[
    "model", "rows", "roc_auc", "average_precision", "log_loss", "brier_score",
    "balanced_accuracy", "precision", "recall", "f1", "average_probability",
]])

Downside TabNet V2:   0%|          | 0/100 [00:00<?, ?epoch/s]

Epoch 001:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 002:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 003:   0%|          | 0/62 [00:00<?, ?batch/s]

Epoch 004:   0%|          | 0/62 [00:00<?, ?batch/s]

...

Epoch 023:   0%|          | 0/62 [00:00<?, ?batch/s]


Early stopping occurred at epoch 22 with best_epoch = 7 and best_valid_auc = 0.62255


c:\Users\ppava\anaconda3\envs\marketguard\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Successfully saved model at E:\Projects\marketguard-india\models\deep_learning\tabnet_downside_purged_v2.zip

Downside TabNet V2 training completed.
Device: cuda
Best epoch: 7
Best validation ROC-AUC: 0.622555
Training time: 170.86 seconds
Saved model: E:\Projects\marketguard-india\models\deep_learning\tabnet_downside_purged_v2.zip
History: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\tabnet_downside_training_history_v2.csv

Natural class distribution was used: weights=0
The previously viewed test period was not used.


,model,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,random_forest,19980,0.659263,0.202046,0.594743,0.202433,0.625919,0.191630,0.498092,0.276776,0.435097
1,tabnet_v1_balanced,19980,0.634317,0.182942,0.821576,0.306999,0.568830,0.124206,0.792939,0.214770,0.550291
2,tabnet_v2_unweighted,19980,0.622555,0.168453,0.335586,0.094252,0.501534,0.220000,0.005248,0.010252,0.138062


## Downside TabNet V2 Result

TabNet V2 used:

- Natural class distribution with `weights=0`
- Three decision steps instead of five
- Smaller hidden dimensions
- Lower learning rate
- Stronger weight decay and sparsity regularization

### Validation Comparison

| Model | ROC-AUC | Average Precision | Log Loss | Brier Score |
|---|---:|---:|---:|---:|
| Random Forest | **0.6593** | **0.2020** | 0.5947 | 0.2024 |
| TabNet V1 balanced | 0.6343 | 0.1829 | 0.8216 | 0.3070 |
| TabNet V2 unweighted | 0.6226 | 0.1685 | **0.3356** | **0.0943** |

Removing class balancing corrected the excessively high probability scale.

However, TabNet V2 produced weaker ROC-AUC and average precision than TabNet V1.
Its low recall at the `0.50` threshold occurred because the unweighted model
rarely predicted downside probabilities above 50%.

The improved log loss mainly reflects predictions close to the overall downside
event rate rather than stronger event ranking.

### Decision

TabNet V2 will not replace TabNet V1 or the Random Forest.

The Random Forest remains the strongest downside model.

The TabNet tuning suggestions improved probability scale but did not solve the
main ranking-performance limitation.

In [40]:
# Final downside validation ensemble search
downside_targets = np.asarray(y_valid_downside, dtype=np.int64)

downside_probabilities = {
    "random_forest": np.asarray(downside_rf_valid_probability, dtype=np.float64),
    "weighted_mlp": np.asarray(downside_mlp_valid_probability, dtype=np.float64),
    "ft_transformer": np.asarray(downside_ft_valid_probability, dtype=np.float64),
    "tabnet_v1": np.asarray(downside_tabnet_valid_probability, dtype=np.float64),
}

if any(len(probability) != len(downside_targets) for probability in downside_probabilities.values()):
    raise ValueError("Downside validation arrays are not aligned.")

# Individual model comparison
individual_rows = []

for model_name, probability in downside_probabilities.items():
    individual_rows.append({"model": model_name, **calculate_binary_metrics(downside_targets, probability, 0.50)})

downside_individual_comparison = pd.DataFrame(individual_rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

# Coarse ensemble search using 10% increments
weight_units = 10
ensemble_rows = []

for rf_units in range(weight_units + 1):
    for mlp_units in range(weight_units - rf_units + 1):
        for ft_units in range(weight_units - rf_units - mlp_units + 1):
            tabnet_units = weight_units - rf_units - mlp_units - ft_units

            rf_weight = rf_units / weight_units
            mlp_weight = mlp_units / weight_units
            ft_weight = ft_units / weight_units
            tabnet_weight = tabnet_units / weight_units

            probability = (
                rf_weight * downside_probabilities["random_forest"]
                + mlp_weight * downside_probabilities["weighted_mlp"]
                + ft_weight * downside_probabilities["ft_transformer"]
                + tabnet_weight * downside_probabilities["tabnet_v1"]
            )

            metrics = calculate_binary_metrics(downside_targets, probability, 0.50)

            ensemble_rows.append({
                "random_forest_weight": rf_weight,
                "weighted_mlp_weight": mlp_weight,
                "ft_transformer_weight": ft_weight,
                "tabnet_v1_weight": tabnet_weight,
                **metrics,
            })

downside_ensemble_search = pd.DataFrame(ensemble_rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

best_downside_ensemble = downside_ensemble_search.iloc[0]

# Correlations
downside_probability_correlations = pd.DataFrame(downside_probabilities).corr()

# Save reports
DOWNSIDE_FINAL_VALIDATION_PATH = REPORT_DIR / "downside_final_validation_comparison_v1.csv"
DOWNSIDE_ENSEMBLE_SEARCH_PATH = REPORT_DIR / "downside_final_ensemble_search_v1.csv"
DOWNSIDE_CORRELATION_PATH = REPORT_DIR / "downside_model_probability_correlations_v1.csv"

downside_individual_comparison.to_csv(DOWNSIDE_FINAL_VALIDATION_PATH, index=False)
downside_ensemble_search.to_csv(DOWNSIDE_ENSEMBLE_SEARCH_PATH, index=False)
downside_probability_correlations.to_csv(DOWNSIDE_CORRELATION_PATH)

print("Best validation ensemble weights:")
print("  Random Forest:", best_downside_ensemble["random_forest_weight"])
print("  Weighted MLP:", best_downside_ensemble["weighted_mlp_weight"])
print("  FT-Transformer:", best_downside_ensemble["ft_transformer_weight"])
print("  TabNet V1:", best_downside_ensemble["tabnet_v1_weight"])

print("\nBest ensemble ROC-AUC:", round(float(best_downside_ensemble["roc_auc"]), 6))
print("Best ensemble average precision:", round(float(best_downside_ensemble["average_precision"]), 6))
print("\nThe previously viewed test period was not used in this search.")

print("\nIndividual models:")
display(downside_individual_comparison[[
    "model", "roc_auc", "average_precision", "log_loss", "brier_score",
    "precision", "recall", "f1", "average_probability",
]])

print("\nTop 10 ensembles:")
display(downside_ensemble_search[[
    "random_forest_weight", "weighted_mlp_weight", "ft_transformer_weight",
    "tabnet_v1_weight", "roc_auc", "average_precision", "log_loss",
    "brier_score", "precision", "recall", "f1",
]].head(10))

print("\nProbability correlations:")
display(downside_probability_correlations)

Best validation ensemble weights:
  Random Forest: 0.7
  Weighted MLP: 0.0
  FT-Transformer: 0.1
  TabNet V1: 0.2

Best ensemble ROC-AUC: 0.677019
Best ensemble average precision: 0.22878

The previously viewed test period was not used in this search.

Individual models:


,model,roc_auc,average_precision,log_loss,brier_score,precision,recall,f1,average_probability
0,random_forest,0.659263,0.202046,0.594743,0.202433,0.191630,0.498092,0.276776,0.435097
1,tabnet_v1,0.634317,0.182942,0.821576,0.306999,0.124206,0.792939,0.214770,0.550291
2,ft_transformer,0.623660,0.170670,0.825156,0.289719,0.138597,0.616412,0.226309,0.478920
3,weighted_mlp,0.617987,0.194551,0.709208,0.257413,0.135373,0.652195,0.224209,0.489839



Top 10 ensembles:


,random_forest_weight,weighted_mlp_weight,ft_transformer_weight,tabnet_v1_weight,roc_auc,average_precision,log_loss,brier_score,precision,recall,f1
0,0.7,0.0,0.1,0.2,0.677019,0.228780,0.631749,0.220279,0.172043,0.603053,0.267712
1,0.6,0.0,0.1,0.3,0.676672,0.228189,0.649947,0.229115,0.164375,0.644561,0.261949
2,0.8,0.0,0.1,0.1,0.674083,0.226694,0.615041,0.212210,0.183474,0.572042,0.277836
3,0.5,0.0,0.1,0.4,0.673822,0.225685,0.669754,0.238718,0.155186,0.680344,0.252725
4,0.7,0.0,0.0,0.3,0.673679,0.219986,0.643072,0.225749,0.166223,0.625000,0.262604
5,0.6,0.1,0.1,0.2,0.673525,0.230273,0.640966,0.224759,0.165736,0.623092,0.261828
6,0.8,0.0,0.0,0.2,0.673132,0.219104,0.625537,0.217210,0.176395,0.586832,0.271254
7,0.5,0.1,0.1,0.3,0.673096,0.229726,0.659694,0.233842,0.157967,0.658397,0.254801
8,0.6,0.0,0.2,0.2,0.672587,0.226787,0.640741,0.224737,0.168266,0.627385,0.265362
9,0.5,0.0,0.2,0.3,0.672206,0.225730,0.659752,0.233870,0.159708,0.656966,0.256951



Probability correlations:


,random_forest,weighted_mlp,ft_transformer,tabnet_v1
random_forest,1.000000,0.670221,0.372173,0.422827
weighted_mlp,0.670221,1.000000,0.379522,0.504923
ft_transformer,0.372173,0.379522,1.000000,0.363374
tabnet_v1,0.422827,0.504923,0.363374,1.000000


## Final Downside Ensemble Validation Result

The best validation ensemble used:

```text
70% Random Forest
10% FT-Transformer
20% TabNet
0% Weighted MLP
```

### Validation Results

| Model | ROC-AUC | Average Precision |
|---|---:|---:|
| Random Forest | 0.6593 | 0.2020 |
| Downside ensemble | **0.6770** | **0.2288** |

The ensemble improved:

- ROC-AUC by approximately `0.0178`
- Average precision by approximately `0.0267`
- Recall from `49.8%` to `60.3%`

The FT-Transformer and TabNet predictions had relatively low correlations with
the Random Forest, so they added complementary ranking information despite being
weaker standalone models.

The ensemble produced worse log loss and Brier score than the Random Forest.
Therefore, its output should be treated as a relative downside-risk score rather
than a literal probability.

### Decision

The fixed ensemble remains a validation candidate:

```text
70% Random Forest
10% FT-Transformer
20% TabNet
```

The weighted MLP received zero weight and is excluded.

A 20-trading-day block bootstrap will be used to test whether the ranking
improvement is stable across different market periods.

In [41]:
# Fixed downside ensemble
DOWNSIDE_RF_WEIGHT = 0.70
DOWNSIDE_FT_WEIGHT = 0.10
DOWNSIDE_TABNET_WEIGHT = 0.20
DOWNSIDE_BLOCK_ITERATIONS = 2000
DOWNSIDE_BLOCK_LENGTH = 20

downside_ensemble_probability = (
    DOWNSIDE_RF_WEIGHT * downside_rf_valid_probability
    + DOWNSIDE_FT_WEIGHT * downside_ft_valid_probability
    + DOWNSIDE_TABNET_WEIGHT * downside_tabnet_valid_probability
)

targets = np.asarray(y_valid_downside, dtype=np.int64)
rf_probability = np.asarray(downside_rf_valid_probability, dtype=np.float64)
ensemble_probability = np.asarray(downside_ensemble_probability, dtype=np.float64)

metadata = model_data.loc[valid_mask, ["date", "yf_ticker"]].reset_index(drop=True)
dates = pd.DatetimeIndex(metadata["date"].drop_duplicates().sort_values())
date_to_indices = {date: group.index.to_numpy() for date, group in metadata.groupby("date", sort=True)}

max_start = len(dates) - DOWNSIDE_BLOCK_LENGTH
blocks_needed = int(np.ceil(len(dates) / DOWNSIDE_BLOCK_LENGTH))

observed_rf = calculate_probability_metrics(targets, rf_probability)
observed_ensemble = calculate_probability_metrics(targets, ensemble_probability)

rng = np.random.default_rng(RANDOM_SEED)
rows = []

for iteration in tqdm(range(DOWNSIDE_BLOCK_ITERATIONS), desc="Downside ensemble block bootstrap", unit="sample", dynamic_ncols=True):
    starts = rng.integers(0, max_start + 1, size=blocks_needed)
    positions = np.concatenate([np.arange(start, start + DOWNSIDE_BLOCK_LENGTH) for start in starts])[:len(dates)]
    indices = np.concatenate([date_to_indices[date] for date in dates[positions]])
    sampled_targets = targets[indices]

    if np.unique(sampled_targets).size < 2:
        continue

    rf_metrics = calculate_probability_metrics(sampled_targets, rf_probability[indices])
    ensemble_metrics = calculate_probability_metrics(sampled_targets, ensemble_probability[indices])

    rows.append({
        "iteration": iteration + 1,
        "roc_auc_difference": ensemble_metrics["roc_auc"] - rf_metrics["roc_auc"],
        "average_precision_difference": ensemble_metrics["average_precision"] - rf_metrics["average_precision"],
        "log_loss_difference": ensemble_metrics["log_loss"] - rf_metrics["log_loss"],
        "brier_difference": ensemble_metrics["brier_score"] - rf_metrics["brier_score"],
    })

downside_block_results = pd.DataFrame(rows)

metric_directions = {
    "roc_auc_difference": "higher",
    "average_precision_difference": "higher",
    "log_loss_difference": "lower",
    "brier_difference": "lower",
}

observed_differences = {
    "roc_auc_difference": observed_ensemble["roc_auc"] - observed_rf["roc_auc"],
    "average_precision_difference": observed_ensemble["average_precision"] - observed_rf["average_precision"],
    "log_loss_difference": observed_ensemble["log_loss"] - observed_rf["log_loss"],
    "brier_difference": observed_ensemble["brier_score"] - observed_rf["brier_score"],
}

summary_rows = []

for metric, direction in metric_directions.items():
    values = downside_block_results[metric].to_numpy()
    lower, upper = np.quantile(values, [0.025, 0.975])
    win_rate = np.mean(values > 0) if direction == "higher" else np.mean(values < 0)

    summary_rows.append({
        "metric": metric,
        "observed_difference": observed_differences[metric],
        "bootstrap_mean_difference": values.mean(),
        "ci_2_5_pct": lower,
        "ci_97_5_pct": upper,
        "ensemble_win_rate": win_rate,
        "ci_excludes_zero": bool(lower > 0 or upper < 0),
        "better_direction": direction,
    })

downside_block_summary = pd.DataFrame(summary_rows)

DOWNSIDE_BLOCK_DETAIL_PATH = REPORT_DIR / "downside_final_ensemble_block_bootstrap_v1.csv"
DOWNSIDE_BLOCK_SUMMARY_PATH = REPORT_DIR / "downside_final_ensemble_block_bootstrap_summary_v1.csv"

downside_block_results.to_csv(DOWNSIDE_BLOCK_DETAIL_PATH, index=False)
downside_block_summary.to_csv(DOWNSIDE_BLOCK_SUMMARY_PATH, index=False)

print("Completed samples:", len(downside_block_results))
print("Fixed weights: RF=0.70, FT=0.10, TabNet=0.20")
print("Positive ROC-AUC/AP differences favour the ensemble.")
print("Negative log-loss/Brier differences favour the ensemble.")

display(downside_block_summary)

Downside ensemble block bootstrap:   0%|          | 0/2000 [00:00<?, ?sample/s]

Completed samples: 2000
Fixed weights: RF=0.70, FT=0.10, TabNet=0.20
Positive ROC-AUC/AP differences favour the ensemble.
Negative log-loss/Brier differences favour the ensemble.


,metric,observed_difference,bootstrap_mean_difference,ci_2_5_pct,ci_97_5_pct,ensemble_win_rate,ci_excludes_zero,better_direction
0,roc_auc_difference,0.017755,0.010925,-0.006623,0.027790,0.8655,False,higher
1,average_precision_difference,0.026735,0.017793,-0.001618,0.038688,0.9620,False,higher
2,log_loss_difference,0.037006,0.041760,0.020269,0.059634,0.0000,True,lower
3,brier_difference,0.017845,0.020171,0.009865,0.028682,0.0000,True,lower


## Downside Ensemble Block-Bootstrap Result

The fixed downside ensemble used:

```text
70% Random Forest
10% FT-Transformer
20% TabNet
```

It was compared with the Random Forest using 2,000 bootstrap samples containing
complete 20-trading-day blocks.

### Results

| Metric | Observed Difference | 95% Confidence Interval | Ensemble Win Rate |
|---|---:|---:|---:|
| ROC-AUC | +0.0178 | −0.0066 to +0.0278 | 86.55% |
| Average precision | +0.0267 | −0.0016 to +0.0387 | 96.20% |
| Log loss | +0.0370 | +0.0203 to +0.0596 | 0.00% |
| Brier score | +0.0178 | +0.0099 to +0.0287 | 0.00% |

### Interpretation

The ensemble achieved better observed ROC-AUC and average precision than the
Random Forest.

It also improved ranking in most sampled market periods.

However, the ranking confidence intervals included zero, so the improvement was
not statistically reliable across different market periods.

The log-loss and Brier-score differences were entirely above zero. Because
lower values are better, the Random Forest produced reliably better probability
estimates.

### Decision

The ensemble remains an exploratory downside-ranking candidate.

The Random Forest remains the preferred production model because it is more
stable and produces better probability-quality metrics.

Any evaluation on the previously viewed test period will be labelled
exploratory and will not be used to change the original production decision.

In [43]:
# Exploratory downside test comparison
DOWNSIDE_EXPLORATORY_TEST_PATH = REPORT_DIR / "downside_exploratory_ensemble_test_v1.csv"

test_targets = np.asarray(y_test_downside, dtype=np.int64)
X_rf_test = model_data.loc[test_mask, feature_cols]
X_tabnet_test = np.ascontiguousarray(X_test, dtype=np.float32)

rf_test_probability = purged_rf_downside.predict_proba(X_rf_test)[:, 1]

ft_test_loader = create_data_loader(downside_test_dataset, DOWNSIDE_FT_EVAL_BATCH_SIZE, False, RANDOM_SEED)
_, ft_test_targets, ft_test_probability = evaluate_binary_model(
    downside_ft, ft_test_loader, downside_ft_loss, DEVICE, 0.50
)

ft_test_targets = np.asarray(ft_test_targets, dtype=np.int64)
ft_test_probability = np.asarray(ft_test_probability, dtype=np.float64)
tabnet_test_probability = downside_tabnet.predict_proba(X_tabnet_test)[:, 1]

if not np.array_equal(test_targets, ft_test_targets):
    raise ValueError("FT-Transformer test targets are not aligned.")

if not (len(test_targets) == len(rf_test_probability) == len(ft_test_probability) == len(tabnet_test_probability)):
    raise ValueError("Downside test probabilities are not aligned.")

ensemble_test_probability = (
    0.70 * rf_test_probability
    + 0.10 * ft_test_probability
    + 0.20 * tabnet_test_probability
)

test_probabilities = {
    "random_forest": rf_test_probability,
    "ft_transformer": ft_test_probability,
    "tabnet_v1": tabnet_test_probability,
    "rf_70_ft_10_tabnet_20": ensemble_test_probability,
}

rows = []

for model_name, probability in test_probabilities.items():
    metrics = calculate_binary_metrics(test_targets, probability, 0.50)
    rows.append({"model": model_name, **metrics})

downside_exploratory_test = pd.DataFrame(rows).sort_values(
    ["roc_auc", "average_precision", "log_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

downside_exploratory_test.to_csv(DOWNSIDE_EXPLORATORY_TEST_PATH, index=False)

print("Exploratory downside test comparison completed.")
print("Fixed weights: RF=0.70, FT-Transformer=0.10, TabNet=0.20")
print("Actual downside-event rate:", round(float(test_targets.mean()), 6))
print("Report:", DOWNSIDE_EXPLORATORY_TEST_PATH)
print("\nThis test period was previously viewed, so these results are exploratory.")
print("No model weights or production decisions will be changed.")

display(downside_exploratory_test[[
    "model", "rows", "roc_auc", "average_precision", "log_loss", "brier_score",
    "balanced_accuracy", "precision", "recall", "f1", "average_probability",
]])

Exploratory downside test comparison completed.
Fixed weights: RF=0.70, FT-Transformer=0.10, TabNet=0.20
Actual downside-event rate: 0.111965
Report: E:\Projects\marketguard-india\reports\deep_learning_experiments\tabular_mlp\downside_exploratory_ensemble_test_v1.csv

This test period was previously viewed, so these results are exploratory.
No model weights or production decisions will be changed.


,model,rows,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,precision,recall,f1,average_probability
0,random_forest,32135,0.684487,0.205591,0.545953,0.184022,0.624656,0.195056,0.519733,0.283656,0.393105
1,rf_70_ft_10_tabnet_20,32135,0.640791,0.167518,0.559847,0.190745,0.579987,0.163161,0.452752,0.239876,0.393799
2,ft_transformer,32135,0.588253,0.134428,0.612678,0.199303,0.545745,0.148754,0.328516,0.204782,0.318162
3,tabnet_v1,32135,0.534010,0.125209,0.713771,0.256741,0.502715,0.113249,0.425236,0.178864,0.434048


## Exploratory Downside Ensemble Test Result

The fixed exploratory ensemble used:

```text
70% Random Forest
10% FT-Transformer
20% TabNet
```

### Test Comparison

| Model | ROC-AUC | Average Precision | Precision | Recall |
|---|---:|---:|---:|---:|
| Random Forest | **0.6845** | **0.2056** | **0.1951** | **0.5197** |
| Ensemble | 0.6408 | 0.1675 | 0.1632 | 0.4528 |
| FT-Transformer | 0.5883 | 0.1344 | 0.1488 | 0.3285 |
| TabNet | 0.5340 | 0.1252 | 0.1132 | 0.4252 |

The ensemble validation improvement did not generalize to the later test period.

Adding the weaker neural models reduced ROC-AUC, average precision, precision,
recall, log loss, and Brier score compared with the Random Forest alone.

### Final Decision

The Random Forest remains the final downside-risk model.

The neural models and ensemble will not be promoted.

The downside deep-learning experiment is complete.

### Overall Deep-Learning Conclusion

For both MarketGuard targets:

```text
Outperformance model: Random Forest retained
Downside-risk model:  Random Forest retained
```

The deep-learning models learned historical patterns but did not provide stable
improvements on later market data.

